In [ ]:

# =============================================================================
# NETWORK ATTACK FORECASTING PIPELINE
# CELL 1 — ENVIRONMENT, CHECKPOINT VERIFICATION & MODEL REGISTRY
#
# PURPOSE:
# Establish the saved models and data contracts used by
# the final forecasting engine.
#
# NO TRAINING
# NO MODEL MODIFICATION
# NO TEST ACCESS
# NO FILES WRITTEN
# =============================================================================

from pathlib import Path
import hashlib
import json
import sys
import warnings

import numpy as np
import pandas as pd
import torch

from google.colab import drive

# =============================================================================
# 1. GOOGLE DRIVE
# =============================================================================

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

if not ROOT.is_dir():
    raise FileNotFoundError(
        f"Project folder not found:\n{ROOT}"
    )

V2_ROOT = ROOT / "Synthetic_NAF_v2"

DATA_DIR = (
    V2_ROOT / "release_candidate_corrected"
)

GRAPH_ROOT = (
    ROOT / "Synthetic_NAF_v2_GraphCompanion"
)

CONTRACT_FILE = (
    DATA_DIR / "FEATURE_AND_DATASET_CONTRACT.json"
)

HASH_MANIFEST = (
    DATA_DIR / "SHA256SUMS.json"
)

print("=" * 90)
print("NETWORK ATTACK FORECASTING — PIPELINE SETUP")
print("=" * 90)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Project:", ROOT)

# =============================================================================
# 2. VERIFY FROZEN INPUT CONTRACT
# =============================================================================

for path in [CONTRACT_FILE, HASH_MANIFEST]:
    if not path.is_file():
        raise FileNotFoundError(path)

CONTRACT = json.loads(
    CONTRACT_FILE.read_text(encoding="utf-8")
)

SOURCE_HASHES = json.loads(
    HASH_MANIFEST.read_text(encoding="utf-8")
)

FEATURE_NAMES = list(CONTRACT["features"])

HISTORY_MINUTES = int(
    CONTRACT["history_minutes"]
)

FORECAST_HORIZONS = list(
    CONTRACT["forecast_horizons"]
)

if HISTORY_MINUTES != 10:
    raise RuntimeError(
        "Unexpected frozen history length."
    )

if FORECAST_HORIZONS != [5, 10, 15]:
    raise RuntimeError(
        "Unexpected forecasting horizons."
    )

if len(FEATURE_NAMES) != 14:
    raise RuntimeError(
        "Expected exactly 14 input features."
    )

if len(set(FEATURE_NAMES)) != 14:
    raise RuntimeError(
        "Duplicate names in frozen feature contract."
    )

print("\n" + "-" * 90)
print("FROZEN DATA CONTRACT")
print("-" * 90)

print("History:", HISTORY_MINUTES, "minutes")
print("Features:", len(FEATURE_NAMES))
print("Horizons:", FORECAST_HORIZONS)

for i, feature in enumerate(
    FEATURE_NAMES,
    start=1
):
    print(f"  {i:02d}. {feature}")

print("\nFrozen contract: PASS")

# =============================================================================
# 3. CHECKSUM FUNCTION
# =============================================================================

def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as stream:

        for chunk in iter(
            lambda: stream.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


# =============================================================================
# 4. REGISTER TRAINED PYTORCH CHECKPOINTS
# =============================================================================

CHECKPOINT_PATHS = {

    "single_lstm": (
        V2_ROOT / "models/LSTM_V2_SingleLayer"
        / "LSTM_V2_SINGLE_LAYER_STATE.pt"
    ),

    "stacked_lstm": (
        V2_ROOT / "models/LSTM_V2_Horizon"
        / "final_train_model"
        / "LSTM_V2_MONOTONIC_HORIZON_STATE.pt"
    ),

    "competing_risk_lstm": (
        V2_ROOT / "models/LSTM_V2_CompetingRisk"
        / "final_train_model"
        / "LSTM_V2_CR_STATE.pt"
    ),

    "competing_risk_tcn": (
        V2_ROOT / "models/TCN_V2_CompetingRisk"
        / "final_train_model"
        / "TCN_V2_CR_STATE.pt"
    ),

    "node_aware_lstm": (
        GRAPH_ROOT / "models/NodeAware_LSTM"
        / "train_only_v1"
        / "NODE_LSTM_STATE.pt"
    ),

    "custom_graph_lstm": (
        GRAPH_ROOT / "models/Graph_LSTM"
        / "train_only_v1"
        / "GRAPH_LSTM_STATE.pt"
    ),

    "optimized_graph_lstm": (
        GRAPH_ROOT / "models/Graph_LSTM"
        / "optimized_full_train_v1"
        / "GRAPH_LSTM_C02_OPTIMIZED_STATE.pt"
    ),

    "gcn_lstm": (
        GRAPH_ROOT / "models/GCN_LSTM"
        / "train_only_v1"
        / "GCN_LSTM_STATE.pt"
    ),

    "gat_lstm": (
        GRAPH_ROOT / "models/GAT_LSTM"
        / "train_only_v1"
        / "GAT_LSTM_STATE.pt"
    )
}

# These models are essential for the first version of
# our forecasting and attack-family/onset pipeline.
#
# Graph models are optional experimental adapters.

REQUIRED_MODELS = {
    "single_lstm",
    "stacked_lstm",
    "competing_risk_lstm"
}

MODEL_REGISTRY = {}

print("\n" + "-" * 90)
print("CHECKPOINT REGISTRY")
print("-" * 90)

for model_name, path in CHECKPOINT_PATHS.items():

    entry = {
        "name": model_name,
        "path": str(path),
        "required": model_name in REQUIRED_MODELS,
        "exists": path.is_file(),
        "checkpoint_verified": False,
        "sha256": None,
        "checkpoint_keys": [],
        "error": None
    }

    if not path.is_file():

        entry["error"] = "Checkpoint missing"

        MODEL_REGISTRY[model_name] = entry

        print(
            f"[MISSING] {model_name}"
        )

        continue

    try:

        # Trusted checkpoints generated by our own
        # previous Colab training notebooks.
        #
        # Load on CPU only to inspect checkpoint
        # structure. No forward inference yet.

        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        if not isinstance(checkpoint, dict):
            raise TypeError(
                "Expected a checkpoint dictionary."
            )

        keys = list(
            checkpoint.keys()
        )

        entry["checkpoint_keys"] = [
            str(key)
            for key in keys
        ]

        entry["sha256"] = sha256(path)

        # Some models use different checkpoint keys.
        # Record their structure rather than guessing.
        state_keys = [
            "model_state",
            "model_state_dict",
            "state_dict"
        ]

        entry["has_standard_state_key"] = any(
            key in checkpoint
            for key in state_keys
        )

        entry["checkpoint_verified"] = True

        print(
            f"[PASS] {model_name}"
        )

        print(
            "       SHA256:",
            entry["sha256"][:20] + "..."
        )

        print(
            "       Keys:",
            ", ".join(entry["checkpoint_keys"][:10])
        )

        del checkpoint

    except Exception as exc:

        entry["error"] = (
            f"{type(exc).__name__}: {exc}"
        )

        print(
            f"[ERROR] {model_name}: {entry['error']}"
        )

    MODEL_REGISTRY[model_name] = entry

# =============================================================================
# 5. DISCOVER XGBOOST MODEL FILES
# =============================================================================

print("\n" + "-" * 90)
print("XGBOOST MODEL DISCOVERY")
print("-" * 90)

XGB_DIR = (
    V2_ROOT / "models/XGBoost_V2_FLAT140"
)

XGB_CANDIDATES = []

if XGB_DIR.is_dir():

    for path in XGB_DIR.rglob("*"):

        if not path.is_file():
            continue

        # Never inspect TEST-related artifacts.
        if any(
            "test" in part.lower()
            for part in path.parts[len(XGB_DIR.parts):]
        ):
            continue

        name = path.name.lower()

        if (
            name in {
                "model.json",
                "model.ubj",
                "model.bin"
            }
            or (
                path.suffix.lower() in {
                    ".ubj", ".model"
                }
            )
        ):

            XGB_CANDIDATES.append(path)

XGB_CANDIDATES = sorted(
    set(XGB_CANDIDATES)
)

MODEL_REGISTRY["xgboost"] = {
    "name": "xgboost",
    "search_directory": str(XGB_DIR),
    "candidate_paths": [
        str(p)
        for p in XGB_CANDIDATES
    ],
    "exists": len(XGB_CANDIDATES) > 0,
    "checkpoint_verified": False,
    "note": (
        "Candidate model files discovered. "
        "Exact H5/H10 model association and "
        "preprocessing must be verified "
        "before inference."
    )
}

if XGB_CANDIDATES:

    print(
        "Potential XGBoost model files:",
        len(XGB_CANDIDATES)
    )

    for path in XGB_CANDIDATES[:30]:

        print(
            " ",
            path.relative_to(ROOT)
        )

else:

    print(
        "No XGBoost serialized model found "
        "in the expected directory."
    )

    print(
        "We will inspect XGBoost's training "
        "artifacts before enabling live inference."
    )

# =============================================================================
# 6. REGISTER GRAPH INPUT ARTIFACTS
# =============================================================================

print("\n" + "-" * 90)
print("SYNTHETIC GRAPH INPUT REGISTRY")
print("-" * 90)

GRAPH_TRAIN = (
    GRAPH_ROOT / "TRAIN_only_v1"
    / "TRAIN_temporal_graphs.npz"
)

GRAPH_TRAIN_MANIFEST = (
    GRAPH_ROOT / "TRAIN_only_v1"
    / "GRAPH_COMPANION_MANIFEST.json"
)

GRAPH_VAL = (
    GRAPH_ROOT / "VALIDATION_only_v1"
    / "VALIDATION_temporal_graphs.npz"
)

GRAPH_INPUT_REGISTRY = {
    "train_graph_path": str(GRAPH_TRAIN),
    "train_manifest_path": str(
        GRAPH_TRAIN_MANIFEST
    ),
    "validation_graph_path": str(
        GRAPH_VAL
    ),
    "synthetic_topology": True,
    "real_host_attribution_supported": False
}

for label, path in [
    ("TRAIN graph", GRAPH_TRAIN),
    ("TRAIN graph manifest", GRAPH_TRAIN_MANIFEST),
    ("VALIDATION graph", GRAPH_VAL)
]:

    status = (
        "FOUND" if path.is_file()
        else "MISSING"
    )

    print(
        f"{label}: {status}"
    )

# No TRAIN/VALIDATION graph arrays are loaded here.
# Official TEST is never accessed.

# =============================================================================
# 7. VERIFY REQUIRED MODELS
# =============================================================================

print("\n" + "=" * 90)
print("PIPELINE READINESS SUMMARY")
print("=" * 90)

rows = []

for name, entry in MODEL_REGISTRY.items():

    rows.append({
        "Model": name,
        "Required": entry.get("required", False),
        "Found": entry.get("exists", False),
        "Checkpoint inspected": entry.get(
            "checkpoint_verified", False
        )
    })

readiness = pd.DataFrame(rows)

print(
    readiness.to_string(index=False)
)

missing_required = [
    name
    for name in REQUIRED_MODELS
    if not MODEL_REGISTRY[name]["checkpoint_verified"]
]

if missing_required:

    raise RuntimeError(
        "Required model checkpoints are unavailable: "
        + ", ".join(sorted(missing_required))
    )

print("\n" + "=" * 90)
print("CELL 1 — MODEL REGISTRY: PASS")
print("=" * 90)

print("Required models located and inspected: YES")
print("Frozen feature contract verified: YES")
print("Model checkpoints modified: NO")
print("Model training performed: NO")
print("Official TEST accessed: NO")
print("Files written to Drive: NO")

print(
    "\nNEXT: Cell 2 — Define and validate "
    "the 10-minute traffic input contract."
)

print("=" * 90)



In [ ]:

# =============================================================================
# NETWORK ATTACK FORECASTING PIPELINE
# CELL 2 — INPUT CONTRACT AND TRAFFIC HISTORY VALIDATION
#
# Input:
#   10 consecutive minutes
#   14 aggregate network traffic features per minute
#
# Output:
#   Validated numpy array, shape (1, 10, 14)
#
# No training
# No checkpoint modification
# No TEST access
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 90)
print("CELL 2 — NETWORK TRAFFIC INPUT CONTRACT")
print("=" * 90)

# Cell 1 must have established:
# ROOT, DATA_DIR, FEATURE_NAMES,
# HISTORY_MINUTES and FORECAST_HORIZONS.

assert HISTORY_MINUTES == 10
assert len(FEATURE_NAMES) == 14
assert FORECAST_HORIZONS == [5, 10, 15]

EXPECTED_FEATURES = tuple(FEATURE_NAMES)

# These fields identify time but are NOT model predictors.
TIME_COLUMNS = {"minute", "timestamp"}


# =============================================================================
# 1. DEFINE STRICT TRAFFIC INPUT VALIDATOR
# =============================================================================

def validate_traffic_history(
    traffic_df,
    require_time=True
):
    """
    Validate a single 10-minute network traffic history.

    Parameters
    ----------
    traffic_df : pandas.DataFrame
        Must contain all 14 frozen V2 feature columns,
        plus exactly one of:
          - minute: consecutive integer minute indices
          - timestamp: consecutive minute-level datetimes

    require_time : bool
        If True, require an explicit chronological
        time identifier.

    Returns
    -------
    result : dict
        {
          "X": numpy float32 array (1, 10, 14),
          "features": list,
          "reference_time": int/str/None,
          "time_type": str,
          "status": "VALID"
        }
    """

    # -------------------------------------------------------------------------
    # A. CHECK INPUT TYPE AND NUMBER OF ROWS
    # -------------------------------------------------------------------------

    if not isinstance(traffic_df, pd.DataFrame):
        raise TypeError(
            "Traffic input must be a pandas DataFrame."
        )

    if len(traffic_df) != HISTORY_MINUTES:
        raise ValueError(
            f"Expected exactly {HISTORY_MINUTES} minutes, "
            f"received {len(traffic_df)} rows."
        )

    df = traffic_df.copy()

    if df.columns.duplicated().any():
        raise ValueError(
            "Duplicate DataFrame column names are not allowed."
        )

    # -------------------------------------------------------------------------
    # B. VERIFY EXACT FEATURE NAMES
    # -------------------------------------------------------------------------

    missing_features = [
        feature
        for feature in EXPECTED_FEATURES
        if feature not in df.columns
    ]

    if missing_features:
        raise ValueError(
            "Missing required traffic features: "
            + ", ".join(missing_features)
        )

    extra_columns = (
        set(df.columns)
        - set(EXPECTED_FEATURES)
        - TIME_COLUMNS
    )

    if extra_columns:
        raise ValueError(
            "Unexpected columns: "
            + ", ".join(sorted(map(str, extra_columns)))
        )

    has_minute = "minute" in df.columns
    has_timestamp = "timestamp" in df.columns

    if has_minute and has_timestamp:
        raise ValueError(
            "Provide either minute or timestamp, not both."
        )

    if require_time and not (
        has_minute or has_timestamp
    ):
        raise ValueError(
            "A minute or timestamp column is required "
            "to verify chronological ordering."
        )

    # -------------------------------------------------------------------------
    # C. VERIFY MINUTE/TIMESTAMP ORDERING
    # -------------------------------------------------------------------------

    reference_time = None
    time_type = "unspecified"

    if has_minute:

        minute_values = pd.to_numeric(
            df["minute"],
            errors="coerce"
        ).to_numpy(dtype=np.float64)

        if not np.isfinite(minute_values).all():
            raise ValueError(
                "Minute indices contain invalid values."
            )

        if not np.all(
            minute_values == np.floor(minute_values)
        ):
            raise ValueError(
                "Minute indices must be whole numbers."
            )

        if not np.all(
            np.diff(minute_values) == 1
        ):
            raise ValueError(
                "Traffic minutes must be consecutive "
                "and ordered from oldest to newest."
            )

        reference_time = int(minute_values[-1])
        time_type = "relative_minute_index"

    elif has_timestamp:

        timestamps = pd.to_datetime(
            df["timestamp"],
            errors="raise",
            utc=True
        )

        if timestamps.isna().any():
            raise ValueError(
                "Traffic timestamps cannot be missing."
            )

        # Require exact minute boundaries.
        if not (
            (timestamps.dt.second == 0).all()
            and (timestamps.dt.microsecond == 0).all()
            and (timestamps.dt.nanosecond == 0).all()
        ):
            raise ValueError(
                "Timestamps must be aligned to whole minutes."
            )

        differences = timestamps.diff().dropna()

        if not (
            differences == pd.Timedelta(minutes=1)
        ).all():
            raise ValueError(
                "Timestamps must be consecutive and "
                "ordered oldest to newest."
            )

        reference_time = timestamps.iloc[-1].isoformat()
        time_type = "utc_timestamp"

    # -------------------------------------------------------------------------
    # D. EXTRACT THE EXACT 14 FEATURES IN TRAINING ORDER
    # -------------------------------------------------------------------------

    # Reorder columns to match the frozen V2 feature contract.
    # Never rely on the user's incoming DataFrame column order.

    feature_frame = df.loc[
        :,
        list(EXPECTED_FEATURES)
    ]

    # Reject missing/non-numeric values rather than filling
    # them with invented or validation-fitted statistics.
    try:
        numeric_frame = feature_frame.apply(
            pd.to_numeric,
            errors="raise"
        )
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "Traffic features must all be numeric."
        ) from exc

    if numeric_frame.isna().any().any():
        raise ValueError(
            "Missing traffic feature values are not allowed."
        )

    values = numeric_frame.to_numpy(
        dtype=np.float64
    )

    if not np.isfinite(values).all():
        raise ValueError(
            "Traffic features contain NaN or infinity."
        )

    if (values < 0).any():
        raise ValueError(
            "Negative traffic feature values are not allowed "
            "under the frozen V2 input contract."
        )

    # Guard float32 conversion against overflow.
    if (
        values > np.finfo(np.float32).max
    ).any():
        raise ValueError(
            "Traffic features exceed float32 range."
        )

    X = values.astype(
        np.float32
    )[None, :, :]

    expected_shape = (
        1,
        HISTORY_MINUTES,
        len(EXPECTED_FEATURES)
    )

    if X.shape != expected_shape:
        raise RuntimeError(
            f"Wrong model input shape: {X.shape}"
        )

    return {
        "X": X,
        "features": list(EXPECTED_FEATURES),
        "reference_time": reference_time,
        "time_type": time_type,
        "history_minutes": HISTORY_MINUTES,
        "status": "VALID"
    }


# =============================================================================
# 2. LOAD ONE REAL FROZEN V2 VALIDATION HISTORY
# =============================================================================

# This is an input-contract smoke test, NOT model evaluation.
# Only X, episode_id, and minute are read.
# No target labels are loaded.

VALIDATION_FILE = (
    DATA_DIR / "windows_W10_VALIDATION.npz"
)

if not VALIDATION_FILE.is_file():
    raise FileNotFoundError(
        VALIDATION_FILE
    )

with np.load(
    VALIDATION_FILE,
    allow_pickle=False
) as data:

    sample_history = data["X"][0].astype(
        np.float32
    )

    sample_episode = str(
        data["episode_id"][0]
    )

    sample_end_minute = int(
        data["minute"][0]
    )

assert sample_history.shape == (10, 14)

# The sample covers minutes:
# end-9, end-8, ..., end.

sample_minutes = np.arange(
    sample_end_minute - 9,
    sample_end_minute + 1,
    dtype=np.int64
)

# Create a proper DataFrame containing the exact
# frozen traffic features and relative minute indices.

sample_df = pd.DataFrame(
    sample_history,
    columns=FEATURE_NAMES
)

sample_df.insert(
    0,
    "minute",
    sample_minutes
)

print("\nSynthetic benchmark sample:")
print("Episode:", sample_episode)
print("Minutes:", sample_minutes.tolist())
print("Input shape:", sample_history.shape)

# =============================================================================
# 3. RUN THE INPUT VALIDATOR
# =============================================================================

result = validate_traffic_history(
    sample_df
)

FORECAST_READY_X = result["X"]

assert result["status"] == "VALID"

assert FORECAST_READY_X.shape == (
    1, 10, 14
)

assert FORECAST_READY_X.dtype == np.float32

assert np.array_equal(
    FORECAST_READY_X[0],
    sample_history
)

print("\n" + "-" * 90)
print("VALIDATED TRAFFIC HISTORY")
print("-" * 90)

print("Validation status:", result["status"])
print("Reference time:", result["reference_time"])
print("Time representation:", result["time_type"])
print("Model input shape:", FORECAST_READY_X.shape)
print("Model input dtype:", FORECAST_READY_X.dtype)
print("Feature order verified: YES")
print("Chronological continuity verified: YES")

# =============================================================================
# 4. RUN NEGATIVE-INPUT TESTS
# =============================================================================

print("\nRunning invalid-input rejection tests...")

def must_reject(name, bad_frame):

    try:

        validate_traffic_history(
            bad_frame
        )

    except (ValueError, TypeError):

        print(
            f"[PASS] Rejected {name}"
        )

        return

    raise RuntimeError(
        f"Validator incorrectly accepted: {name}"
    )


# Missing historical minute
must_reject(
    "missing minute",
    sample_df.drop(
        index=sample_df.index[0]
    )
)

# Corrupted chronology
bad_order = sample_df.copy()

bad_order.loc[
    bad_order.index[4],
    "minute"
] += 2

must_reject(
    "nonconsecutive minutes",
    bad_order
)

# Missing required feature
must_reject(
    "missing feature",
    sample_df.drop(
        columns=[FEATURE_NAMES[0]]
    )
)

# Invalid numeric input
bad_nan = sample_df.copy()

bad_nan.loc[
    bad_nan.index[0],
    FEATURE_NAMES[0]
] = np.nan

must_reject(
    "NaN feature",
    bad_nan
)

# Negative traffic count
bad_negative = sample_df.copy()

bad_negative.loc[
    bad_negative.index[0],
    "Flow_Count"
] = -1

must_reject(
    "negative traffic value",
    bad_negative
)

# =============================================================================
# 5. FINAL REPORT
# =============================================================================

print("\n" + "=" * 90)
print("CELL 2 — INPUT CONTRACT: PASS")
print("=" * 90)

print("Required historical minutes:", HISTORY_MINUTES)
print("Required features:", len(FEATURE_NAMES))
print("Validated input shape:", FORECAST_READY_X.shape)

print("\nInput validation tests: PASS")
print("Chronological order enforced: YES")
print("Feature order enforced: YES")
print("Missing-value rejection: YES")
print("Nonnegative-value enforcement: YES")

print("\nModel inference performed: NO")
print("Model training performed: NO")
print("Frozen dataset modified: NO")
print("Official TEST accessed: NO")

print(
    "\nNEXT: Cell 3 — Restore forecasting architectures "
    "and run actual inference."
)

print("=" * 90)


In [ ]:

# =============================================================================
# NETWORK ATTACK FORECASTING PIPELINE
# CELL 3 — RESTORE LSTM MODELS AND GENERATE ACTUAL FORECAST
#
# Models:
#   1. Single-layer LSTM
#   2. Stacked LSTM
#
# Input:  (1, 10, 14) validated traffic history
# Output: H5, H10, H15 attack onset probabilities
#
# Safety:
#   No training
#   No scaler refitting
#   No checkpoint changes
#   No TEST access
# =============================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

print("=" * 90)
print("CELL 3 — LSTM FORECASTING INFERENCE")
print("=" * 90)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

# =============================================================================
# 1. SAVED MODEL PATHS
# =============================================================================

MODEL_SPECS = {
    "single_lstm": {
        "checkpoint": (
            V2_ROOT / "models/LSTM_V2_SingleLayer"
            / "LSTM_V2_SINGLE_LAYER_STATE.pt"
        ),
        "scaler": (
            V2_ROOT / "models/LSTM_V2_SingleLayer"
            / "LSTM_V2_SINGLE_LAYER_SCALER.json"
        ),
        "predictions": (
            V2_ROOT / "models/LSTM_V2_SingleLayer"
            / "official_validation/VALIDATION_PREDICTIONS.csv"
        ),
        "expected_layers": 1
    },

    "stacked_lstm": {
        "checkpoint": (
            V2_ROOT / "models/LSTM_V2_Horizon"
            / "final_train_model"
            / "LSTM_V2_MONOTONIC_HORIZON_STATE.pt"
        ),
        "scaler": (
            V2_ROOT / "models/LSTM_V2_Horizon"
            / "final_train_model"
            / "LSTM_V2_FULL_TRAIN_SCALER.json"
        ),
        "predictions": (
            V2_ROOT / "models/LSTM_V2_Horizon"
            / "official_validation/VALIDATION_PREDICTIONS.csv"
        ),
        "expected_layers": 2
    }
}

for name, spec in MODEL_SPECS.items():

    for key in ["checkpoint", "scaler", "predictions"]:

        if not spec[key].is_file():
            raise FileNotFoundError(
                f"{name} missing {key}: {spec[key]}"
            )

print("\nLSTM model artifacts: FOUND")

# =============================================================================
# 2. MODEL ARCHITECTURE
# =============================================================================

class MonotonicHorizonHead(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.linear = nn.Linear(
            input_dim,
            3
        )

    def forward(self, features):

        logits = self.linear(features)

        # Conditional interval hazards:
        # 0-5, 5-10, 10-15 minutes.
        hazards = torch.sigmoid(logits)

        # Cumulative probability of onset within:
        # 5, 10, and 15 minutes.
        survival = torch.cumprod(
            1.0 - hazards,
            dim=-1
        )

        return 1.0 - survival


class HorizonLSTM(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        dense_dim,
        num_layers
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        self.projection = nn.Sequential(
            nn.Linear(hidden_dim, dense_dim),
            nn.ReLU(),
            nn.Dropout(0.0)
        )

        self.head = MonotonicHorizonHead(
            dense_dim
        )

    def forward(self, x):

        sequence, _ = self.lstm(x)

        last_hidden = sequence[:, -1, :]

        projected = self.projection(
            last_hidden
        )

        return self.head(projected)


# =============================================================================
# 3. LOAD TRAIN-FITTED SCALER
# =============================================================================

def load_train_scaler(path):

    scaler = json.loads(
        path.read_text(encoding="utf-8")
    )

    required = {
        "feature_names",
        "median",
        "scale",
        "clip_value"
    }

    if not required.issubset(scaler):
        raise RuntimeError(
            f"Unexpected scaler format at {path}. "
            f"Found keys: {list(scaler)}"
        )

    if list(scaler["feature_names"]) != list(FEATURE_NAMES):
        raise RuntimeError(
            "Scaler feature order does not match "
            "frozen dataset contract."
        )

    median = np.asarray(
        scaler["median"],
        dtype=np.float32
    )

    scale = np.asarray(
        scaler["scale"],
        dtype=np.float32
    )

    clip = float(
        scaler["clip_value"]
    )

    if median.shape != (14,) or scale.shape != (14,):
        raise RuntimeError(
            "Invalid scaler dimensions."
        )

    if not (
        np.isfinite(median).all()
        and np.isfinite(scale).all()
        and np.all(scale > 0)
        and np.isfinite(clip)
        and clip > 0
    ):
        raise RuntimeError(
            "Invalid TRAIN scaler parameters."
        )

    return {
        "median": median,
        "scale": scale,
        "clip_value": clip
    }


def preprocess_lstm_input(X, scaler):

    X = np.asarray(
        X,
        dtype=np.float32
    )

    if X.ndim != 3 or X.shape[1:] != (10, 14):
        raise ValueError(
            f"Expected (batch,10,14), received {X.shape}"
        )

    if not np.isfinite(X).all() or (X < 0).any():
        raise ValueError(
            "Input contains invalid traffic features."
        )

    # Exact intended TRAIN transformation:
    # LOG1P -> MEDIAN/IQR -> CLIP

    transformed = np.log1p(X)

    transformed = (
        transformed - scaler["median"]
    ) / scaler["scale"]

    transformed = np.clip(
        transformed,
        -scaler["clip_value"],
        scaler["clip_value"]
    ).astype(np.float32)

    if not np.isfinite(transformed).all():
        raise RuntimeError(
            "Non-finite values after preprocessing."
        )

    return transformed


# =============================================================================
# 4. RECONSTRUCT MODEL FROM SAVED WEIGHTS
# =============================================================================

def restore_lstm_model(spec):

    # Own previously saved TRAIN checkpoint.
    state = torch.load(
        spec["checkpoint"],
        map_location="cpu",
        weights_only=True
    )

    if not isinstance(state, dict):
        raise RuntimeError(
            "Checkpoint must contain a state dictionary."
        )

    required_keys = {
        "lstm.weight_ih_l0",
        "lstm.weight_hh_l0",
        "projection.0.weight",
        "head.linear.weight"
    }

    if not required_keys.issubset(state):
        raise RuntimeError(
            "Saved LSTM state does not match "
            "the expected architecture."
        )

    input_dim = state[
        "lstm.weight_ih_l0"
    ].shape[1]

    hidden_dim = state[
        "lstm.weight_hh_l0"
    ].shape[1]

    dense_dim = state[
        "projection.0.weight"
    ].shape[0]

    num_layers = sum(
        f"lstm.weight_ih_l{i}" in state
        for i in range(20)
    )

    if num_layers != spec["expected_layers"]:
        raise RuntimeError(
            f"Expected {spec['expected_layers']} "
            f"layers, found {num_layers}"
        )

    if input_dim != 14:
        raise RuntimeError(
            "Wrong checkpoint input dimension."
        )

    if state["head.linear.weight"].shape != (
        3, dense_dim
    ):
        raise RuntimeError(
            "Horizon output head has unexpected shape."
        )

    model = HorizonLSTM(
        input_dim=input_dim,
        hidden_dim=hidden_dim,
        dense_dim=dense_dim,
        num_layers=num_layers
    )

    model.load_state_dict(
        state,
        strict=True
    )

    model = model.to(DEVICE)
    model.eval()

    return model


# =============================================================================
# 5. RESTORE BOTH TRAINED LSTMs
# =============================================================================

FORECAST_MODELS = {}
FORECAST_SCALERS = {}

for name, spec in MODEL_SPECS.items():

    model = restore_lstm_model(spec)

    scaler = load_train_scaler(
        spec["scaler"]
    )

    FORECAST_MODELS[name] = model
    FORECAST_SCALERS[name] = scaler

    print(
        f"\n{name}: LOADED"
    )

    print(
        "  Layers:",
        spec["expected_layers"]
    )

    print(
        "  Parameters:",
        sum(p.numel() for p in model.parameters())
    )

    print(
        "  Checkpoint SHA256:",
        sha256(spec["checkpoint"])[:24] + "..."
    )

    print(
        "  Scaler SHA256:",
        sha256(spec["scaler"])[:24] + "..."
    )


# =============================================================================
# 6. DEFINE REUSABLE FORECASTING FUNCTION
# =============================================================================

def predict_lstm_horizons(model_name, X):

    if model_name not in FORECAST_MODELS:
        raise ValueError(
            f"Unknown LSTM model: {model_name}"
        )

    model = FORECAST_MODELS[
        model_name
    ]

    scaler = FORECAST_SCALERS[
        model_name
    ]

    X_processed = preprocess_lstm_input(
        X,
        scaler
    )

    tensor = torch.from_numpy(
        X_processed
    ).to(DEVICE)

    with torch.inference_mode():

        probabilities = model(
            tensor
        ).cpu().numpy()

    probabilities = probabilities.astype(
        np.float64
    )

    if probabilities.shape != (len(X), 3):
        raise RuntimeError(
            "Model returned unexpected output shape."
        )

    if not (
        np.isfinite(probabilities).all()
        and (probabilities >= 0).all()
        and (probabilities <= 1).all()
    ):
        raise RuntimeError(
            "Invalid model probabilities."
        )

    if not (
        np.all(
            probabilities[:, 0]
            <= probabilities[:, 1] + 1e-7
        )
        and np.all(
            probabilities[:, 1]
            <= probabilities[:, 2] + 1e-7
        )
    ):
        raise RuntimeError(
            "Model output violates horizon monotonicity."
        )

    return probabilities


# =============================================================================
# 7. VERIFY RESTORED INFERENCE AGAINST SAVED OUTPUTS
#
# This is a small numerical reproduction check,
# not a model comparison or accuracy evaluation.
# No target labels are loaded.
# =============================================================================

VALIDATION_FILE = (
    DATA_DIR / "windows_W10_VALIDATION.npz"
)

with np.load(
    VALIDATION_FILE,
    allow_pickle=False
) as d:

    val_X = d["X"]
    val_episodes = d["episode_id"].astype(str)
    val_minutes = d["minute"].astype(np.int64)

N = len(val_X)

sample_indices = sorted(set([
    0,
    min(1024, N - 1),
    min(7532, N - 1),
    min(15401, N - 1),
    N - 1
]))

replay_X = val_X[
    sample_indices
].astype(np.float32)

replay_episodes = val_episodes[
    sample_indices
]

replay_minutes = val_minutes[
    sample_indices
]

MAX_ALLOWED_DIFFERENCE = 1e-4

print("\n" + "-" * 90)
print("SAVED INFERENCE REPRODUCTION CHECK")
print("-" * 90)

for name, spec in MODEL_SPECS.items():

    df = pd.read_csv(
        spec["predictions"]
    )

    reference = df.set_index(
        ["Episode_ID", "Minute"],
        verify_integrity=True
    )

    reference_keys = pd.MultiIndex.from_arrays(
        [replay_episodes, replay_minutes],
        names=["Episode_ID", "Minute"]
    )

    expected = reference.loc[
        reference_keys,
        ["P_H5", "P_H10", "P_H15"]
    ].to_numpy(dtype=np.float64)

    observed = predict_lstm_horizons(
        name,
        replay_X
    )

    max_difference = float(
        np.max(
            np.abs(observed - expected)
        )
    )

    print(
        f"{name}: max saved-prediction "
        f"difference = {max_difference:.8f}"
    )

    if max_difference > MAX_ALLOWED_DIFFERENCE:

        print(
            "\nExpected first row:",
            expected[0].tolist()
        )

        print(
            "Observed first row:",
            observed[0].tolist()
        )

        raise RuntimeError(
            f"{name} inference does not reproduce "
            "saved validation predictions. "
            "Do not use its forecasts until the "
            "original forward-pass implementation "
            "has been reconciled."
        )

    print(f"{name}: REPRODUCTION PASS")


# =============================================================================
# 8. FIRST ACTUAL FORECAST
# =============================================================================

# This input was validated in Cell 2.
# It contains observed synthetic validation traffic,
# not target labels or stored predictions.

INPUT_X = FORECAST_READY_X

single_probs = predict_lstm_horizons(
    "single_lstm",
    INPUT_X
)[0]

stacked_probs = predict_lstm_horizons(
    "stacked_lstm",
    INPUT_X
)[0]

# Stacked LSTM is our provisional primary
# horizon forecasting source.
PRIMARY_MODEL = "stacked_lstm"

PRIMARY_PROBS = stacked_probs

FORECAST_RESULT = {
    "mode": "synthetic_validation_smoke_test",
    "input_episode": sample_episode,
    "reference_minute": int(sample_end_minute),
    "primary_model": PRIMARY_MODEL,

    "forecast_probabilities": {
        "H5": float(PRIMARY_PROBS[0]),
        "H10": float(PRIMARY_PROBS[1]),
        "H15": float(PRIMARY_PROBS[2])
    },

    "auxiliary_models": {
        "single_lstm": {
            "H5": float(single_probs[0]),
            "H10": float(single_probs[1]),
            "H15": float(single_probs[2])
        }
    },

    "current_attack_status": "UNKNOWN",
    "attack_family_status": "NOT_CONNECTED",
    "onset_estimation_status": "NOT_CONNECTED",
    "alert_policy_status": "NOT_CONFIGURED"
}

print("\n" + "=" * 90)
print("FIRST NETWORK ATTACK FORECAST")
print("=" * 90)

print("Input episode:", sample_episode)
print("Last observed minute:", sample_end_minute)
print("Primary model:", PRIMARY_MODEL)

print("\nFUTURE ATTACK ONSET RISK")

for horizon, probability in zip(
    [5, 10, 15],
    PRIMARY_PROBS
):
    print(
        f"  Next {horizon:02d} minutes: "
        f"{probability * 100:.4f}%"
    )

print(
    "\nMachine-readable forecast:"
)

print(
    json.dumps(
        FORECAST_RESULT,
        indent=2
    )
)

print("\n" + "=" * 90)
print("CELL 3 — FORECAST INFERENCE: COMPLETE")
print("=" * 90)

print("Single LSTM inference: PASS")
print("Stacked LSTM inference: PASS")
print("TRAIN scalers reused: YES")
print("Saved inference reproduction: PASS")
print("Model retraining: NO")
print("Official TEST accessed: NO")

print(
    "\nNEXT: Connect competing-risk inference "
    "for attack family and estimated onset."
)

print("=" * 90)


In [ ]:

# =============================================================================
# NETWORK ATTACK FORECASTING PIPELINE
# CELL 4 — COMPETING-RISK LSTM: ATTACK FAMILY + ONSET ETA
#
# Existing trained model:
#   LSTM_V2_COMPETING_RISK
#
# Classes:
#   0       = NO_ONSET_WITHIN_15_MINUTES
#   1..75   = 5 * (onset_minute - 1) + family_id
#
# Five attack families, fifteen possible onset minutes.
#
# NO RETRAINING
# NO SCALER REFITTING
# NO TEST ACCESS
# =============================================================================

from pathlib import Path
from copy import deepcopy

import json
import numpy as np
import torch
import torch.nn as nn

print("=" * 90)
print("CELL 4 — COMPETING-RISK LSTM INFERENCE")
print("=" * 90)

# =============================================================================
# 1. ORIGINAL TRAINED ARTIFACTS
# =============================================================================

CR_DIR = (
    V2_ROOT
    / "models/LSTM_V2_CompetingRisk"
    / "final_train_model"
)

CR_MODEL_FILE = (
    CR_DIR / "LSTM_V2_CR_STATE.pt"
)

CR_SCALER_FILE = (
    CR_DIR / "LSTM_V2_CR_FULL_TRAIN_SCALER.json"
)

CR_RECORD_FILE = (
    CR_DIR / "LSTM_V2_CR_FINAL_TRAIN_RECORD.json"
)

CR_REFERENCE_FILE = (
    V2_ROOT
    / "models/LSTM_V2_CompetingRisk"
    / "official_validation"
    / "JOINT_76_CLASS_VALIDATION_PREDICTIONS_SAFE.npz"
)

for path in [
    CR_MODEL_FILE,
    CR_SCALER_FILE,
    CR_RECORD_FILE,
    CR_REFERENCE_FILE
]:
    if not path.is_file():
        raise FileNotFoundError(path)

CR_RECORD = json.loads(
    CR_RECORD_FILE.read_text(encoding="utf-8")
)

# Verify exact original artifact hashes.
assert sha256(CR_MODEL_FILE) == CR_RECORD["model_sha256"]
assert sha256(CR_SCALER_FILE) == CR_RECORD["scaler_sha256"]

CR_CONFIG = CR_RECORD["model_architecture"]

CR_FAMILIES = list(
    CR_RECORD["output_definition"]["family_names"]
)

EXPECTED_FAMILIES = [
    "BRUTE_AUTH",
    "DOS_DDOS",
    "EXPLOIT_APP",
    "FUZZING",
    "RECON_SCAN"
]

assert CR_FAMILIES == EXPECTED_FAMILIES

assert CR_CONFIG["num_classes"] == 76
assert CR_CONFIG["input_dim"] == 14

assert CR_RECORD["output_definition"]["class_0"] == (
    "NO_ONSET_WITHIN_15_MINUTES"
)

assert CR_RECORD["output_definition"]["class_1_to_75"] == (
    "CLASS = 5 * (ONSET_MINUTE - 1) + FAMILY_ID"
)

assert CR_RECORD["output_definition"]["onset_minutes"] == list(
    range(1, 16)
)

print("\nTRAIN artifact verification: PASS")
print("Families:", CR_FAMILIES)
print("Output classes:", CR_CONFIG["num_classes"])

# =============================================================================
# 2. RECONSTRUCT ORIGINAL MODEL ARCHITECTURE
# =============================================================================

class CompetingRiskLSTM(nn.Module):

    def __init__(
        self,
        input_dim=14,
        hidden_dim=64,
        dense_dim=32,
        num_layers=2,
        num_classes=76,
        dropout=0.2
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True
        )

        self.projection = nn.Sequential(
            nn.Linear(hidden_dim, dense_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classifier = nn.Linear(
            dense_dim,
            num_classes
        )

    def forward(self, x):

        sequence, _ = self.lstm(x)

        last_hidden = sequence[:, -1, :]

        hidden = self.projection(
            last_hidden
        )

        logits = self.classifier(hidden)

        return logits


CR_MODEL = CompetingRiskLSTM(
    input_dim=CR_CONFIG["input_dim"],
    hidden_dim=CR_CONFIG["hidden_dim"],
    dense_dim=CR_CONFIG["dense_dim"],
    num_layers=CR_CONFIG["num_layers"],
    num_classes=CR_CONFIG["num_classes"],
    dropout=CR_CONFIG["dropout"]
).to(DEVICE)

# The original checkpoint is a plain PyTorch state_dict.
CR_STATE = torch.load(
    CR_MODEL_FILE,
    map_location="cpu",
    weights_only=True
)

CR_MODEL.load_state_dict(
    CR_STATE,
    strict=True
)

CR_MODEL.eval()

# Reuse preprocessing functions from Cell 3.
CR_SCALER = load_train_scaler(
    CR_SCALER_FILE
)

print("\nCompeting-risk LSTM loaded: PASS")
print(
    "Trainable parameters:",
    sum(p.numel() for p in CR_MODEL.parameters())
)
print("TRAIN-fitted scaler restored: PASS")

# =============================================================================
# 3. PREDICT THE FULL 76-CLASS DISTRIBUTION
# =============================================================================

def predict_competing_risk_joint(X):

    X_scaled = preprocess_lstm_input(
        X,
        CR_SCALER
    )

    tensor = torch.from_numpy(
        X_scaled
    ).to(DEVICE)

    with torch.inference_mode():

        logits = CR_MODEL(tensor)

        probabilities = torch.softmax(
            logits,
            dim=-1
        ).cpu().numpy().astype(np.float64)

    if probabilities.shape != (len(X), 76):
        raise RuntimeError(
            "Incorrect competing-risk output shape."
        )

    if not np.isfinite(probabilities).all():
        raise RuntimeError(
            "Non-finite competing-risk probabilities."
        )

    if (probabilities < 0).any():
        raise RuntimeError(
            "Negative competing-risk probabilities."
        )

    if not np.allclose(
        probabilities.sum(axis=1),
        1.0,
        atol=1e-5
    ):
        raise RuntimeError(
            "Competing-risk probabilities do not sum to 1."
        )

    return probabilities


# =============================================================================
# 4. DECODE THE JOINT ONSET-FAMILY DISTRIBUTION
# =============================================================================

def decode_competing_risk(joint):

    joint = np.asarray(
        joint,
        dtype=np.float64
    )

    if joint.shape != (76,):
        raise ValueError(
            "Expected one 76-class probability vector."
        )

    # Class 0 = no onset within 15 minutes.
    p_no_event = float(joint[0])

    # Classes 1..75 correspond to:
    # minute 1: five families
    # minute 2: five families
    # ...
    # minute 15: five families
    #
    # Shape: [15 onset minutes, 5 families]

    time_family = joint[1:].reshape(15, 5)

    # Unconditional onset-minute mass.
    minute_mass = time_family.sum(axis=1)

    # Unconditional attack-family mass.
    family_mass = time_family.sum(axis=0)

    p_event = float(minute_mass.sum())

    p_h5 = float(minute_mass[:5].sum())
    p_h10 = float(minute_mass[:10].sum())
    p_h15 = p_event

    if not (
        -1e-8 <= p_h5 <= p_h10 + 1e-8
        and p_h10 <= p_h15 + 1e-8
        and p_h15 <= 1.0 + 1e-8
    ):
        raise RuntimeError(
            "Invalid cumulative horizon probabilities."
        )

    # Conditional distributions only make mathematical
    # sense when probability of any onset is nonzero.

    if p_event > 1e-12:

        conditional_family = (
            family_mass / p_event
        )

        conditional_minute = (
            minute_mass / p_event
        )

        predicted_family_index = int(
            np.argmax(conditional_family)
        )

        predicted_family = CR_FAMILIES[
            predicted_family_index
        ]

        family_conditional_probability = float(
            conditional_family[predicted_family_index]
        )

        most_likely_minute = int(
            np.argmax(conditional_minute) + 1
        )

        expected_eta = float(
            np.dot(
                np.arange(1, 16),
                conditional_minute
            )
        )

        family_distribution = {
            name: float(probability)
            for name, probability in zip(
                CR_FAMILIES,
                conditional_family
            )
        }

    else:

        predicted_family = None
        family_conditional_probability = None
        most_likely_minute = None
        expected_eta = None

        family_distribution = {
            name: None
            for name in CR_FAMILIES
        }

    return {
        "p_no_onset_within_15": p_no_event,

        "horizon_probabilities": {
            "H5": p_h5,
            "H10": p_h10,
            "H15": p_h15
        },

        "predicted_family_if_onset": predicted_family,

        "predicted_family_probability_if_onset":
            family_conditional_probability,

        "family_distribution_conditional_on_onset":
            family_distribution,

        "most_likely_onset_minute_if_onset":
            most_likely_minute,

        "expected_eta_minutes_if_onset":
            expected_eta,

        "minute_probabilities_unconditional":
            minute_mass.tolist()
    }


# =============================================================================
# 5. VERIFY INFERENCE AGAINST PREVIOUSLY SAVED OUTPUTS
#
# A small exact-replay correctness check.
# Not new model evaluation or model comparison.
# =============================================================================

with np.load(
    CR_REFERENCE_FILE,
    allow_pickle=False
) as reference:

    saved_episodes = reference["episode_id"].astype(str)

    saved_minutes = reference["minute"].astype(np.int64)

    saved_joint = reference[
        "joint_probabilities"
    ]

    saved_horizons = reference[
        "horizon_probabilities"
    ]

assert np.array_equal(
    saved_episodes,
    val_episodes
)

assert np.array_equal(
    saved_minutes,
    val_minutes
)

assert saved_joint.shape == (
    len(val_episodes), 76
)

assert saved_horizons.shape == (
    len(val_episodes), 3
)

# Use the small input subset already established
# in Cell 3, not every validation row.

replayed_joint = predict_competing_risk_joint(
    replay_X
)

expected_joint = saved_joint[
    sample_indices
]

joint_difference = float(
    np.max(
        np.abs(
            replayed_joint - expected_joint
        )
    )
)

# Also verify the decoder against the saved horizons.
decoded_horizons = np.array([
    list(
        decode_competing_risk(p)[
            "horizon_probabilities"
        ].values()
    )
    for p in replayed_joint
])

horizon_difference = float(
    np.max(
        np.abs(
            decoded_horizons
            - saved_horizons[sample_indices]
        )
    )
)

print("\n" + "-" * 90)
print("COMPETING-RISK INFERENCE REPRODUCTION")
print("-" * 90)

print(
    "Maximum joint-probability difference:",
    f"{joint_difference:.10f}"
)

print(
    "Maximum horizon-probability difference:",
    f"{horizon_difference:.10f}"
)

if (
    joint_difference > 1e-4
    or horizon_difference > 1e-4
):
    raise RuntimeError(
        "Competing-risk inference does not reproduce "
        "saved predictions. Stop and verify the original "
        "model forward pass and decoding."
    )

print("Saved inference reproduction: PASS")


# =============================================================================
# 6. BUILD UNIFIED FORECAST FUNCTION
#
# Stacked LSTM remains the provisional primary source
# of H5/H10/H15 probabilities.
#
# Competing-risk LSTM independently supplies a
# joint onset/family probability distribution.
#
# We DO NOT pretend the two outputs form a single
# mathematically calibrated joint model.
# =============================================================================

# A presentation guard, NOT a validated security
# alert threshold. Low-event forecasts should not
# prominently show speculative family/ETA estimates.

MIN_EVENT_PROBABILITY_FOR_DISPLAY = 0.20


def forecast_network_window(traffic_df):

    validated = validate_traffic_history(
        traffic_df
    )

    X = validated["X"]

    primary = predict_lstm_horizons(
        "stacked_lstm",
        X
    )[0]

    auxiliary = predict_lstm_horizons(
        "single_lstm",
        X
    )[0]

    joint = predict_competing_risk_joint(
        X
    )[0]

    competing = decode_competing_risk(
        joint
    )

    cr_event_probability = competing[
        "horizon_probabilities"
    ]["H15"]

    can_display = (
        cr_event_probability
        >= MIN_EVENT_PROBABILITY_FOR_DISPLAY
    )

    if can_display:

        family_status = "AVAILABLE_CONDITIONAL"
        eta_status = "AVAILABLE_CONDITIONAL"

        family_name = competing[
            "predicted_family_if_onset"
        ]

        family_probability = competing[
            "predicted_family_probability_if_onset"
        ]

        expected_eta = competing[
            "expected_eta_minutes_if_onset"
        ]

        most_likely_minute = competing[
            "most_likely_onset_minute_if_onset"
        ]

    else:

        family_status = (
            "WITHHELD_LOW_EVENT_PROBABILITY"
        )

        eta_status = (
            "WITHHELD_LOW_EVENT_PROBABILITY"
        )

        family_name = None
        family_probability = None
        expected_eta = None
        most_likely_minute = None

    report = {

        "mode": "synthetic_benchmark",

        "reference_time": validated[
            "reference_time"
        ],

        "time_type": validated[
            "time_type"
        ],

        "input_shape": list(X.shape),

        "primary_model": "stacked_lstm",

        "forecast_probabilities": {
            "H5": float(primary[0]),
            "H10": float(primary[1]),
            "H15": float(primary[2])
        },

        "auxiliary_models": {

            "single_lstm": {
                "H5": float(auxiliary[0]),
                "H10": float(auxiliary[1]),
                "H15": float(auxiliary[2])
            },

            "competing_risk_lstm": {
                **competing["horizon_probabilities"]
            }
        },

        "attack_family": {

            "source_model": "competing_risk_lstm",

            "status": family_status,

            "predicted_family": family_name,

            "conditional_probability":
                family_probability,

            "conditional_on": (
                "attack_onset_within_15_minutes"
            ),

            "model_event_probability_15min":
                cr_event_probability,

            "family_distribution_if_onset": (
                competing[
                    "family_distribution_conditional_on_onset"
                ]
                if can_display else None
            )
        },

        "estimated_onset": {

            "source_model": "competing_risk_lstm",

            "status": eta_status,

            "conditional_expected_minutes":
                expected_eta,

            "conditional_modal_minute":
                most_likely_minute,

            "conditional_on": (
                "attack_onset_within_15_minutes"
            )
        },

        "current_attack": {
            "status": "UNKNOWN",
            "reason": (
                "Current-attack detector not integrated"
            )
        },

        "alert": {
            "status": "NOT_CONFIGURED",
            "threshold_policy": None
        },

        "limitations": [
            "Synthetic-benchmark forecasting",
            "No confirmed current-attack detection",
            "No real host/IP attribution",
            "Forecast and family/onset outputs come "
            "from different models",
            "Family/ETA visibility gate is provisional"
        ]
    }

    return report


# =============================================================================
# 7. RUN FIRST UNIFIED FORECAST
# =============================================================================

UNIFIED_FORECAST = forecast_network_window(
    sample_df
)

assert UNIFIED_FORECAST["input_shape"] == [
    1, 10, 14
]

assert np.allclose(
    [
        UNIFIED_FORECAST[
            "forecast_probabilities"
        ][f"H{h}"]
        for h in (5, 10, 15)
    ],
    PRIMARY_PROBS,
    atol=1e-6
)

print("\n" + "=" * 90)
print("UNIFIED NETWORK ATTACK FORECAST")
print("=" * 90)

print("Input source: Synthetic V2 validation history")
print(
    "Last observed minute:",
    UNIFIED_FORECAST["reference_time"]
)

print(
    "\nPRIMARY FUTURE RISK — STACKED LSTM"
)

for h in (5, 10, 15):

    probability = UNIFIED_FORECAST[
        "forecast_probabilities"
    ][f"H{h}"]

    print(
        f"H{h:02d}: {probability * 100:.4f}%"
    )

print(
    "\nATTACK FAMILY — COMPETING-RISK LSTM"
)

family = UNIFIED_FORECAST["attack_family"]

print("Status:", family["status"])
print(
    "Competing-risk P(onset within 15 min):",
    f"{family['model_event_probability_15min'] * 100:.4f}%"
)

if family["predicted_family"] is not None:

    print(
        "Likely family, conditional on onset:",
        family["predicted_family"]
    )

    print(
        "Conditional family probability:",
        f"{family['conditional_probability'] * 100:.2f}%"
    )

    print(
        "Estimated ETA, conditional on onset:",
        f"{UNIFIED_FORECAST['estimated_onset']['conditional_expected_minutes']:.2f} minutes"
    )

else:

    print(
        "Family/ETA withheld: onset probability "
        "is below the provisional display gate."
    )

print("\nCURRENT ATTACK STATUS: UNKNOWN")
print("ALERT POLICY: NOT CONFIGURED")

print("\n" + "-" * 90)
print("MACHINE-READABLE FORECAST JSON")
print("-" * 90)

print(
    json.dumps(
        UNIFIED_FORECAST,
        indent=2
    )
)

print("\n" + "=" * 90)
print("CELL 4 — COMPETING-RISK INTEGRATION COMPLETE")
print("=" * 90)

print("Stacked LSTM forecast: CONNECTED")
print("Single-layer LSTM: CONNECTED")
print("Competing-risk LSTM: CONNECTED")
print("Attack-family output: CONNECTED")
print("Conditional ETA output: CONNECTED")
print("Checkpoint reproduction: PASS")

print("\nModel training: NO")
print("Scaler refitting: NO")
print("Official TEST accessed: NO")
print("Files modified: NO")

print("=" * 90)


In [ ]:

# =============================================================================
# NETWORK ATTACK FORECASTING PIPELINE
# CELL 5 — ROLLING NETWORK FORECAST ENGINE
#
# Converts consecutive minute-level traffic records into:
#   10-minute input history
#   -> Stacked LSTM H5/H10/H15 forecast
#   -> Competing-risk attack family and ETA
#   -> Structured forecast output
#
# Features:
#   - One-minute-at-a-time ingestion
#   - Automatic rolling 10-minute buffer
#   - Strict chronology and input validation
#   - Reset at episode/source boundaries
#   - Reset after missing-minute gaps
#   - No forecasts until 10 minutes are available
#   - Synthetic validation streaming demonstration
#
# NO TRAINING
# NO NEW MODEL SELECTION
# NO TEST ACCESS
# NO FILES WRITTEN
# =============================================================================

from collections import deque
import numpy as np
import pandas as pd

print("=" * 90)
print("CELL 5 — ROLLING NETWORK FORECAST ENGINE")
print("=" * 90)


# =============================================================================
# 1. ROLLING FORECASTER
# =============================================================================

class RollingNetworkForecaster:

    def __init__(
        self,
        time_column="minute",
        stream_id="default_network"
    ):

        if time_column not in ("minute", "timestamp"):
            raise ValueError(
                "time_column must be minute or timestamp"
            )

        self.time_column = time_column
        self.stream_id = str(stream_id)

        self.history = deque(
            maxlen=HISTORY_MINUTES
        )

        self.last_time = None
        self.episode_id = None

        self.total_records = 0
        self.forecast_count = 0
        self.reset_count = 0

    # -------------------------------------------------------------------------
    # RESET BUFFER
    # -------------------------------------------------------------------------

    def reset(self, episode_id=None):

        self.history.clear()

        self.last_time = None

        self.episode_id = episode_id

        self.reset_count += 1

    # -------------------------------------------------------------------------
    # VALIDATE A SINGLE OBSERVATION
    # -------------------------------------------------------------------------

    def _validate_record(self, record):

        if isinstance(record, pd.Series):
            record = record.to_dict()

        if not isinstance(record, dict):
            raise TypeError(
                "Each minute must be supplied as a dictionary "
                "or pandas Series."
            )

        required = (
            set(FEATURE_NAMES)
            | {self.time_column}
        )

        missing = required - set(record)
        unexpected = set(record) - required

        if missing:
            raise ValueError(
                f"Missing record fields: {sorted(missing)}"
            )

        if unexpected:
            raise ValueError(
                f"Unexpected record fields: {sorted(unexpected)}"
            )

        # Ensure every input feature is present, finite,
        # nonnegative, and representable as float32.

        values = []

        for feature in FEATURE_NAMES:

            value = record[feature]

            if isinstance(value, (bool, np.bool_)):
                raise ValueError(
                    f"Boolean feature not permitted: {feature}"
                )

            try:
                numeric = float(value)
            except (TypeError, ValueError):
                raise ValueError(
                    f"Non-numeric feature: {feature}"
                )

            if not np.isfinite(numeric):
                raise ValueError(
                    f"Non-finite feature: {feature}"
                )

            if numeric < 0:
                raise ValueError(
                    f"Negative feature: {feature}"
                )

            if numeric > np.finfo(np.float32).max:
                raise ValueError(
                    f"Feature exceeds float32 range: {feature}"
                )

            values.append(np.float32(numeric))

        raw_time = record[self.time_column]

        if self.time_column == "minute":

            if isinstance(raw_time, (bool, np.bool_)):
                raise ValueError(
                    "Minute index cannot be boolean."
                )

            try:
                numeric_time = float(raw_time)
            except (TypeError, ValueError):
                raise ValueError(
                    "Minute index must be numeric."
                )

            if (
                not np.isfinite(numeric_time)
                or not numeric_time.is_integer()
            ):
                raise ValueError(
                    "Minute index must be a finite integer."
                )

            if abs(numeric_time) > 2**53:
                raise ValueError(
                    "Minute index is outside safe integer range."
                )

            normalized_time = int(numeric_time)

        else:

            normalized_time = pd.to_datetime(
                raw_time,
                utc=True,
                errors="raise"
            )

            if pd.isna(normalized_time):
                raise ValueError(
                    "Timestamp cannot be missing."
                )

            if (
                normalized_time.second != 0
                or normalized_time.microsecond != 0
                or normalized_time.nanosecond != 0
            ):
                raise ValueError(
                    "Timestamp must be aligned "
                    "to an exact minute."
                )

        normalized = {
            self.time_column: normalized_time
        }

        normalized.update(
            zip(FEATURE_NAMES, values)
        )

        return normalized

    # -------------------------------------------------------------------------
    # PROCESS NEXT OBSERVED MINUTE
    # -------------------------------------------------------------------------

    def update(
        self,
        minute_record,
        episode_id=None
    ):

        # Validate before changing buffer state.
        record = self._validate_record(
            minute_record
        )

        current_time = record[
            self.time_column
        ]

        episode_changed = (
            episode_id is not None
            and self.episode_id is not None
            and str(episode_id) != self.episode_id
        )

        if episode_changed:

            self.reset(
                episode_id=str(episode_id)
            )

        if episode_id is not None:
            self.episode_id = str(episode_id)

        # Verify chronological continuity.
        if self.last_time is not None:

            if current_time <= self.last_time:

                raise ValueError(
                    "Out-of-order or duplicate minute. "
                    "Record rejected."
                )

            if self.time_column == "minute":

                contiguous = (
                    current_time - self.last_time == 1
                )

            else:

                contiguous = (
                    current_time - self.last_time
                    == pd.Timedelta(minutes=1)
                )

            if not contiguous:

                # Missing minutes invalidate the
                # previous continuous 10-minute history.
                self.reset(
                    episode_id=self.episode_id
                )

                print(
                    "[INFO] Missing-minute gap detected. "
                    "Rolling history restarted."
                )

        # Add the newest traffic record.
        self.history.append(record)

        self.last_time = current_time

        self.total_records += 1

        # No forecast before ten observations.
        if len(self.history) < HISTORY_MINUTES:

            return {
                "status": "WARMUP",
                "stream_id": self.stream_id,
                "episode_id": self.episode_id,
                "records_available": len(self.history),
                "records_required": HISTORY_MINUTES,
                "minutes_remaining": (
                    HISTORY_MINUTES - len(self.history)
                ),
                "forecast": None
            }

        # Convert buffered records to a dataframe.
        traffic_df = pd.DataFrame(
            list(self.history)
        )

        # Reuse actual Cell 2 input validator
        # and Cell 4 unified forecasting engine.
        forecast = forecast_network_window(
            traffic_df
        )

        forecast["stream_id"] = self.stream_id

        forecast["episode_id"] = self.episode_id

        forecast["rolling_history_minutes"] = (
            HISTORY_MINUTES
        )

        forecast["input_origin"] = (
            "OBSERVED_MINUTE_RECORDS"
        )

        self.forecast_count += 1

        return {
            "status": "FORECAST_READY",
            "stream_id": self.stream_id,
            "episode_id": self.episode_id,
            "records_available": HISTORY_MINUTES,
            "forecast": forecast
        }


# =============================================================================
# 2. TEST THE BUFFER USING OUR EXISTING SAMPLE
# =============================================================================

rolling = RollingNetworkForecaster(
    time_column="minute",
    stream_id="synthetic_validation_replay"
)

print("\nInitializing rolling forecaster...")
print("History length:", HISTORY_MINUTES)
print("Features per minute:", len(FEATURE_NAMES))

FIRST_ROLLING_FORECAST = None

for _, row in sample_df.iterrows():

    result = rolling.update(
        row,
        episode_id=sample_episode
    )

    if result["status"] == "FORECAST_READY":

        FIRST_ROLLING_FORECAST = result["forecast"]

assert FIRST_ROLLING_FORECAST is not None

# First rolling forecast must exactly reproduce
# the first unified forecast from Cell 4.

for horizon in ("H5", "H10", "H15"):

    actual = FIRST_ROLLING_FORECAST[
        "forecast_probabilities"
    ][horizon]

    expected = UNIFIED_FORECAST[
        "forecast_probabilities"
    ][horizon]

    if not np.isclose(
        actual,
        expected,
        atol=1e-6,
        rtol=0
    ):
        raise RuntimeError(
            "Rolling inference did not reproduce "
            "the original forecast."
        )

print("\nFirst rolling forecast: PASS")
print("Reference minute:", rolling.last_time)


# =============================================================================
# 3. REPLAY CONSECUTIVE OBSERVED MINUTES
#
# Input source:
# Frozen V2 VALIDATION only.
#
# We read:
#   X
#   episode_id
#   minute
#
# We DO NOT read attack labels.
#
# Each additional window supplies only the
# newly observed last minute. We do not pass
# its precomputed model predictions.
# =============================================================================

VALIDATION_FILE = (
    DATA_DIR / "windows_W10_VALIDATION.npz"
)

with np.load(
    VALIDATION_FILE,
    allow_pickle=False
) as data:

    # Read only the first few sample windows.
    replay_windows = data["X"][:8].astype(
        np.float32
    )

    replay_episode_ids = data[
        "episode_id"
    ][:8].astype(str)

    replay_end_minutes = data[
        "minute"
    ][:8].astype(np.int64)

assert len(replay_windows) >= 2

assert replay_episode_ids[0] == sample_episode

assert replay_end_minutes[0] == sample_end_minute

# Make sure replay stays within the same episode,
# with consecutive minute observations.

valid_replay_count = 1

for i in range(1, len(replay_windows)):

    if (
        replay_episode_ids[i] != sample_episode
        or replay_end_minutes[i]
        != replay_end_minutes[i - 1] + 1
    ):
        break

    valid_replay_count += 1


# =============================================================================
# 4. BUILD THE FORECAST TIMELINE
# =============================================================================

timeline_rows = []


def collect_forecast(forecast):

    risk = forecast[
        "forecast_probabilities"
    ]

    family = forecast[
        "attack_family"
    ]

    onset = forecast[
        "estimated_onset"
    ]

    timeline_rows.append({
        "Minute": forecast["reference_time"],
        "H5_Risk": risk["H5"],
        "H10_Risk": risk["H10"],
        "H15_Risk": risk["H15"],
        "Family": family["predicted_family"],
        "Family_Status": family["status"],
        "ETA_Minutes": onset[
            "conditional_expected_minutes"
        ],
        "CR_Event_Probability": family[
            "model_event_probability_15min"
        ]
    })


collect_forecast(
    FIRST_ROLLING_FORECAST
)

print("\nReplaying consecutive minute observations...")

for i in range(1, valid_replay_count):

    # Extract only the newest observation
    # from each chronological validation window.
    latest_features = replay_windows[
        i, -1, :
    ]

    latest_minute = int(
        replay_end_minutes[i]
    )

    new_record = {
        "minute": latest_minute
    }

    new_record.update({
        feature: float(value)
        for feature, value in zip(
            FEATURE_NAMES,
            latest_features
        )
    })

    result = rolling.update(
        new_record,
        episode_id=sample_episode
    )

    if result["status"] != "FORECAST_READY":
        raise RuntimeError(
            "Expected rolling forecast after warmup."
        )

    collect_forecast(
        result["forecast"]
    )


FORECAST_TIMELINE = pd.DataFrame(
    timeline_rows
)

assert len(FORECAST_TIMELINE) == (
    valid_replay_count
)

assert FORECAST_TIMELINE["Minute"].is_unique

assert np.isfinite(
    FORECAST_TIMELINE[
        ["H5_Risk", "H10_Risk", "H15_Risk"]
    ].to_numpy()
).all()


# =============================================================================
# 5. DISPLAY A READABLE FORECAST TIMELINE
# =============================================================================

display_df = FORECAST_TIMELINE.copy()

for column in [
    "H5_Risk",
    "H10_Risk",
    "H15_Risk",
    "CR_Event_Probability"
]:

    display_df[column] = (
        display_df[column] * 100
    ).round(4)

display_df["ETA_Minutes"] = (
    display_df["ETA_Minutes"].round(2)
)

print("\n" + "=" * 90)
print("ROLLING FUTURE ATTACK FORECAST TIMELINE")
print("=" * 90)

print("Episode:", sample_episode)
print("Model:", PRIMARY_MODEL)
print("Risk values shown as percentages.")

print(
    display_df.to_string(
        index=False
    )
)


# =============================================================================
# 6. LATEST FORECAST
# =============================================================================

latest = FORECAST_TIMELINE.iloc[-1]

print("\n" + "=" * 90)
print("LATEST ROLLING NETWORK FORECAST")
print("=" * 90)

print("Last observed minute:", int(latest["Minute"]))

print(
    f"Next 5 min risk : {latest['H5_Risk'] * 100:.4f}%"
)

print(
    f"Next 10 min risk: {latest['H10_Risk'] * 100:.4f}%"
)

print(
    f"Next 15 min risk: {latest['H15_Risk'] * 100:.4f}%"
)

if pd.notna(latest["Family"]):

    print("Likely attack family:", latest["Family"])

    print(
        "Estimated conditional ETA:",
        f"{latest['ETA_Minutes']:.2f} minutes"
    )

else:

    print(
        "Attack family: WITHHELD "
        "(low competing-risk event probability)"
    )

    print("ETA: WITHHELD")

print("Current attack: UNKNOWN")
print("Alert policy: NOT CONFIGURED")


# =============================================================================
# 7. FINAL STATUS
# =============================================================================

print("\n" + "=" * 90)
print("CELL 5 — ROLLING FORECAST ENGINE: COMPLETE")
print("=" * 90)

print("One-minute ingestion: WORKING")
print("10-minute rolling history: WORKING")
print("Chronological validation: ENABLED")
print("Missing-minute gap reset: ENABLED")
print("Episode reset: ENABLED")

print(
    "Forecasts generated:",
    rolling.forecast_count
)

print(
    "Latest input minute:",
    rolling.last_time
)

print("Stacked LSTM: CONNECTED")
print("Competing-risk LSTM: CONNECTED")
print("Forecast timeline: AVAILABLE")

print("\nTraining performed: NO")
print("Official TEST accessed: NO")
print("Files written: NO")

print("=" * 90)


In [ ]:

# =============================================================================
# NETWORK ATTACK FORECASTING PIPELINE
# CELL 6 — FUTURE-RISK POLICY AND ALERT ENGINE
#
# Uses existing Stacked LSTM H5/H10/H15 predictions.
#
# Adds:
#   - Risk levels
#   - Early-warning decisions
#   - Severity escalation
#   - Alert cooldown / duplicate suppression
#   - Alert reset after sustained low risk
#   - Episode and time-gap isolation
#   - Rolling forecast integration
#
# IMPORTANT:
# Thresholds below are PROVISIONAL operational defaults.
# They are NOT validated deployment thresholds.
#
# NO TRAINING / NO TEST ACCESS / NO FILE WRITES
# =============================================================================

from copy import deepcopy
from dataclasses import dataclass
import numpy as np
import pandas as pd

print("=" * 90)
print("CELL 6 — NETWORK FORECAST ALERT ENGINE")
print("=" * 90)


# =============================================================================
# 1. DEFINE EXPLICIT POLICY
# =============================================================================

@dataclass(frozen=True)
class ForecastAlertConfig:

    watch_h15: float = 0.20

    warning_h15: float = 0.50

    high_h10: float = 0.50

    critical_h5: float = 0.50

    cooldown_minutes: int = 5

    recovery_minutes: int = 3

    policy_version: str = (
        "PROVISIONAL_FORECAST_POLICY_V1"
    )


ALERT_CONFIG = ForecastAlertConfig()

SEVERITY_RANK = {
    "LOW": 0,
    "WATCH": 1,
    "WARNING": 2,
    "HIGH": 3,
    "CRITICAL": 4
}

# WARNING and above can generate alerts.
# WATCH is displayed but does not dispatch an alert.

print("\nPolicy:", ALERT_CONFIG.policy_version)
print("WATCH if H15 >=", ALERT_CONFIG.watch_h15)
print("WARNING if H15 >=", ALERT_CONFIG.warning_h15)
print("HIGH if H10 >=", ALERT_CONFIG.high_h10)
print("CRITICAL if H5 >=", ALERT_CONFIG.critical_h5)
print("Alert cooldown:", ALERT_CONFIG.cooldown_minutes, "minutes")
print("Recovery:", ALERT_CONFIG.recovery_minutes, "low-risk minutes")

print("\nThresholds are provisional: YES")
print("Current attack detection enabled: NO")


# =============================================================================
# 2. RISK CLASSIFICATION
# =============================================================================

def classify_future_risk(probabilities):

    p = np.asarray(
        [
            probabilities["H5"],
            probabilities["H10"],
            probabilities["H15"]
        ],
        dtype=np.float64
    )

    if (
        not np.isfinite(p).all()
        or (p < 0).any()
        or (p > 1).any()
    ):
        raise ValueError(
            "Invalid forecast probabilities."
        )

    if not (
        p[0] <= p[1] + 1e-7
        and p[1] <= p[2] + 1e-7
    ):
        raise ValueError(
            "Forecast horizon monotonicity violated."
        )

    h5, h10, h15 = p

    if h5 >= ALERT_CONFIG.critical_h5:

        return {
            "level": "CRITICAL",
            "trigger_horizon": "H5",
            "message": (
                "Elevated predicted attack-onset "
                "risk within 5 minutes."
            )
        }

    if h10 >= ALERT_CONFIG.high_h10:

        return {
            "level": "HIGH",
            "trigger_horizon": "H10",
            "message": (
                "Elevated predicted attack-onset "
                "risk within 10 minutes."
            )
        }

    if h15 >= ALERT_CONFIG.warning_h15:

        return {
            "level": "WARNING",
            "trigger_horizon": "H15",
            "message": (
                "Elevated predicted attack-onset "
                "risk within 15 minutes."
            )
        }

    if h15 >= ALERT_CONFIG.watch_h15:

        return {
            "level": "WATCH",
            "trigger_horizon": "H15",
            "message": (
                "Forecast risk is above the "
                "provisional monitoring threshold."
            )
        }

    return {
        "level": "LOW",
        "trigger_horizon": None,
        "message": (
            "Future attack-onset risk is below "
            "the configured warning thresholds."
        )
    }


# =============================================================================
# 3. STATEFUL ALERT ENGINE
# =============================================================================

class ForecastAlertEngine:

    def __init__(self, config=ALERT_CONFIG):

        self.config = config

        # Independent state per stream.
        self.states = {}

    def _time_key(self, forecast):

        value = forecast["reference_time"]

        if forecast["time_type"] == "relative_minute_index":

            return int(value)

        if forecast["time_type"] == "utc_timestamp":

            timestamp = pd.Timestamp(value)

            if timestamp.tzinfo is None:
                raise ValueError(
                    "Forecast timestamp requires a timezone."
                )

            timestamp = timestamp.tz_convert("UTC")

            return int(
                timestamp.value // (60 * 10**9)
            )

        raise ValueError(
            "Unsupported forecast time representation."
        )

    def evaluate(self, forecast):

        if not isinstance(forecast, dict):
            raise TypeError(
                "Forecast must be a dictionary."
            )

        if forecast.get("primary_model") != "stacked_lstm":
            raise ValueError(
                "This policy is configured for stacked_lstm."
            )

        risk = classify_future_risk(
            forecast["forecast_probabilities"]
        )

        severity = SEVERITY_RANK[
            risk["level"]
        ]

        stream_id = str(
            forecast.get("stream_id", "default_network")
        )

        episode_id = str(
            forecast.get("episode_id")
        )

        current_minute = self._time_key(
            forecast
        )

        state = self.states.get(stream_id)

        reset_reason = None

        if state is not None:

            if state["episode_id"] != episode_id:

                state = None
                reset_reason = "EPISODE_CHANGED"

            elif current_minute <= state["last_minute"]:

                raise ValueError(
                    "Duplicate or out-of-order alert evaluation."
                )

            elif current_minute != state["last_minute"] + 1:

                state = None
                reset_reason = "TIME_GAP"

        if state is None:

            state = {
                "episode_id": episode_id,
                "last_minute": None,
                "last_alert_minute": None,
                "last_alert_severity": 0,
                "below_warning_streak": 0,
                "emitted_count": 0
            }

        decision = "NO_ALERT"

        reason = "Risk below warning threshold"

        # -----------------------------------------------------
        # 4. LOW/WATCH STATES AND RECOVERY
        # -----------------------------------------------------

        if severity < SEVERITY_RANK["WARNING"]:

            state["below_warning_streak"] += 1

            if (
                state["below_warning_streak"]
                >= self.config.recovery_minutes
                and state["last_alert_severity"] > 0
            ):

                decision = "RESOLVED"

                reason = (
                    "Risk remained below warning level "
                    "for the required recovery period."
                )

                state["last_alert_severity"] = 0
                state["last_alert_minute"] = None

            elif risk["level"] == "WATCH":

                decision = "MONITOR"

                reason = (
                    "Monitoring threshold reached; "
                    "no alert dispatched."
                )

        # -----------------------------------------------------
        # 5. WARNING/HIGH/CRITICAL STATES
        # -----------------------------------------------------

        else:

            state["below_warning_streak"] = 0

            last_alert_minute = state[
                "last_alert_minute"
            ]

            last_alert_severity = state[
                "last_alert_severity"
            ]

            new_alert = last_alert_minute is None

            escalated = (
                severity > last_alert_severity
            )

            cooldown_expired = (
                last_alert_minute is not None
                and current_minute - last_alert_minute
                >= self.config.cooldown_minutes
            )

            if new_alert or escalated or cooldown_expired:

                decision = "EMIT_ALERT"

                if new_alert:
                    reason = "New forecasting warning"

                elif escalated:
                    reason = "Forecast risk severity escalated"

                else:
                    reason = "Cooldown expired; warning persists"

                state["last_alert_minute"] = (
                    current_minute
                )

                state["last_alert_severity"] = (
                    severity
                )

                state["emitted_count"] += 1

            else:

                decision = "SUPPRESSED"

                reason = (
                    "Existing warning remains active; "
                    "duplicate alert suppressed."
                )

        state["last_minute"] = current_minute

        self.states[stream_id] = state

        result = deepcopy(forecast)

        result["alert"] = {
            "policy_version": self.config.policy_version,
            "policy_status": "PROVISIONAL_NOT_DEPLOYMENT_VALIDATED",
            "decision": decision,
            "risk_level": risk["level"],
            "trigger_horizon": risk["trigger_horizon"],
            "message": risk["message"],
            "reason": reason,
            "stream_id": stream_id,
            "total_alerts_emitted": state["emitted_count"],
            "state_reset_reason": reset_reason,
            "forecast_only": True,
            "confirmed_current_attack": False,
            "cooldown_minutes": self.config.cooldown_minutes,
            "thresholds": {
                "watch_h15": self.config.watch_h15,
                "warning_h15": self.config.warning_h15,
                "high_h10": self.config.high_h10,
                "critical_h5": self.config.critical_h5
            }
        }

        # Important: EMIT_ALERT is a decision output.
        # It does not mean an email, webhook or security
        # notification was actually delivered.

        return result


# =============================================================================
# 6. CONNECT THE ALERT ENGINE TO ROLLING FORECASTS
# =============================================================================

class RollingForecastWithAlerts:

    def __init__(
        self,
        time_column="minute",
        stream_id="default_network"
    ):

        self.rolling = RollingNetworkForecaster(
            time_column=time_column,
            stream_id=stream_id
        )

        self.alerts = ForecastAlertEngine()

    def update(self, record, episode_id=None):

        result = self.rolling.update(
            record,
            episode_id=episode_id
        )

        if result["status"] != "FORECAST_READY":
            return result

        enhanced_forecast = self.alerts.evaluate(
            result["forecast"]
        )

        result["forecast"] = enhanced_forecast

        return result


# =============================================================================
# 7. REPLAY EXISTING SYNTHETIC VALIDATION MINUTES
# =============================================================================

engine = RollingForecastWithAlerts(
    time_column="minute",
    stream_id="synthetic_validation_replay"
)

ALERT_TIMELINE_ROWS = []

# Feed the first ten observed minutes.
for _, row in sample_df.iterrows():

    response = engine.update(
        row.to_dict(),
        episode_id=sample_episode
    )

    if response["status"] == "FORECAST_READY":

        forecast = response["forecast"]

        ALERT_TIMELINE_ROWS.append({
            "Minute": forecast["reference_time"],
            "H5_%": 100 * forecast[
                "forecast_probabilities"
            ]["H5"],
            "H10_%": 100 * forecast[
                "forecast_probabilities"
            ]["H10"],
            "H15_%": 100 * forecast[
                "forecast_probabilities"
            ]["H15"],
            "Risk_Level": forecast[
                "alert"
            ]["risk_level"],
            "Alert_Decision": forecast[
                "alert"
            ]["decision"]
        })

# Continue using only the new final minute of each
# subsequent saved synthetic validation window.
for i in range(1, valid_replay_count):

    record = {
        "minute": int(
            replay_end_minutes[i]
        )
    }

    record.update({
        feature: float(value)
        for feature, value in zip(
            FEATURE_NAMES,
            replay_windows[i, -1, :]
        )
    })

    response = engine.update(
        record,
        episode_id=sample_episode
    )

    assert response["status"] == "FORECAST_READY"

    forecast = response["forecast"]

    ALERT_TIMELINE_ROWS.append({
        "Minute": forecast["reference_time"],
        "H5_%": 100 * forecast[
            "forecast_probabilities"
        ]["H5"],
        "H10_%": 100 * forecast[
            "forecast_probabilities"
        ]["H10"],
        "H15_%": 100 * forecast[
            "forecast_probabilities"
        ]["H15"],
        "Risk_Level": forecast[
            "alert"
        ]["risk_level"],
        "Alert_Decision": forecast[
            "alert"
        ]["decision"]
    })


ALERT_TIMELINE = pd.DataFrame(
    ALERT_TIMELINE_ROWS
)

assert len(ALERT_TIMELINE) == valid_replay_count

# Check that alerts used the exact existing forecasts,
# not changed or recomputed probability definitions.
assert np.allclose(
    ALERT_TIMELINE["H5_%"].to_numpy() / 100,
    FORECAST_TIMELINE["H5_Risk"].to_numpy(),
    atol=1e-6
)

assert np.allclose(
    ALERT_TIMELINE["H10_%"].to_numpy() / 100,
    FORECAST_TIMELINE["H10_Risk"].to_numpy(),
    atol=1e-6
)

assert np.allclose(
    ALERT_TIMELINE["H15_%"].to_numpy() / 100,
    FORECAST_TIMELINE["H15_Risk"].to_numpy(),
    atol=1e-6
)

print("\n" + "=" * 90)
print("ROLLING FORECAST WITH RISK AND ALERT DECISIONS")
print("=" * 90)

print(
    ALERT_TIMELINE.round(4).to_string(
        index=False
    )
)


# =============================================================================
# 8. SMALL ALERT STATE-MACHINE CHECK
#
# Handcrafted probabilities only test software logic.
# They are NOT model predictions.
# =============================================================================

test_engine = ForecastAlertEngine()


def test_forecast(minute, h5, h10, h15):

    return {
        "reference_time": minute,
        "time_type": "relative_minute_index",
        "primary_model": "stacked_lstm",
        "stream_id": "alert_policy_unit_test",
        "episode_id": "TEST_POLICY_LOGIC_ONLY",
        "forecast_probabilities": {
            "H5": h5,
            "H10": h10,
            "H15": h15
        }
    }


cases = [
    (1, 0.01, 0.05, 0.10, "NO_ALERT"),
    (2, 0.01, 0.10, 0.60, "EMIT_ALERT"),
    (3, 0.01, 0.10, 0.65, "SUPPRESSED"),
    (4, 0.01, 0.60, 0.70, "EMIT_ALERT"),
    (5, 0.60, 0.70, 0.80, "EMIT_ALERT"),
    (6, 0.01, 0.05, 0.10, "NO_ALERT"),
    (7, 0.01, 0.05, 0.10, "NO_ALERT"),
    (8, 0.01, 0.05, 0.10, "RESOLVED"),
    (9, 0.01, 0.10, 0.60, "EMIT_ALERT")
]

for minute, h5, h10, h15, expected in cases:

    result = test_engine.evaluate(
        test_forecast(
            minute,
            h5,
            h10,
            h15
        )
    )

    observed = result["alert"]["decision"]

    assert observed == expected, (
        f"Minute {minute}: "
        f"expected {expected}, got {observed}"
    )

print("\nAlert escalation/cooldown/recovery checks: PASS")


# =============================================================================
# 9. LATEST RESULT
# =============================================================================

LATEST_ALERT_FORECAST = response["forecast"]

print("\n" + "=" * 90)
print("LATEST NETWORK FORECAST + ALERT STATE")
print("=" * 90)

risk = LATEST_ALERT_FORECAST[
    "forecast_probabilities"
]

alert = LATEST_ALERT_FORECAST["alert"]

print(
    "Reference minute:",
    LATEST_ALERT_FORECAST["reference_time"]
)

for horizon in ("H5", "H10", "H15"):

    print(
        f"{horizon}: "
        f"{risk[horizon] * 100:.4f}%"
    )

print("\nRisk level:", alert["risk_level"])
print("Alert decision:", alert["decision"])
print("Reason:", alert["reason"])

print(
    "Alerts emitted in replay:",
    alert["total_alerts_emitted"]
)

print(
    "\nNote: decisions are provisional and "
    "no external notifications were sent."
)

print("\n" + "=" * 90)
print("CELL 6 — FORECAST ALERT ENGINE: COMPLETE")
print("=" * 90)

print("Rolling forecast: CONNECTED")
print("Future-risk classification: CONNECTED")
print("Alert escalation: ENABLED")
print("Duplicate suppression: ENABLED")
print("Recovery handling: ENABLED")
print("Current attack detection: NOT CONNECTED")
print("External alert delivery: NOT CONNECTED")

print("\nTraining performed: NO")
print("Official TEST accessed: NO")
print("Files written: NO")

print("=" * 90)


In [ ]:

# =============================================================================
# NETWORK ATTACK FORECASTING PIPELINE
# CELL 7 — ANALYST REPORT AND FORECAST ARTIFACT EXPORT
#
# Produces:
#   1. Structured forecast JSON
#   2. Human-readable Markdown analyst report
#   3. Rolling forecast timeline CSV
#   4. Forecast risk timeline PNG
#
# Sources: Actual inference outputs from Cells 3–6.
#
# No model training
# No threshold optimization
# No official TEST access
# No modification of trained models or frozen datasets
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
from dataclasses import asdict
from copy import deepcopy
from uuid import uuid4

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

print("=" * 90)
print("CELL 7 — NETWORK ATTACK FORECAST REPORT GENERATOR")
print("=" * 90)

# =============================================================================
# 1. ESTABLISH REPORT OUTPUT DIRECTORY
# =============================================================================

OUTPUT_DIR = (
    ROOT
    / "ForecastPipeline_Outputs"
    / "synthetic_validation_demo"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

generated_at = datetime.now(
    timezone.utc
).isoformat()

report_id = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_"
    + uuid4().hex[:8]
)

print("\nReport ID:", report_id)
print("Output directory:", OUTPUT_DIR)

# =============================================================================
# 2. VERIFY THE LATEST COMPLETED FORECAST
# =============================================================================

FINAL_FORECAST = deepcopy(
    LATEST_ALERT_FORECAST
)

required_forecast_sections = [
    "forecast_probabilities",
    "attack_family",
    "estimated_onset",
    "current_attack",
    "alert",
    "primary_model",
    "reference_time"
]

for section in required_forecast_sections:

    if section not in FINAL_FORECAST:
        raise RuntimeError(
            f"Missing forecast section: {section}"
        )

risk = FINAL_FORECAST[
    "forecast_probabilities"
]

family = FINAL_FORECAST[
    "attack_family"
]

onset = FINAL_FORECAST[
    "estimated_onset"
]

alert = FINAL_FORECAST[
    "alert"
]

risk_values = np.array(
    [risk["H5"], risk["H10"], risk["H15"]],
    dtype=np.float64
)

assert np.isfinite(risk_values).all()

assert np.all(
    (risk_values >= 0)
    & (risk_values <= 1)
)

assert (
    risk_values[0]
    <= risk_values[1] + 1e-7
    <= risk_values[2] + 2e-7
)

assert FINAL_FORECAST["primary_model"] == "stacked_lstm"

# Ensure the risk policy did not alter model probabilities.
last_timeline = FORECAST_TIMELINE.iloc[-1]

assert np.allclose(
    risk_values,
    last_timeline[
        ["H5_Risk", "H10_Risk", "H15_Risk"]
    ].to_numpy(dtype=np.float64),
    rtol=0,
    atol=1e-6
)

# Verify the latest alert matches the risk policy.
expected_risk = classify_future_risk(risk)

assert alert["risk_level"] == expected_risk["level"]

print("\nLatest forecast integrity: PASS")
print("Primary model:", FINAL_FORECAST["primary_model"])
print("Latest minute:", FINAL_FORECAST["reference_time"])


# =============================================================================
# 3. OBSERVED TRAFFIC EVIDENCE
#
# Use the actual final 10-minute rolling buffer,
# not invented explanatory features.
# =============================================================================

history_df = pd.DataFrame(
    list(engine.rolling.history)
)

validated_history = validate_traffic_history(
    history_df
)

assert validated_history["X"].shape == (1, 10, 14)

assert (
    validated_history["reference_time"]
    == FINAL_FORECAST["reference_time"]
)

TRAFFIC_EVIDENCE = []

for feature in FEATURE_NAMES:

    values = history_df[
        feature
    ].to_numpy(dtype=np.float64)

    evidence = {
        "feature": feature,
        "first_observed": float(values[0]),
        "last_observed": float(values[-1]),
        "absolute_change": float(
            values[-1] - values[0]
        ),
        "minimum": float(values.min()),
        "maximum": float(values.max()),
        "mean": float(values.mean())
    }

    TRAFFIC_EVIDENCE.append(evidence)

print("Observed traffic evidence: VERIFIED")
print("Traffic features summarized:", len(TRAFFIC_EVIDENCE))

# =============================================================================
# 4. MODEL AND POLICY PROVENANCE
# =============================================================================

PROVENANCE = {}

for name in [
    "single_lstm",
    "stacked_lstm",
    "competing_risk_lstm"
]:

    entry = MODEL_REGISTRY[name]

    if not entry["checkpoint_verified"]:
        raise RuntimeError(
            f"Unverified model in report: {name}"
        )

    PROVENANCE[name] = {
        "checkpoint_path": entry["path"],
        "checkpoint_sha256": entry["sha256"],
        "checkpoint_verified": True
    }

PROVENANCE["single_lstm"]["scaler_sha256"] = sha256(
    MODEL_SPECS["single_lstm"]["scaler"]
)

PROVENANCE["stacked_lstm"]["scaler_sha256"] = sha256(
    MODEL_SPECS["stacked_lstm"]["scaler"]
)

PROVENANCE["competing_risk_lstm"]["scaler_sha256"] = sha256(
    CR_SCALER_FILE
)

POLICY_RECORD = {
    "configuration": asdict(ALERT_CONFIG),
    "validated_for_deployment": False,
    "external_notifications_connected": False,
    "current_attack_detection_connected": False,
    "family_eta_display_gate_provisional": True,
    "minimum_competing_risk_event_probability_for_display":
        MIN_EVENT_PROBABILITY_FOR_DISPLAY
}

# =============================================================================
# 5. STRUCTURED TIMELINE
# =============================================================================

timeline = ALERT_TIMELINE.copy()

assert len(timeline) > 0

assert timeline["Minute"].is_unique

assert int(timeline.iloc[-1]["Minute"]) == int(
    FINAL_FORECAST["reference_time"]
)

# Keep percentages in the CSV with explicit column names.
timeline = timeline.rename(
    columns={
        "H5_%": "H5_Risk_Percent",
        "H10_%": "H10_Risk_Percent",
        "H15_%": "H15_Risk_Percent"
    }
)

# =============================================================================
# 6. ASSEMBLE MACHINE-READABLE REPORT
# =============================================================================

REPORT_DATA = {
    "schema_version": "network-attack-forecast-report/v1",
    "report_id": report_id,
    "generated_at_utc": generated_at,

    "environment": {
        "mode": "synthetic_validation_replay",
        "dataset": "Synthetic_NAF_v2",
        "data_split": "VALIDATION",
        "is_live_network_data": False,
        "is_real_ip_topology": False
    },

    "input": {
        "stream_id": FINAL_FORECAST.get("stream_id"),
        "episode_id": FINAL_FORECAST.get("episode_id"),
        "reference_time": FINAL_FORECAST["reference_time"],
        "time_type": FINAL_FORECAST["time_type"],
        "history_minutes": HISTORY_MINUTES,
        "feature_names": list(FEATURE_NAMES),
        "shape": list(validated_history["X"].shape)
    },

    "forecast": deepcopy(
        FINAL_FORECAST["forecast_probabilities"]
    ),

    "forecast_source": {
        "primary_model": FINAL_FORECAST["primary_model"],
        "model_family": "LSTM",
        "task": "CUMULATIVE_FUTURE_ATTACK_ONSET"
    },

    "auxiliary_predictions": deepcopy(
        FINAL_FORECAST.get("auxiliary_models", {})
    ),

    "attack_family": deepcopy(family),
    "estimated_onset": deepcopy(onset),
    "current_attack": deepcopy(
        FINAL_FORECAST["current_attack"]
    ),
    "alert": deepcopy(alert),

    "traffic_evidence": TRAFFIC_EVIDENCE,

    "rolling_timeline": timeline.to_dict(
        orient="records"
    ),

    "model_provenance": PROVENANCE,
    "alert_policy": POLICY_RECORD,

    "limitations": [
        "Generated from synthetic validation traffic.",
        "Not validated on live network telemetry.",
        "Current attack presence is unknown.",
        "Alert thresholds are provisional.",
        "No external notification was sent.",
        "Family and ETA are conditional model estimates.",
        "Independent model probabilities are not fused.",
        "Traffic feature changes are descriptive, "
        "not causal attack explanations.",
        "No real host or IP attribution is available."
    ]
}

# Reject invalid NaN/Infinity values at export time.
JSON_TEXT = json.dumps(
    REPORT_DATA,
    indent=2,
    allow_nan=False,
    default=lambda obj: (
        obj.item()
        if isinstance(obj, np.generic)
        else (
            obj.isoformat()
            if isinstance(obj, (datetime, pd.Timestamp))
            else str(obj)
        )
    )
)

# Ensure serialized report is parseable.
json.loads(JSON_TEXT)

print("Structured report serialization: PASS")


# =============================================================================
# 7. HUMAN-READABLE REPORT CONTENT
# =============================================================================

def percent(probability):
    return f"{100 * float(probability):.4f}%"


def describe_family(family_record):

    if family_record["predicted_family"] is None:

        return (
            "Withheld because the competing-risk "
            "model's onset probability is below "
            "the provisional display threshold."
        )

    return (
        f"{family_record['predicted_family']} "
        f"({100 * family_record['conditional_probability']:.2f}% "
        "conditional family probability)"
    )


def describe_eta(onset_record):

    eta = onset_record[
        "conditional_expected_minutes"
    ]

    if eta is None:
        return "Withheld"

    return (
        f"Approximately {eta:.2f} minutes, "
        "conditional on attack onset within 15 minutes"
    )


family_text = describe_family(family)
eta_text = describe_eta(onset)

cr_h15 = family[
    "model_event_probability_15min"
]

# Human-readable table of observed feature changes.
evidence_lines = [
    "| Observed feature | First minute | Latest minute | Change |",
    "|---|---:|---:|---:|"
]

for entry in TRAFFIC_EVIDENCE:

    evidence_lines.append(
        "| "
        + entry["feature"]
        + f" | {entry['first_observed']:.6g}"
        + f" | {entry['last_observed']:.6g}"
        + f" | {entry['absolute_change']:+.6g} |"
    )

evidence_table = "\n".join(evidence_lines)

chart_filename = report_id + "_risk_timeline.png"

MARKDOWN_REPORT = f"""# Network Attack Forecast Report

**Report ID:** `{report_id}`

**Generated (UTC):** {generated_at}

**Source:** Synthetic NAF V2 validation replay

**Last observed minute:** {FINAL_FORECAST['reference_time']}

**Primary forecasting model:** Stacked LSTM

---

## 1. Future Attack Onset Forecast

| Prediction horizon | Attack onset probability |
|---|---:|
| Next 5 minutes (H5) | **{percent(risk['H5'])}** |
| Next 10 minutes (H10) | **{percent(risk['H10'])}** |
| Next 15 minutes (H15) | **{percent(risk['H15'])}** |

These are cumulative probabilities of future attack onset.
They are not probabilities that an attack is occurring now.

## 2. Risk and Early-Warning Decision

**Risk level:** {alert['risk_level']}

**Alert decision:** {alert['decision']}

**Policy:** {alert['policy_version']}

**Explanation:** {alert['reason']}

The policy is provisional and not deployment-calibrated.
No notification was sent to an external security system.

## 3. Attack-Family Forecast

**Status:** {family['status']}

**Prediction:** {family_text}

**Competing-risk model P(onset within 15 minutes):**
{percent(cr_h15)}

This comes from the competing-risk LSTM, which is
separate from the primary Stacked LSTM horizon model.

## 4. Estimated Attack Onset

**Status:** {onset['status']}

**Conditional ETA:** {eta_text}

An ETA is an estimate conditional on future attack
onset, not a guaranteed event time.

## 5. Current Attack Detection

**Status: UNKNOWN**

A current-attack detector is not yet connected.

Low future risk does not prove the network is
currently attack-free.

## 6. Rolling Forecast History

![Rolling forecast risk timeline]({chart_filename})

The chart shows the model's forecasts as consecutive
minutes of synthetic network traffic were observed.

## 7. Observed Network Traffic Evidence

{evidence_table}

Values are reported in the frozen model-input feature
units. These are observed descriptive changes, not
proof of a particular attack mechanism.

## 8. Model Provenance

| Model | Role | Checkpoint SHA256 |
|---|---|---|
| Stacked LSTM | Primary H5/H10/H15 | `{PROVENANCE['stacked_lstm']['checkpoint_sha256'][:24]}...` |
| Single-layer LSTM | Auxiliary | `{PROVENANCE['single_lstm']['checkpoint_sha256'][:24]}...` |
| Competing-risk LSTM | Family/ETA | `{PROVENANCE['competing_risk_lstm']['checkpoint_sha256'][:24]}...` |

All checkpoints are loaded from previously trained
and frozen model artifacts. No model was retrained.

## 9. Security Analyst Interpretation

The primary forecasting model estimates a
{percent(risk['H15'])} probability of attack onset
within the next 15 minutes, using the latest ten
minutes of synthetic traffic.

The current risk-policy classification is
**{alert['risk_level']}**, with decision
**{alert['decision']}**.

Attack family and timing are reported only where
supported by the competing-risk model.

These predictions provide an early-warning
estimate, not confirmation that an attack
is happening or will definitely happen.

## 10. Important Limitations

- This is synthetic-benchmark inference, not live
  network monitoring.
- Thresholds require operational calibration.
- Current-attack detection is not connected.
- No real IP or host attribution is available.
- Family/ETA estimates are conditional.
- Model probabilities are not fused.
- No RAG-grounded explanation has been added yet.
- Official TEST data remains untouched.
"""


# =============================================================================
# 8. GENERATE THE ACTUAL RISK TIMELINE CHART
# =============================================================================

FIGURE_PATH = (
    OUTPUT_DIR / chart_filename
)

fig, ax = plt.subplots(
    figsize=(10, 5)
)

for horizon in [
    "H5_Risk_Percent",
    "H10_Risk_Percent",
    "H15_Risk_Percent"
]:

    ax.plot(
        timeline["Minute"],
        timeline[horizon],
        marker="o",
        linewidth=1.8,
        label=horizon.replace(
            "_Risk_Percent", ""
        )
    )

ax.set_title(
    "Rolling Network Attack Onset Forecast"
)

ax.set_xlabel(
    "Last observed minute"
)

ax.set_ylabel(
    "Predicted onset probability (%)"
)

ax.grid(
    True,
    alpha=0.3
)

ax.legend()

fig.tight_layout()

fig.savefig(
    FIGURE_PATH,
    dpi=160,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# =============================================================================
# 9. SAVE REPORTS
# =============================================================================

JSON_PATH = (
    OUTPUT_DIR
    / f"{report_id}_forecast.json"
)

MARKDOWN_PATH = (
    OUTPUT_DIR
    / f"{report_id}_report.md"
)

CSV_PATH = (
    OUTPUT_DIR
    / f"{report_id}_timeline.csv"
)

JSON_PATH.write_text(
    JSON_TEXT,
    encoding="utf-8"
)

MARKDOWN_PATH.write_text(
    MARKDOWN_REPORT,
    encoding="utf-8"
)

timeline.to_csv(
    CSV_PATH,
    index=False
)

# =============================================================================
# 10. VERIFY SAVED OUTPUTS
# =============================================================================

for path in [
    JSON_PATH,
    MARKDOWN_PATH,
    CSV_PATH,
    FIGURE_PATH
]:

    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(
            f"Report output missing or empty: {path}"
        )

with JSON_PATH.open(
    "r",
    encoding="utf-8"
) as stream:

    verified_json = json.load(stream)

assert verified_json["report_id"] == report_id

assert verified_json["forecast"] == REPORT_DATA["forecast"]

verified_timeline = pd.read_csv(
    CSV_PATH
)

assert len(verified_timeline) == len(
    timeline
)

print("\nSaved report integrity: PASS")


# =============================================================================
# 11. DISPLAY THE ANALYST REPORT
# =============================================================================

print("\n" + "=" * 90)
print("GENERATED ANALYST REPORT")
print("=" * 90)

display(
    Markdown(MARKDOWN_REPORT)
)


# =============================================================================
# 12. FINAL OUTPUT PATHS AND STATUS
# =============================================================================

print("\n" + "=" * 90)
print("CELL 7 — FINAL REPORT ARTIFACTS CREATED")
print("=" * 90)

print("\nJSON:", JSON_PATH)
print("\nMarkdown:", MARKDOWN_PATH)
print("\nCSV:", CSV_PATH)
print("\nPNG:", FIGURE_PATH)

print("\nForecast probabilities: INCLUDED")
print("Attack-family prediction: INCLUDED")
print("Conditional ETA: INCLUDED")
print("Current-attack status: INCLUDED")
print("Alert/risk decision: INCLUDED")
print("Traffic evidence: INCLUDED")
print("Model provenance: INCLUDED")
print("Rolling timeline: INCLUDED")

print("\nFrozen dataset modified: NO")
print("Model checkpoints modified: NO")
print("Model training performed: NO")
print("Official TEST accessed: NO")

print("\nCELL 7 STATUS: COMPLETE")
print("=" * 90)


In [ ]:

# =============================================================================
# NETWORK ATTACK FORECASTING PIPELINE
# CELL 8 — RESTORE AND INTEGRATE XGBOOST H5/H10
#
# Loads the original final TRAIN models:
#   XGBoost_V2_FLAT140_H5.json
#   XGBoost_V2_FLAT140_H10.json
#
# Representation:
#   (10, 14) -> log1p -> flattened (140,)
#
# Integration:
#   XGBoost becomes an AUXILIARY prediction source.
#   Stacked LSTM remains the primary H5/H10/H15 model.
#
# NO RETRAINING
# NO FUSION
# NO TEST ACCESS
# NO FILE WRITES
# =============================================================================

import json
import numpy as np
import pandas as pd
import xgboost as xgb

print("=" * 90)
print("CELL 8 — XGBOOST FORECAST INFERENCE")
print("=" * 90)

print("XGBoost version:", xgb.__version__)

# =============================================================================
# 1. LOCATE EXACT SAVED TRAIN ARTIFACTS
# =============================================================================

XGB_DIR = (
    V2_ROOT
    / "models/XGBoost_V2_FLAT140"
    / "final_train_models"
)

XGB_REGISTRY_FILE = (
    XGB_DIR / "XGBOOST_V2_FINAL_TRAIN_REGISTRY.json"
)

XGB_RECORD_FILES = {
    "H5": XGB_DIR / "XGBoost_V2_FLAT140_H5_TRAIN_RECORD.json",
    "H10": XGB_DIR / "XGBoost_V2_FLAT140_H10_TRAIN_RECORD.json"
}

if not XGB_REGISTRY_FILE.is_file():
    raise FileNotFoundError(XGB_REGISTRY_FILE)

XGB_TRAIN_REGISTRY = json.loads(
    XGB_REGISTRY_FILE.read_text(encoding="utf-8")
)

assert XGB_TRAIN_REGISTRY["representation"] == "FLAT140_LOG1P"
assert XGB_TRAIN_REGISTRY["training_status"] == (
    "FINAL_FULL_TRAIN_COMPLETED"
)

XGB_MODELS = {}
XGB_RECORDS = {}
XGB_MODEL_PATHS = {}

# =============================================================================
# 2. LOAD AND VERIFY TRAINED MODELS
# =============================================================================

for horizon in ("H5", "H10"):

    registry_entry = XGB_TRAIN_REGISTRY["models"][horizon]

    checkpoint = (
        XGB_DIR / registry_entry["model_file"]
    )

    record_path = XGB_RECORD_FILES[horizon]

    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)

    if not record_path.is_file():
        raise FileNotFoundError(record_path)

    record = json.loads(
        record_path.read_text(encoding="utf-8")
    )

    expected_hash = registry_entry["model_sha256"]

    actual_hash = sha256(checkpoint)

    if actual_hash != expected_hash:
        raise RuntimeError(
            f"{horizon}: checkpoint SHA256 mismatch"
        )

    if record["model_sha256"] != actual_hash:
        raise RuntimeError(
            f"{horizon}: TRAIN record hash mismatch"
        )

    assert record["representation"] == "FLAT140_LOG1P"
    assert record["input_shape"] == [10, 14]
    assert record["forecast_horizon_minutes"] == int(
        horizon[1:]
    )

    expected_features = [
        f"t_minus_{9-t}__{feature}"
        for t in range(10)
        for feature in FEATURE_NAMES
    ]

    if record["feature_names"] != expected_features:
        raise RuntimeError(
            f"{horizon}: frozen feature order mismatch"
        )

    model = xgb.Booster()

    model.load_model(
        str(checkpoint)
    )

    if model.num_features() != 140:
        raise RuntimeError(
            f"{horizon}: model does not expect 140 features"
        )

    # If feature names are embedded in the model,
    # ensure that they agree with the training record.
    stored_feature_names = model.feature_names

    if (
        stored_feature_names is not None
        and list(stored_feature_names) != expected_features
    ):
        raise RuntimeError(
            f"{horizon}: embedded model feature names mismatch"
        )

    XGB_MODELS[horizon] = model
    XGB_RECORDS[horizon] = record
    XGB_MODEL_PATHS[horizon] = checkpoint

    print(f"\n{horizon}: LOADED")
    print("  Model:", checkpoint.name)
    print("  Features:", model.num_features())
    print("  Trees:", model.num_boosted_rounds())
    print("  SHA256:", actual_hash[:24] + "...")

print("\nXGBoost TRAIN artifact verification: PASS")

# =============================================================================
# 3. IMPLEMENT ORIGINAL FLAT140_LOG1P INPUT
# =============================================================================

def preprocess_xgboost_input(X):

    X = np.asarray(
        X,
        dtype=np.float32
    )

    if X.ndim != 3 or X.shape[1:] != (10, 14):
        raise ValueError(
            f"Expected (batch,10,14), received {X.shape}"
        )

    if not np.isfinite(X).all() or (X < 0).any():
        raise ValueError(
            "XGBoost input contains invalid traffic values"
        )

    # No scaling or refitting: original FLAT140_LOG1P.
    transformed = np.log1p(X).reshape(
        len(X), 140
    ).astype(np.float32)

    return transformed


# =============================================================================
# 4. PREDICT H5 AND H10
# =============================================================================

def predict_xgboost_horizons(X):

    flat = preprocess_xgboost_input(X)

    feature_names = XGB_RECORDS["H5"]["feature_names"]

    matrix = xgb.DMatrix(
        flat,
        feature_names=feature_names
    )

    predictions = {}

    for horizon in ("H5", "H10"):

        probability = XGB_MODELS[horizon].predict(
            matrix
        )

        probability = np.asarray(
            probability,
            dtype=np.float64
        ).reshape(-1)

        if probability.shape != (len(flat),):
            raise RuntimeError(
                f"{horizon}: unexpected prediction shape"
            )

        if (
            not np.isfinite(probability).all()
            or (probability < 0).any()
            or (probability > 1).any()
        ):
            raise RuntimeError(
                f"{horizon}: invalid probabilities"
            )

        predictions[horizon] = probability

    return predictions


# =============================================================================
# 5. VERIFY AGAINST SAVED OFFICIAL VALIDATION PREDICTIONS
#
# Numerical replay only.
# No target labels, AP calculation or tuning.
# =============================================================================

XGB_VALIDATION_PREDICTIONS = (
    V2_ROOT
    / "models/XGBoost_V2_FLAT140"
    / "official_validation"
    / "VALIDATION_PREDICTIONS.parquet"
)

if not XGB_VALIDATION_PREDICTIONS.is_file():
    raise FileNotFoundError(
        XGB_VALIDATION_PREDICTIONS
    )

saved_df = pd.read_parquet(
    XGB_VALIDATION_PREDICTIONS,
    columns=[
        "Episode_ID",
        "Minute",
        "H5_Probability",
        "H10_Probability"
    ]
)

saved_df = saved_df.set_index(
    ["Episode_ID", "Minute"],
    verify_integrity=True
)

replay_keys = pd.MultiIndex.from_arrays(
    [
        replay_episodes,
        replay_minutes
    ],
    names=["Episode_ID", "Minute"]
)

expected = saved_df.loc[
    replay_keys,
    ["H5_Probability", "H10_Probability"]
].to_numpy(dtype=np.float64)

observed = predict_xgboost_horizons(
    replay_X
)

observed_matrix = np.column_stack([
    observed["H5"],
    observed["H10"]
])

max_difference = float(
    np.max(
        np.abs(observed_matrix - expected)
    )
)

print("\n" + "-" * 90)
print("XGBOOST SAVED-PREDICTION REPRODUCTION")
print("-" * 90)

print(
    "Maximum probability difference:",
    f"{max_difference:.10f}"
)

if max_difference > 1e-4:

    print("Expected:", expected[0].tolist())
    print("Observed:", observed_matrix[0].tolist())

    raise RuntimeError(
        "XGBoost inference did not reproduce the "
        "original saved predictions. "
        "Do not enable its adapter."
    )

print("Saved inference reproduction: PASS")


# =============================================================================
# 6. INTEGRATE INTO EXISTING UNIFIED FORECASTER
#
# Keep original forecasting implementation intact.
# Wrap it with the verified XGBoost adapter.
# =============================================================================

# Preserve the original Cell 4 implementation,
# including when this cell is executed again.

if "_CELL8_ORIGINAL_FORECAST" not in globals():

    _CELL8_ORIGINAL_FORECAST = forecast_network_window


def forecast_network_window(traffic_df):

    # Original forecasting engine from Cell 4:
    # Stacked LSTM + Single LSTM + competing-risk LSTM.

    result = _CELL8_ORIGINAL_FORECAST(
        traffic_df
    )

    # Same validated history, independent XGBoost inference.
    validated = validate_traffic_history(
        traffic_df
    )

    xgb_probs = predict_xgboost_horizons(
        validated["X"]
    )

    result["auxiliary_models"]["xgboost"] = {
        "H5": float(xgb_probs["H5"][0]),
        "H10": float(xgb_probs["H10"][0]),
        "H15": None,
        "availability": "H5_AND_H10_ONLY"
    }

    result["limitations"].append(
        "XGBoost supplies H5/H10 only and is "
        "not fused into the primary forecast."
    )

    return result


# =============================================================================
# 7. UPDATE MODEL REGISTRY
# =============================================================================

MODEL_REGISTRY["xgboost"] = {
    "name": "xgboost",
    "exists": True,
    "checkpoint_verified": True,
    "representation": "FLAT140_LOG1P",
    "horizons": [5, 10],
    "models": {
        horizon: {
            "path": str(XGB_MODEL_PATHS[horizon]),
            "sha256": sha256(XGB_MODEL_PATHS[horizon]),
            "features": 140
        }
        for horizon in ("H5", "H10")
    }
}

# =============================================================================
# 8. GENERATE FIRST INTEGRATED FORECAST
# =============================================================================

XGB_INTEGRATED_FORECAST = forecast_network_window(
    sample_df
)

primary = XGB_INTEGRATED_FORECAST[
    "forecast_probabilities"
]

xgb_auxiliary = XGB_INTEGRATED_FORECAST[
    "auxiliary_models"
]["xgboost"]

# Verify that integrating XGBoost did not alter
# the original Stacked LSTM forecast.

for horizon in ("H5", "H10", "H15"):

    assert np.isclose(
        primary[horizon],
        UNIFIED_FORECAST[
            "forecast_probabilities"
        ][horizon],
        atol=1e-7,
        rtol=0
    )

print("\n" + "=" * 90)
print("NETWORK FORECAST — XGBOOST CONNECTED")
print("=" * 90)

print("Reference minute:", sample_end_minute)

print("\nPRIMARY MODEL — STACKED LSTM")

for horizon in ("H5", "H10", "H15"):

    print(
        f"  {horizon}: {primary[horizon]*100:.4f}%"
    )

print("\nAUXILIARY MODEL — XGBOOST")

for horizon in ("H5", "H10"):

    print(
        f"  {horizon}: "
        f"{xgb_auxiliary[horizon]*100:.4f}%"
    )

print("  H15: NOT AVAILABLE")

print("\nPrimary forecasting model unchanged: YES")
print("Independent model probabilities fused: NO")

# =============================================================================
# 9. SMOKE TEST WITH ROLLING FORECAST ENGINE
# =============================================================================

# The existing RollingNetworkForecaster calls the
# global forecast_network_window function.
# It now automatically receives XGBoost auxiliary outputs.

xgb_rolling = RollingNetworkForecaster(
    time_column="minute",
    stream_id="xgboost_integration_smoke_test"
)

rolling_result = None

for _, row in sample_df.iterrows():

    rolling_result = xgb_rolling.update(
        row.to_dict(),
        episode_id=sample_episode
    )

assert rolling_result["status"] == "FORECAST_READY"

assert "xgboost" in rolling_result[
    "forecast"
]["auxiliary_models"]

print("\nRolling engine XGBoost integration: PASS")

# =============================================================================
# 10. FINAL STATUS
# =============================================================================

print("\n" + "=" * 90)
print("CELL 8 — XGBOOST INTEGRATION: COMPLETE")
print("=" * 90)

print("XGBoost H5 checkpoint: VERIFIED")
print("XGBoost H10 checkpoint: VERIFIED")
print("Saved prediction reproduction: PASS")
print("Unified forecasting function: UPDATED")
print("Rolling forecasting engine: CONNECTED")

print("\nPrimary Stacked LSTM unchanged: YES")
print("Retraining performed: NO")
print("Official TEST accessed: NO")
print("Files written: NO")

print("\nNEXT: Competing-risk TCN and graph-model adapters.")

print("=" * 90)


In [ ]:

# =============================================================================
# NETWORK ATTACK FORECASTING PIPELINE
# CELL 10 — COMPETING-RISK TCN INFERENCE AND INTEGRATION
#
# Original architecture:
#   3 causal residual TCN blocks
#   Channels: 32
#   Kernel size: 3
#   Dilations: 1, 2, 4
#   Output: 76 joint onset-minute / attack-family classes
#
# Verification:
#   - Original checkpoint and TRAIN scaler hashes
#   - Exact architecture/state_dict restoration
#   - Numerical replay against saved VALIDATION predictions
#   - Primary forecast remains unchanged
#
# NO TRAINING / NO REFITTING / NO TEST ACCESS / NO FILE WRITES
# =============================================================================

from pathlib import Path
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=" * 90)
print("CELL 10 — COMPETING-RISK TCN INTEGRATION")
print("=" * 90)

# =============================================================================
# 1. ORIGINAL TRAINED MODEL ARTIFACTS
# =============================================================================

TCN_DIR = (
    V2_ROOT
    / "models"
    / "TCN_V2_CompetingRisk"
    / "final_train_model"
)

TCN_VALIDATION_DIR = (
    V2_ROOT
    / "models"
    / "TCN_V2_CompetingRisk"
    / "official_validation"
)

TCN_REGISTRY_FILE = (
    TCN_DIR / "TCN_V2_CR_MODEL_REGISTRY.json"
)

TCN_RECORD_FILE = (
    TCN_DIR / "TCN_V2_CR_FINAL_TRAIN_RECORD.json"
)

TCN_REFERENCE_FILE = (
    TCN_VALIDATION_DIR
    / "JOINT_76_CLASS_VALIDATION_PREDICTIONS.npz"
)

TCN_VALIDATION_HASH_FILE = (
    TCN_VALIDATION_DIR
    / "ARTIFACT_SHA256SUMS.json"
)

for path in [
    TCN_REGISTRY_FILE,
    TCN_RECORD_FILE,
    TCN_REFERENCE_FILE,
    TCN_VALIDATION_HASH_FILE
]:
    if not path.is_file():
        raise FileNotFoundError(path)

TCN_REGISTRY = json.loads(
    TCN_REGISTRY_FILE.read_text(encoding="utf-8")
)

TCN_RECORD = json.loads(
    TCN_RECORD_FILE.read_text(encoding="utf-8")
)

TCN_VALIDATION_HASHES = json.loads(
    TCN_VALIDATION_HASH_FILE.read_text(encoding="utf-8")
)

TCN_MODEL_FILE = (
    TCN_DIR / TCN_REGISTRY["model_file"]
)

TCN_SCALER_FILE = (
    TCN_DIR / TCN_REGISTRY["scaler_file"]
)

for path in [TCN_MODEL_FILE, TCN_SCALER_FILE]:
    if not path.is_file():
        raise FileNotFoundError(path)

# =============================================================================
# 2. VERIFY FROZEN ARTIFACT INTEGRITY
# =============================================================================

assert sha256(TCN_MODEL_FILE) == (
    TCN_REGISTRY["model_sha256"]
)

assert sha256(TCN_MODEL_FILE) == (
    TCN_RECORD["model_sha256"]
)

assert sha256(TCN_SCALER_FILE) == (
    TCN_REGISTRY["scaler_sha256"]
)

assert sha256(TCN_SCALER_FILE) == (
    TCN_RECORD["scaler_sha256"]
)

assert sha256(TCN_RECORD_FILE) == (
    TCN_REGISTRY["training_record_sha256"]
)

assert sha256(TCN_REFERENCE_FILE) == (
    TCN_VALIDATION_HASHES[
        TCN_REFERENCE_FILE.name
    ]
)

assert TCN_RECORD["training_split"] == "TRAIN"
assert TCN_RECORD["training_status"] == "COMPLETED"
assert TCN_RECORD["official_test_used"] is False
assert TCN_RECORD["input_shape"] == [10, 14]

assert TCN_RECORD["feature_names"] == list(FEATURE_NAMES)

assert TCN_RECORD["output_definition"]["family_names"] == (
    CR_FAMILIES
)

assert TCN_RECORD["output_definition"]["onset_minutes"] == (
    list(range(1, 16))
)

TCN_CONFIG = TCN_RECORD["model_architecture"]

assert TCN_CONFIG == {
    "channels": 32,
    "dense_dim": 32,
    "dropout": 0.2,
    "input_dim": 14,
    "kernel_size": 3,
    "num_blocks": 3,
    "num_classes": 76
}

print("\nTRAIN checkpoint integrity: PASS")
print("TRAIN scaler integrity: PASS")
print("Saved validation artifact integrity: PASS")
print("Feature contract: PASS")
print("76-class output contract: PASS")

# =============================================================================
# 3. RESTORE ORIGINAL CAUSAL CONVOLUTION
# =============================================================================

class TCN_CausalConv1d(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
        dilation=1
    ):
        super().__init__()

        self.left_padding = (
            (kernel_size - 1) * dilation
        )

        self.conv = nn.Conv1d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=0
        )

    def forward(self, x):

        x = F.pad(
            x,
            (self.left_padding, 0)
        )

        return self.conv(x)


# =============================================================================
# 4. RESTORE ORIGINAL TEMPORAL RESIDUAL BLOCK
# =============================================================================

class TCN_TemporalResidualBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        dilation,
        dropout
    ):
        super().__init__()

        self.conv1 = TCN_CausalConv1d(
            in_channels,
            out_channels,
            kernel_size,
            dilation
        )

        self.conv2 = TCN_CausalConv1d(
            out_channels,
            out_channels,
            kernel_size,
            dilation
        )

        self.relu1 = nn.ReLU()
        self.relu2 = nn.ReLU()

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.residual = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size=1
            )
        )

    def forward(self, x):

        residual = self.residual(x)

        out = self.conv1(x)
        out = self.relu1(out)
        out = self.dropout1(out)

        out = self.conv2(out)
        out = self.relu2(out)
        out = self.dropout2(out)

        return F.relu(out + residual)


# =============================================================================
# 5. RESTORE ORIGINAL COMPETING-RISK TCN ARCHITECTURE
# =============================================================================

class TCNCompetingRisk(nn.Module):

    def __init__(
        self,
        input_dim=14,
        channels=32,
        num_blocks=3,
        kernel_size=3,
        dropout=0.2,
        dense_dim=32,
        num_classes=76
    ):
        super().__init__()

        layers = []
        in_channels = input_dim

        for block_idx in range(num_blocks):

            dilation = 2 ** block_idx

            layers.append(
                TCN_TemporalResidualBlock(
                    in_channels=in_channels,
                    out_channels=channels,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout
                )
            )

            in_channels = channels

        self.tcn = nn.Sequential(*layers)

        self.projection = nn.Sequential(
            nn.Linear(channels, dense_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classifier = nn.Linear(
            dense_dim,
            num_classes
        )

    def encode_sequence(self, x):

        if x.ndim != 3 or x.shape[-1] != 14:
            raise ValueError(
                "Expected input shape (batch,10,14)"
            )

        x = x.transpose(1, 2)

        features = self.tcn(x)

        return features.transpose(1, 2)

    def forward(self, x):

        features = self.encode_sequence(x)

        last_step = features[:, -1, :]

        projected = self.projection(last_step)

        return self.classifier(projected)


# =============================================================================
# 6. LOAD SAVED MODEL AND TRAIN SCALER
# =============================================================================

TCN_MODEL = TCNCompetingRisk(
    **TCN_CONFIG
).to(DEVICE)

TCN_STATE = torch.load(
    TCN_MODEL_FILE,
    map_location="cpu",
    weights_only=True
)

TCN_MODEL.load_state_dict(
    TCN_STATE,
    strict=True
)

TCN_MODEL.eval()
TCN_MODEL.requires_grad_(False)

TCN_SCALER = load_train_scaler(
    TCN_SCALER_FILE
)

parameter_count = sum(
    p.numel()
    for p in TCN_MODEL.parameters()
)

assert parameter_count == 20940

print("\n" + "-" * 90)
print("TCN ARCHITECTURE RESTORED")
print("-" * 90)

print("Temporal blocks:", TCN_CONFIG["num_blocks"])
print("Channels:", TCN_CONFIG["channels"])
print("Kernel size:", TCN_CONFIG["kernel_size"])
print("Dilations:", [1, 2, 4])
print("Parameters:", parameter_count)
print("Output classes:", TCN_CONFIG["num_classes"])
print("Device:", DEVICE)
print("Model loading: PASS")

# =============================================================================
# 7. COMPETING-RISK TCN INFERENCE FUNCTION
# =============================================================================

def predict_tcn_joint(X):

    X = np.asarray(
        X,
        dtype=np.float32
    )

    # Reuse the full-TRAIN preprocessing from Cell 3:
    # log1p -> saved median/IQR -> clip [-8,8].
    #
    # The TCN's own TRAIN scaler has been restored.
    # No statistics are fitted on validation data.

    X_scaled = preprocess_lstm_input(
        X,
        TCN_SCALER
    )

    tensor = torch.from_numpy(
        np.ascontiguousarray(X_scaled)
    ).to(DEVICE)

    with torch.inference_mode():

        logits = TCN_MODEL(tensor)

        probabilities = torch.softmax(
            logits,
            dim=-1
        ).cpu().numpy().astype(np.float64)

    expected_shape = (len(X), 76)

    if probabilities.shape != expected_shape:
        raise RuntimeError(
            f"Expected {expected_shape}, got "
            f"{probabilities.shape}"
        )

    if not np.isfinite(probabilities).all():
        raise RuntimeError(
            "Non-finite TCN predictions"
        )

    if (probabilities < 0).any():
        raise RuntimeError(
            "Negative TCN probabilities"
        )

    if not np.allclose(
        probabilities.sum(axis=1),
        1.0,
        atol=1e-6
    ):
        raise RuntimeError(
            "TCN probabilities do not sum to 1"
        )

    return probabilities


# =============================================================================
# 8. DERIVE ATTACK RISK, FAMILY AND ETA
# =============================================================================

def predict_tcn_forecast(X):

    joint_matrix = predict_tcn_joint(X)

    results = []

    for joint in joint_matrix:

        # Reuse the verified 76-class decoder from Cell 4.
        decoded = decode_competing_risk(joint)

        event_probability = decoded[
            "horizon_probabilities"
        ]["H15"]

        show_family_eta = (
            event_probability
            >= MIN_EVENT_PROBABILITY_FOR_DISPLAY
        )

        results.append({

            "H5": float(
                decoded["horizon_probabilities"]["H5"]
            ),

            "H10": float(
                decoded["horizon_probabilities"]["H10"]
            ),

            "H15": float(
                decoded["horizon_probabilities"]["H15"]
            ),

            "p_no_onset_within_15": float(
                decoded["p_no_onset_within_15"]
            ),

            "predicted_family": (
                decoded["predicted_family_if_onset"]
                if show_family_eta else None
            ),

            "conditional_family_probability": (
                decoded[
                    "predicted_family_probability_if_onset"
                ]
                if show_family_eta else None
            ),

            "conditional_family_distribution": (
                decoded[
                    "family_distribution_conditional_on_onset"
                ]
                if show_family_eta else None
            ),

            "conditional_expected_eta_minutes": (
                decoded["expected_eta_minutes_if_onset"]
                if show_family_eta else None
            ),

            "conditional_modal_onset_minute": (
                decoded[
                    "most_likely_onset_minute_if_onset"
                ]
                if show_family_eta else None
            ),

            "family_eta_status": (
                "AVAILABLE_CONDITIONAL"
                if show_family_eta
                else "WITHHELD_LOW_EVENT_PROBABILITY"
            ),

            "family_eta_conditional_on": (
                "attack_onset_within_15_minutes"
            ),

            "source_model": "competing_risk_tcn",

            "primary_forecast": False
        })

    return results


# =============================================================================
# 9. REPRODUCE SAVED VALIDATION INFERENCE
#
# No ground-truth targets are read.
# Only saved model predictions and row identifiers.
# =============================================================================

print("\n" + "-" * 90)
print("SAVED TCN INFERENCE REPRODUCTION")
print("-" * 90)

with np.load(
    TCN_REFERENCE_FILE,
    allow_pickle=False
) as saved:

    required_keys = {
        "joint_probabilities",
        "horizon_probabilities",
        "episode_id",
        "minute"
    }

    if not required_keys.issubset(set(saved.files)):
        raise RuntimeError(
            "Saved TCN prediction archive has "
            "unexpected keys."
        )

    reference_joint = saved[
        "joint_probabilities"
    ][sample_indices]

    reference_horizons = saved[
        "horizon_probabilities"
    ][sample_indices]

    # Verify exact row identity before comparing.
    saved_episodes = saved[
        "episode_id"
    ].astype(str)

    saved_minutes = saved[
        "minute"
    ].astype(np.int64)

    assert np.array_equal(
        saved_episodes,
        val_episodes
    )

    assert np.array_equal(
        saved_minutes,
        val_minutes
    )

    assert reference_joint.shape == (
        len(sample_indices), 76
    )

    assert reference_horizons.shape == (
        len(sample_indices), 3
    )

replayed_joint = predict_tcn_joint(
    replay_X
)

replayed_horizons = (
    replayed_joint[:, 1:]
    .reshape(-1, 15, 5)
    .sum(axis=2)
    .cumsum(axis=1)[:, [4, 9, 14]]
)

joint_difference = float(
    np.max(
        np.abs(
            replayed_joint - reference_joint
        )
    )
)

horizon_difference = float(
    np.max(
        np.abs(
            replayed_horizons - reference_horizons
        )
    )
)

print(
    "Maximum joint probability difference:",
    f"{joint_difference:.10f}"
)

print(
    "Maximum horizon probability difference:",
    f"{horizon_difference:.10f}"
)

# Matches the tolerance used for numerical
# replay of the original TCN implementation.

REPLAY_TOLERANCE = 2e-6

if (
    joint_difference > REPLAY_TOLERANCE
    or horizon_difference > REPLAY_TOLERANCE
):

    print(
        "\nExpected first saved horizon:",
        reference_horizons[0].tolist()
    )

    print(
        "Replayed first horizon:",
        replayed_horizons[0].tolist()
    )

    raise RuntimeError(
        "Competing-risk TCN failed saved-inference "
        "reproduction. Integration stopped."
    )

print("TCN saved inference reproduction: PASS")


# =============================================================================
# 10. INTEGRATE INTO THE UNIFIED FORECAST FUNCTION
#
# IMPORTANT:
# Existing sources are preserved:
#   - Primary: Stacked LSTM
#   - Auxiliary: Single LSTM
#   - Auxiliary: Competing-risk LSTM
#   - Auxiliary: XGBoost H5/H10
#
# New:
#   - Auxiliary: Competing-risk TCN
#
# No model fusion or primary-source replacement.
# =============================================================================

if "_CELL10_ORIGINAL_FORECAST" not in globals():

    _CELL10_ORIGINAL_FORECAST = (
        forecast_network_window
    )


def forecast_network_window(traffic_df):

    result = _CELL10_ORIGINAL_FORECAST(
        traffic_df
    )

    validated = validate_traffic_history(
        traffic_df
    )

    tcn_result = predict_tcn_forecast(
        validated["X"]
    )[0]

    result["auxiliary_models"][
        "competing_risk_tcn"
    ] = tcn_result

    if (
        "TCN probabilities are independent of "
        "the primary stacked-LSTM forecast."
        not in result["limitations"]
    ):

        result["limitations"].append(
            "TCN probabilities are independent of "
            "the primary stacked-LSTM forecast."
        )

    return result


# =============================================================================
# 11. UPDATE EXISTING MODEL REGISTRY
# =============================================================================

if "competing_risk_tcn" not in MODEL_REGISTRY:
    MODEL_REGISTRY["competing_risk_tcn"] = {}

MODEL_REGISTRY["competing_risk_tcn"].update({

    "name": "competing_risk_tcn",

    "path": str(TCN_MODEL_FILE),

    "sha256": sha256(TCN_MODEL_FILE),

    "scaler_path": str(TCN_SCALER_FILE),

    "scaler_sha256": sha256(TCN_SCALER_FILE),

    "checkpoint_verified": True,

    "inference_connected": True,

    "input_shape": [10, 14],

    "output_classes": 76,

    "horizons": [5, 10, 15],

    "role": "auxiliary_competing_risk"
})


# =============================================================================
# 12. VERIFY EXISTING FORECAST IS UNCHANGED
# =============================================================================

old_forecast = _CELL10_ORIGINAL_FORECAST(
    sample_df
)

new_forecast = forecast_network_window(
    sample_df
)

assert new_forecast["forecast_probabilities"] == (
    old_forecast["forecast_probabilities"]
)

assert new_forecast["primary_model"] == "stacked_lstm"

for model_name in old_forecast["auxiliary_models"]:

    assert (
        new_forecast["auxiliary_models"][model_name]
        == old_forecast["auxiliary_models"][model_name]
    )

assert "competing_risk_tcn" in (
    new_forecast["auxiliary_models"]
)

print("\nPrimary Stacked LSTM unchanged: PASS")
print("Existing auxiliary predictions unchanged: PASS")
print("TCN added to unified forecast: PASS")


# =============================================================================
# 13. DISPLAY TCN FORECAST ON A REAL SYNTHETIC WINDOW
#
# If Cell 9 has an episode replay, demonstrate the
# newest window in that episode.
#
# Otherwise use the original minute-9 sample.
# No label information is used.
# =============================================================================

demo_df = sample_df
demo_episode = sample_episode

if (
    "LATEST_EXPLORER_REPLAY" in globals()
    and LATEST_EXPLORER_REPLAY is not None
):

    demo_episode = LATEST_EXPLORER_REPLAY[
        "episode"
    ]

    latest_index = EPISODE_INDEX[
        demo_episode
    ][-1]

    demo_df = get_window_dataframe(
        latest_index
    )

demo_forecast = forecast_network_window(
    demo_df
)

primary = demo_forecast[
    "forecast_probabilities"
]

tcn = demo_forecast[
    "auxiliary_models"
]["competing_risk_tcn"]

print("\n" + "=" * 90)
print("NETWORK ATTACK FORECAST — TCN CONNECTED")
print("=" * 90)

print("Episode:", demo_episode)

print(
    "Last observed minute:",
    demo_forecast["reference_time"]
)

print("\nPRIMARY: STACKED LSTM")

for horizon in ("H5", "H10", "H15"):

    print(
        f"  {horizon}: "
        f"{primary[horizon] * 100:.4f}%"
    )

print("\nAUXILIARY: COMPETING-RISK TCN")

for horizon in ("H5", "H10", "H15"):

    print(
        f"  {horizon}: "
        f"{tcn[horizon] * 100:.4f}%"
    )

print(
    "\nTCN family/ETA status:",
    tcn["family_eta_status"]
)

if tcn["predicted_family"] is not None:

    print(
        "TCN predicted family:",
        tcn["predicted_family"]
    )

    print(
        "TCN conditional family probability:",
        f"{tcn['conditional_family_probability'] * 100:.2f}%"
    )

    print(
        "TCN conditional expected ETA:",
        f"{tcn['conditional_expected_eta_minutes']:.2f} minutes"
    )

else:

    print(
        "TCN family: WITHHELD"
    )

    print(
        "TCN conditional ETA: WITHHELD"
    )

print("\nIndependent models fused: NO")
print("Alert policy changed: NO")


# =============================================================================
# 14. SMOKE TEST THROUGH ROLLING FORECAST ENGINE
# =============================================================================

tcn_rolling = RollingNetworkForecaster(
    time_column="minute",
    stream_id="tcn_integration_smoke_test"
)

rolling_result = None

for _, row in demo_df.iterrows():

    rolling_result = tcn_rolling.update(
        row.to_dict(),
        episode_id=demo_episode
    )

assert rolling_result["status"] == "FORECAST_READY"

assert "competing_risk_tcn" in (
    rolling_result["forecast"]["auxiliary_models"]
)

print("\nRolling TCN inference integration: PASS")


# =============================================================================
# 15. FINAL STATUS
# =============================================================================

print("\n" + "=" * 90)
print("CELL 10 — COMPETING-RISK TCN INTEGRATION: COMPLETE")
print("=" * 90)

print("Original TCN architecture: RESTORED")
print("Checkpoint SHA256: VERIFIED")
print("TRAIN scaler SHA256: VERIFIED")
print("Validation prediction replay: PASS")

print("\nTCN H5/H10/H15: CONNECTED")
print("TCN attack-family inference: CONNECTED")
print("TCN conditional ETA: CONNECTED")
print("Unified forecasting engine: UPDATED")
print("Rolling forecasting engine: UPDATED")
print("Primary Stacked LSTM: UNCHANGED")
print("Existing alert policy: UNCHANGED")

print("\nTraining performed: NO")
print("Scaler refitting: NO")
print("Official TEST accessed: NO")
print("Files modified: NO")

print("\nNEXT: Node-aware and graph-model integration.")
print("=" * 90)


In [ ]:

# =============================================================================
# NETWORK ATTACK FORECASTING PIPELINE
# CELL 11 — NODE-AWARE LSTM INFERENCE AND INTEGRATION
#
# Original input: (batch, 10 minutes, 8 nodes, 8 features)
#
# Original architecture:
#   Shared node LSTM, hidden_dim=32
#   Node attention pooling
#   Monotonic H5/H10/H15 forecasting head
#
# Uses the existing SYNTHETIC graph companion.
# Graph inputs are NOT real hosts or IP connections.
#
# NO RETRAINING / NO SCALER FITTING / NO TEST ACCESS
# =============================================================================

import json
import hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

print("=" * 90)
print("CELL 11 — NODE-AWARE LSTM INTEGRATION")
print("=" * 90)

# =============================================================================
# 1. ARTIFACT PATHS
# =============================================================================

NODE_MODEL_DIR = (
    ROOT
    / "Synthetic_NAF_v2_GraphCompanion"
    / "models"
    / "NodeAware_LSTM"
    / "train_only_v1"
)

NODE_MODEL_FILE = NODE_MODEL_DIR / "NODE_LSTM_STATE.pt"
NODE_RECORD_FILE = NODE_MODEL_DIR / "NODE_LSTM_TRAIN_RECORD.json"

GRAPH_TRAIN_DIR = (
    ROOT
    / "Synthetic_NAF_v2_GraphCompanion"
    / "TRAIN_only_v1"
)

GRAPH_VALIDATION_DIR = (
    ROOT
    / "Synthetic_NAF_v2_GraphCompanion"
    / "VALIDATION_only_v1"
)

GRAPH_TRAIN_MANIFEST_FILE = (
    GRAPH_TRAIN_DIR / "GRAPH_COMPANION_MANIFEST.json"
)

GRAPH_VALIDATION_MANIFEST_FILE = (
    GRAPH_VALIDATION_DIR
    / "GRAPH_COMPANION_VALIDATION_MANIFEST.json"
)

GRAPH_VALIDATION_FILE = (
    GRAPH_VALIDATION_DIR
    / "VALIDATION_temporal_graphs.npz"
)

for path in [
    NODE_MODEL_FILE,
    NODE_RECORD_FILE,
    GRAPH_TRAIN_MANIFEST_FILE,
    GRAPH_VALIDATION_MANIFEST_FILE,
    GRAPH_VALIDATION_FILE
]:
    if not path.is_file():
        raise FileNotFoundError(path)

NODE_RECORD = json.loads(
    NODE_RECORD_FILE.read_text(encoding="utf-8")
)

GRAPH_TRAIN_MANIFEST = json.loads(
    GRAPH_TRAIN_MANIFEST_FILE.read_text(encoding="utf-8")
)

GRAPH_VAL_MANIFEST = json.loads(
    GRAPH_VALIDATION_MANIFEST_FILE.read_text(encoding="utf-8")
)

assert NODE_RECORD["model"] == "NodeAwareLSTM"
assert NODE_RECORD["input_shape"] == [10, 8, 8]
assert NODE_RECORD["hidden_dim"] == 32
assert NODE_RECORD["data_split"] == "TRAIN_ONLY"
assert NODE_RECORD["test_accessed"] is False

assert GRAPH_TRAIN_MANIFEST["nodes_per_graph"] == 8
assert GRAPH_TRAIN_MANIFEST["history_minutes"] == 10

assert GRAPH_VAL_MANIFEST["nodes_per_graph"] == 8
assert GRAPH_VAL_MANIFEST["history_minutes"] == 10
assert GRAPH_VAL_MANIFEST["forecast_horizons"] == [5, 10, 15]

# Verify generator and source consistency.
assert (
    GRAPH_VAL_MANIFEST["source_train_graph_sha256"]
    == GRAPH_TRAIN_MANIFEST["artifact_sha256"]
)

assert (
    GRAPH_VAL_MANIFEST["generator_seed"]
    == GRAPH_TRAIN_MANIFEST["seed"]
)

assert (
    NODE_RECORD["graph_sha256"]
    == GRAPH_TRAIN_MANIFEST["artifact_sha256"]
)

assert (
    sha256(GRAPH_VALIDATION_FILE)
    == GRAPH_VAL_MANIFEST["artifact_sha256"]
)

assert (
    sha256(DATA_DIR / "windows_W10_VALIDATION.npz")
    == GRAPH_VAL_MANIFEST["source_validation_sha256"]
)

print("\nGraph companion manifests: VERIFIED")
print("Graph validation file SHA256: PASS")
print("Frozen validation input SHA256: PASS")
print("Synthetic graph input:", NODE_RECORD["input_shape"])


# =============================================================================
# 2. RESTORE ORIGINAL NODE-AWARE LSTM ARCHITECTURE
# =============================================================================

class NodeAwareLSTM(nn.Module):

    def __init__(self, input_dim=8, hidden_dim=32):
        super().__init__()

        self.node_lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.attention = nn.Linear(
            hidden_dim, 1
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 3)
        )

    def forward(self, x):

        if x.ndim != 4 or x.shape[1:] != (10, 8, 8):
            raise ValueError(
                f"Expected (batch,10,8,8), got {tuple(x.shape)}"
            )

        batch, time_steps, nodes, features = x.shape

        x = x.permute(0, 2, 1, 3)
        x = x.reshape(
            batch * nodes,
            time_steps,
            features
        )

        _, (hidden, _) = self.node_lstm(x)

        node_embeddings = hidden[-1].reshape(
            batch,
            nodes,
            -1
        )

        weights = torch.softmax(
            self.attention(node_embeddings),
            dim=1
        )

        network_embedding = (
            node_embeddings * weights
        ).sum(dim=1)

        logits = self.head(network_embedding)

        hazards = torch.sigmoid(logits)

        probabilities = 1.0 - torch.cumprod(
            1.0 - hazards,
            dim=1
        )

        return probabilities.clamp(
            min=1e-6,
            max=1.0 - 1e-6
        )


# =============================================================================
# 3. LOAD ORIGINAL CHECKPOINT
# =============================================================================

# This trusted checkpoint contains NumPy arrays alongside model weights.
# It therefore needs weights_only=False. Never use this setting
# with untrusted checkpoint files.

NODE_CHECKPOINT = torch.load(
    NODE_MODEL_FILE,
    map_location="cpu",
    weights_only=False
)

required_keys = {
    "model_state",
    "median",
    "scale",
    "record",
    "graph_sha256"
}

assert required_keys.issubset(NODE_CHECKPOINT)

assert (
    NODE_CHECKPOINT["graph_sha256"]
    == GRAPH_TRAIN_MANIFEST["artifact_sha256"]
)

assert (
    NODE_CHECKPOINT["record"]["input_shape"]
    == [10, 8, 8]
)

NODE_MODEL = NodeAwareLSTM(
    input_dim=8,
    hidden_dim=NODE_RECORD["hidden_dim"]
).to(DEVICE)

NODE_MODEL.load_state_dict(
    NODE_CHECKPOINT["model_state"],
    strict=True
)

NODE_MODEL.eval()
NODE_MODEL.requires_grad_(False)

NODE_MEDIAN = np.asarray(
    NODE_CHECKPOINT["median"],
    dtype=np.float32
)

NODE_SCALE = np.asarray(
    NODE_CHECKPOINT["scale"],
    dtype=np.float32
)

assert NODE_MEDIAN.shape == (8,)
assert NODE_SCALE.shape == (8,)
assert np.isfinite(NODE_MEDIAN).all()
assert np.isfinite(NODE_SCALE).all()
assert (NODE_SCALE > 0).all()

NODE_CHECKPOINT_SHA256 = sha256(
    NODE_MODEL_FILE
)

# Compare against Cell 1's registry if available.
prior_node_entry = MODEL_REGISTRY.get(
    "node_aware_lstm", {}
)

if prior_node_entry.get("sha256"):
    assert (
        NODE_CHECKPOINT_SHA256
        == prior_node_entry["sha256"]
    )

print("\nOriginal Node-aware LSTM: RESTORED")
print(
    "Parameters:",
    sum(p.numel() for p in NODE_MODEL.parameters())
)
print("Checkpoint SHA256:", NODE_CHECKPOINT_SHA256[:24] + "...")
print("TRAIN-fitted node scaler: RESTORED")


# =============================================================================
# 4. LOAD SYNTHETIC VALIDATION GRAPH INPUTS
#
# Important:
# Read node frames, window indices, episode IDs, and minutes.
# Do NOT read future onset labels.
# =============================================================================

with np.load(
    GRAPH_VALIDATION_FILE,
    allow_pickle=False
) as graph_data:

    NODE_VAL_FRAMES = graph_data[
        "node_frames"
    ].astype(np.float32)

    NODE_VAL_END_FRAMES = graph_data[
        "window_end_frame"
    ].astype(np.int64)

    NODE_VAL_EPISODES = graph_data[
        "episode_id"
    ].astype(str)

    NODE_VAL_MINUTES = graph_data[
        "minute"
    ].astype(np.int64)

assert NODE_VAL_FRAMES.ndim == 3
assert NODE_VAL_FRAMES.shape[1:] == (8, 8)

assert len(NODE_VAL_END_FRAMES) == len(
    NODE_VAL_EPISODES
)

assert len(NODE_VAL_END_FRAMES) == len(
    NODE_VAL_MINUTES
)

assert NODE_VAL_END_FRAMES.min() >= 9
assert NODE_VAL_END_FRAMES.max() < len(NODE_VAL_FRAMES)

assert np.isfinite(NODE_VAL_FRAMES).all()
assert (NODE_VAL_FRAMES >= 0).all()

# Verify graph window rows match the frozen aggregate validation windows.
# These identifiers establish the alignment without using target labels.

assert np.array_equal(
    NODE_VAL_EPISODES,
    val_episodes
)

assert np.array_equal(
    NODE_VAL_MINUTES,
    val_minutes
)

assert len(NODE_VAL_END_FRAMES) == 18839

NODE_GRAPH_LOOKUP = {}

for i, (episode, minute) in enumerate(
    zip(NODE_VAL_EPISODES, NODE_VAL_MINUTES)
):

    key = (str(episode), int(minute))

    if key in NODE_GRAPH_LOOKUP:
        raise RuntimeError(
            f"Duplicate graph window: {key}"
        )

    NODE_GRAPH_LOOKUP[key] = i

print("\nSynthetic validation graph: LOADED")
print("Graph snapshots:", len(NODE_VAL_FRAMES))
print("Aligned windows:", len(NODE_GRAPH_LOOKUP))
print("Frozen validation row alignment: PASS")
print("Future labels read: NO")


# =============================================================================
# 5. FETCH CHRONOLOGICAL NODE INPUT FOR ANY VALIDATION WINDOW
# =============================================================================

def get_node_history(episode_id, minute):

    key = (str(episode_id), int(minute))

    if key not in NODE_GRAPH_LOOKUP:
        raise KeyError(
            f"Graph window not found: {key}"
        )

    i = NODE_GRAPH_LOOKUP[key]

    end = int(NODE_VAL_END_FRAMES[i])

    frames = NODE_VAL_FRAMES[
        end - 9 : end + 1
    ]

    if frames.shape != (10, 8, 8):
        raise RuntimeError(
            "Invalid graph history shape."
        )

    return frames.copy()


# =============================================================================
# 6. MODEL PREPROCESSING AND INFERENCE
# =============================================================================

def predict_node_aware(node_history):

    X = np.asarray(
        node_history,
        dtype=np.float32
    )

    if X.shape == (10, 8, 8):
        X = X[None, ...]

    if X.ndim != 4 or X.shape[1:] != (10, 8, 8):
        raise ValueError(
            f"Expected (batch,10,8,8), got {X.shape}"
        )

    if not np.isfinite(X).all() or (X < 0).any():
        raise ValueError(
            "Node inputs contain invalid values."
        )

    # Exact original TRAIN-only transformation:
    # log1p -> median/IQR -> clip to [-8, 8].

    X_scaled = np.clip(
        (np.log1p(X) - NODE_MEDIAN) / NODE_SCALE,
        -8.0,
        8.0
    ).astype(np.float32)

    tensor = torch.from_numpy(
        np.ascontiguousarray(X_scaled)
    ).to(DEVICE)

    with torch.inference_mode():
        probabilities = NODE_MODEL(
            tensor
        ).cpu().numpy().astype(np.float64)

    if probabilities.shape != (len(X), 3):
        raise RuntimeError(
            "Unexpected node-model output shape."
        )

    if not np.isfinite(probabilities).all():
        raise RuntimeError(
            "Non-finite node-model probabilities."
        )

    if (
        (probabilities < 0).any()
        or (probabilities > 1).any()
    ):
        raise RuntimeError(
            "Invalid node-model probabilities."
        )

    if not np.all(
        probabilities[:, 0]
        <= probabilities[:, 1] + 1e-7
    ):
        raise RuntimeError(
            "H5 probability exceeds H10."
        )

    if not np.all(
        probabilities[:, 1]
        <= probabilities[:, 2] + 1e-7
    ):
        raise RuntimeError(
            "H10 probability exceeds H15."
        )

    return probabilities


# =============================================================================
# 7. VERIFY REPEATABILITY AND INPUT VALIDATION
#
# A matching archived per-window prediction file was not saved
# for this model, so do not claim numerical reproduction
# against an independent saved reference.
# =============================================================================

test_episode = str(
    NODE_VAL_EPISODES[0]
)

test_minute = int(
    NODE_VAL_MINUTES[0]
)

test_history = get_node_history(
    test_episode,
    test_minute
)

first_pass = predict_node_aware(
    test_history
)

second_pass = predict_node_aware(
    test_history
)

max_repeat_difference = float(
    np.max(
        np.abs(first_pass - second_pass)
    )
)

if max_repeat_difference > 1e-7:
    raise RuntimeError(
        "Node-aware model is not repeatable in eval mode."
    )

# Reject wrong input dimensions.
try:
    predict_node_aware(
        np.zeros((10, 14), dtype=np.float32)
    )
except ValueError:
    pass
else:
    raise RuntimeError(
        "Invalid input shape was incorrectly accepted."
    )

print("\nModel repeatability: PASS")
print(
    "Maximum repeat difference:",
    f"{max_repeat_difference:.10f}"
)
print("Invalid input shape rejection: PASS")
print("Independent saved-prediction replay: NOT AVAILABLE")


# =============================================================================
# 8. ADD NODE-AWARE LSTM TO UNIFIED FORECAST FUNCTION
#
# Graph input is explicit and optional.
# It is NOT automatically synthesized from ordinary traffic.
# =============================================================================

if "_CELL11_BASE_FORECAST" not in globals():
    _CELL11_BASE_FORECAST = forecast_network_window


def forecast_network_window(
    traffic_df,
    *,
    graph_node_history=None
):

    result = _CELL11_BASE_FORECAST(
        traffic_df
    )

    if graph_node_history is None:

        result["auxiliary_models"][
            "node_aware_lstm"
        ] = {
            "status": "GRAPH_INPUT_REQUIRED",
            "H5": None,
            "H10": None,
            "H15": None
        }

        return result

    node_probabilities = predict_node_aware(
        graph_node_history
    )

    if len(node_probabilities) != 1:
        raise ValueError(
            "Unified forecast expects one graph window."
        )

    probabilities = node_probabilities[0]

    result["auxiliary_models"][
        "node_aware_lstm"
    ] = {
        "status": "AVAILABLE",
        "H5": float(probabilities[0]),
        "H10": float(probabilities[1]),
        "H15": float(probabilities[2]),
        "input_representation": "SYNTHETIC_NODES_10x8x8",
        "graph_origin": "SYNTHETIC_FROM_AGGREGATE_FEATURES",
        "host_attribution_available": False
    }

    return result


# =============================================================================
# 9. CONVENIENT SYNTHETIC GRAPH FORECAST FUNCTION
#
# Given one validation episode/minute:
#   - Retrieve aligned 10-minute aggregate history
#   - Retrieve aligned 10-minute synthetic node history
#   - Run existing models plus node-aware LSTM
#
# No target labels are used.
# =============================================================================

def forecast_synthetic_graph_window(
    episode_id,
    minute
):

    episode_id = str(episode_id)
    minute = int(minute)

    key = (episode_id, minute)

    if key not in NODE_GRAPH_LOOKUP:
        raise KeyError(
            f"Unknown synthetic validation window: {key}"
        )

    index = NODE_GRAPH_LOOKUP[key]

    # These arrays were loaded from frozen validation
    # by the interactive explorer in Cell 9.

    if (
        str(explorer_episode[index]) != episode_id
        or int(explorer_minute[index]) != minute
    ):
        raise RuntimeError(
            "Graph/aggregate row alignment mismatch."
        )

    traffic_df = get_window_dataframe(
        index
    )

    node_history = get_node_history(
        episode_id,
        minute
    )

    result = forecast_network_window(
        traffic_df,
        graph_node_history=node_history
    )

    result["episode_id"] = episode_id

    result["graph_input"] = {
        "source": "FROZEN_SYNTHETIC_VALIDATION_COMPANION",
        "shape": [10, 8, 8],
        "synthetic_nodes": True,
        "real_ip_topology": False,
        "graph_file_sha256": GRAPH_VAL_MANIFEST[
            "artifact_sha256"
        ]
    }

    return result


# =============================================================================
# 10. REGISTER NODE-AWARE INFERENCE
# =============================================================================

MODEL_REGISTRY["node_aware_lstm"] = {
    "name": "node_aware_lstm",
    "path": str(NODE_MODEL_FILE),
    "sha256": NODE_CHECKPOINT_SHA256,
    "checkpoint_verified": True,
    "inference_connected": True,
    "input_shape": [10, 8, 8],
    "required_input": "synthetic_node_history",
    "horizons": [5, 10, 15],
    "role": "auxiliary_graph_companion_model",
    "graph_origin": "synthetic_nodes_and_edges"
}


# =============================================================================
# 11. VERIFY EXISTING MODELS ARE UNCHANGED
# =============================================================================

old_forecast = _CELL11_BASE_FORECAST(
    sample_df
)

new_forecast = forecast_network_window(
    sample_df
)

assert (
    old_forecast["forecast_probabilities"]
    == new_forecast["forecast_probabilities"]
)

for name in old_forecast["auxiliary_models"]:

    assert (
        old_forecast["auxiliary_models"][name]
        == new_forecast["auxiliary_models"][name]
    )

assert new_forecast["auxiliary_models"][
    "node_aware_lstm"
]["status"] == "GRAPH_INPUT_REQUIRED"

print("\nExisting Stacked LSTM forecast: UNCHANGED")
print("Existing auxiliary model outputs: UNCHANGED")
print("Node-aware graph input requirement: ENFORCED")


# =============================================================================
# 12. GENERATE NODE-AWARE PREDICTION
#
# Use the last selected interactive episode if available.
# Otherwise use the previously validated sample episode.
# =============================================================================

demo_episode = sample_episode
demo_minute = int(sample_end_minute)

if (
    "LATEST_EXPLORER_REPLAY" in globals()
    and LATEST_EXPLORER_REPLAY is not None
):

    demo_episode = str(
        LATEST_EXPLORER_REPLAY["episode"]
    )

    demo_minute = int(
        LATEST_EXPLORER_REPLAY[
            "timeline"
        ]["Minute"].max()
    )

GRAPH_INTEGRATED_FORECAST = forecast_synthetic_graph_window(
    demo_episode,
    demo_minute
)

primary = GRAPH_INTEGRATED_FORECAST[
    "forecast_probabilities"
]

node_output = GRAPH_INTEGRATED_FORECAST[
    "auxiliary_models"
]["node_aware_lstm"]

assert node_output["status"] == "AVAILABLE"

print("\n" + "=" * 90)
print("NETWORK FORECAST — NODE-AWARE LSTM CONNECTED")
print("=" * 90)

print("Episode:", demo_episode)
print("Last observed minute:", demo_minute)

print("\nPRIMARY — STACKED LSTM")

for horizon in ("H5", "H10", "H15"):
    print(
        f"  {horizon}: {primary[horizon] * 100:.4f}%"
    )

print("\nAUXILIARY — NODE-AWARE LSTM")

for horizon in ("H5", "H10", "H15"):
    print(
        f"  {horizon}: {node_output[horizon] * 100:.4f}%"
    )

print("\nNode input:", node_output["input_representation"])
print("Synthetic graph input verified: YES")
print("Host/IP attribution supported: NO")
print("Model fusion performed: NO")
print("Primary alert policy changed: NO")


# =============================================================================
# 13. FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("CELL 11 — NODE-AWARE LSTM INTEGRATION: COMPLETE")
print("=" * 90)

print("Checkpoint restoration: PASS")
print("TRAIN scaler restoration: PASS")
print("Validation graph integrity: PASS")
print("Graph/aggregate alignment: PASS")
print("Repeatability check: PASS")
print("Unified forecasting integration: PASS")

print("\nOriginal primary forecast: UNCHANGED")
print("Node-aware H5/H10/H15: CONNECTED")
print("Synthetic graph inputs: REQUIRED")
print("Real IP graph data: NOT AVAILABLE")
print("Model retraining: NO")
print("Scaler refitting: NO")
print("Official TEST accessed: NO")
print("Files written: NO")

print("\nNEXT: Custom Graph LSTM integration.")
print("=" * 90)


In [ ]:

# =============================================================================
# NETWORK ATTACK FORECASTING PIPELINE
# CELL 12 — ORIGINAL CUSTOM GRAPH LSTM INTEGRATION
#
# Architecture:
#   Weighted graph convolution (8 -> 32)
#   Weighted graph convolution (32 -> 32)
#   Mean + max node pooling
#   Temporal LSTM (64 hidden units)
#   Monotonic H5 / H10 / H15 forecasting head
#
# Required graph input:
#   Nodes: (batch, 10, 8, 8)
#   Edges: (batch, 10, 8, 8)
#
# Verifies inference against archived original model predictions.
#
# NO RETRAINING / NO REFITTING / NO TEST ACCESS / NO FILE WRITES
# =============================================================================

from pathlib import Path
import json
import numpy as np
import torch
import torch.nn as nn

print("=" * 90)
print("CELL 12 — CUSTOM GRAPH LSTM INTEGRATION")
print("=" * 90)

# =============================================================================
# 1. LOCATE ORIGINAL GRAPH MODEL ARTIFACTS
# =============================================================================

GRAPH_MODEL_DIR = (
    ROOT
    / "Synthetic_NAF_v2_GraphCompanion"
    / "models"
    / "Graph_LSTM"
)

ORIGINAL_GRAPH_DIR = (
    GRAPH_MODEL_DIR / "train_only_v1"
)

ORIGINAL_GRAPH_FILE = (
    ORIGINAL_GRAPH_DIR / "GRAPH_LSTM_STATE.pt"
)

ORIGINAL_GRAPH_RECORD_FILE = (
    ORIGINAL_GRAPH_DIR / "GRAPH_LSTM_TRAIN_RECORD.json"
)

GRAPH_COMPARISON_DIR = (
    GRAPH_MODEL_DIR / "validation_comparison_v1"
)

ORIGINAL_GRAPH_REFERENCE_FILE = (
    GRAPH_COMPARISON_DIR
    / "ORIGINAL_VS_C02_VAL_PREDICTIONS.npz"
)

ORIGINAL_GRAPH_REPORT_FILE = (
    GRAPH_COMPARISON_DIR
    / "ORIGINAL_VS_C02_REPORT.json"
)

for path in [
    ORIGINAL_GRAPH_FILE,
    ORIGINAL_GRAPH_RECORD_FILE,
    ORIGINAL_GRAPH_REFERENCE_FILE,
    ORIGINAL_GRAPH_REPORT_FILE
]:
    if not path.is_file():
        raise FileNotFoundError(path)

ORIGINAL_GRAPH_RECORD = json.loads(
    ORIGINAL_GRAPH_RECORD_FILE.read_text(encoding="utf-8")
)

ORIGINAL_GRAPH_REPORT = json.loads(
    ORIGINAL_GRAPH_REPORT_FILE.read_text(encoding="utf-8")
)

assert ORIGINAL_GRAPH_RECORD["model"] == (
    "GraphLSTMForecaster"
)

assert ORIGINAL_GRAPH_RECORD["architecture"] == (
    "WeightedGraphConv_GraphConv_LSTM"
)

assert ORIGINAL_GRAPH_RECORD["validation_accessed"] is False
assert ORIGINAL_GRAPH_RECORD["test_accessed"] is False

assert ORIGINAL_GRAPH_RECORD["graph_sha256"] == (
    GRAPH_TRAIN_MANIFEST["artifact_sha256"]
)

assert ORIGINAL_GRAPH_RECORD["checkpoint_sha256"] == (
    sha256(ORIGINAL_GRAPH_FILE)
)

assert ORIGINAL_GRAPH_REPORT[
    "original_checkpoint_sha256"
] == sha256(ORIGINAL_GRAPH_FILE)

assert ORIGINAL_GRAPH_REPORT[
    "validation_graph_sha256"
] == sha256(GRAPH_VALIDATION_FILE)

GRAPH_CONFIG = ORIGINAL_GRAPH_RECORD["config"]

assert GRAPH_CONFIG["input_features"] == 8
assert GRAPH_CONFIG["nodes"] == 8
assert GRAPH_CONFIG["history"] == 10
assert GRAPH_CONFIG["graph_hidden"] == 32
assert GRAPH_CONFIG["lstm_hidden"] == 64

print("\nOriginal graph model checkpoint: VERIFIED")
print("Original training record: VERIFIED")
print("Frozen graph companion: VERIFIED")
print("Archived validation report: VERIFIED")

# =============================================================================
# 2. RECONSTRUCT WEIGHTED GRAPH CONVOLUTION
# =============================================================================

class OriginalWeightedGraphConv(nn.Module):

    def __init__(self, in_dim, out_dim):
        super().__init__()

        self.self_linear = nn.Linear(
            in_dim, out_dim
        )

        self.neighbor_linear = nn.Linear(
            in_dim, out_dim
        )

    def forward(self, x, adjacency):

        incoming = torch.bmm(
            adjacency, x
        )

        result = (
            self.self_linear(x)
            + self.neighbor_linear(incoming)
        )

        return torch.relu(result)


# =============================================================================
# 3. RECONSTRUCT THE EXACT ORIGINAL GRAPH LSTM
# =============================================================================

class OriginalGraphLSTMForecaster(nn.Module):

    def __init__(
        self,
        input_dim=8,
        graph_hidden=32,
        lstm_hidden=64,
        dropout=0.1
    ):
        super().__init__()

        self.graph_conv1 = OriginalWeightedGraphConv(
            input_dim,
            graph_hidden
        )

        self.graph_conv2 = OriginalWeightedGraphConv(
            graph_hidden,
            graph_hidden
        )

        self.graph_dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=graph_hidden * 2,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True
        )

        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 3)
        )

    def forward(self, nodes, edges):

        if nodes.ndim != 4:
            raise ValueError("Nodes must be 4-dimensional")

        if edges.ndim != 4:
            raise ValueError("Edges must be 4-dimensional")

        b, t, n, f = nodes.shape

        if (t, n, f) != (10, 8, 8):
            raise ValueError(
                "Expected nodes (batch,10,8,8)"
            )

        if edges.shape != (b, 10, 8, 8):
            raise ValueError(
                "Expected edges (batch,10,8,8)"
            )

        x = nodes.reshape(
            b * t, n, f
        )

        adjacency = edges.reshape(
            b * t, n, n
        )

        # Saved training edges use [source,target].
        # Incoming message aggregation uses [target,source].

        adjacency = adjacency.transpose(1, 2)

        degree = adjacency.sum(
            dim=-1,
            keepdim=True
        )

        adjacency = adjacency / degree.clamp(
            min=1e-8
        )

        h = self.graph_conv1(
            x, adjacency
        )

        h = self.graph_dropout(h)

        h = self.graph_conv2(
            h, adjacency
        )

        mean_pool = h.mean(dim=1)

        max_pool = h.max(dim=1).values

        graph_embedding = torch.cat(
            [mean_pool, max_pool],
            dim=-1
        )

        graph_sequence = graph_embedding.reshape(
            b, t, -1
        )

        temporal_output, _ = self.lstm(
            graph_sequence
        )

        last_state = temporal_output[:, -1, :]

        logits = self.head(last_state)

        hazards = torch.sigmoid(logits)

        probabilities = 1.0 - torch.cumprod(
            1.0 - hazards,
            dim=1
        )

        return probabilities.clamp(
            min=1e-6,
            max=1.0 - 1e-6
        )


# =============================================================================
# 4. RESTORE ORIGINAL SAVED CHECKPOINT AND SCALER
# =============================================================================

# Trusted checkpoint from our own training workflow.
# It contains NumPy arrays as well as PyTorch tensors.

ORIGINAL_GRAPH_CHECKPOINT = torch.load(
    ORIGINAL_GRAPH_FILE,
    map_location="cpu",
    weights_only=False
)

required_checkpoint_keys = {
    "model_state",
    "median",
    "scale",
    "graph_sha256",
    "config"
}

assert required_checkpoint_keys.issubset(
    ORIGINAL_GRAPH_CHECKPOINT
)

assert ORIGINAL_GRAPH_CHECKPOINT["graph_sha256"] == (
    GRAPH_TRAIN_MANIFEST["artifact_sha256"]
)

assert ORIGINAL_GRAPH_CHECKPOINT["config"][
    "graph_hidden"
] == 32

ORIGINAL_GRAPH_MODEL = OriginalGraphLSTMForecaster(
    input_dim=GRAPH_CONFIG["input_features"],
    graph_hidden=GRAPH_CONFIG["graph_hidden"],
    lstm_hidden=GRAPH_CONFIG["lstm_hidden"],
    dropout=GRAPH_CONFIG["dropout"]
).to(DEVICE)

ORIGINAL_GRAPH_MODEL.load_state_dict(
    ORIGINAL_GRAPH_CHECKPOINT["model_state"],
    strict=True
)

ORIGINAL_GRAPH_MODEL.eval()
ORIGINAL_GRAPH_MODEL.requires_grad_(False)

ORIGINAL_GRAPH_MEDIAN = np.asarray(
    ORIGINAL_GRAPH_CHECKPOINT["median"],
    dtype=np.float32
)

ORIGINAL_GRAPH_SCALE = np.asarray(
    ORIGINAL_GRAPH_CHECKPOINT["scale"],
    dtype=np.float32
)

assert ORIGINAL_GRAPH_MEDIAN.shape == (8,)
assert ORIGINAL_GRAPH_SCALE.shape == (8,)

assert np.isfinite(ORIGINAL_GRAPH_MEDIAN).all()
assert np.isfinite(ORIGINAL_GRAPH_SCALE).all()
assert (ORIGINAL_GRAPH_SCALE > 0).all()

original_parameters = sum(
    p.numel()
    for p in ORIGINAL_GRAPH_MODEL.parameters()
)

assert original_parameters == 38147

print("\nOriginal Custom Graph LSTM: RESTORED")
print("Parameters:", original_parameters)
print("Graph hidden size:", GRAPH_CONFIG["graph_hidden"])
print("LSTM hidden size:", GRAPH_CONFIG["lstm_hidden"])
print("Checkpoint state_dict: PASS")
print("TRAIN-fitted graph scaler: RESTORED")

# =============================================================================
# 5. LOAD SYNTHETIC GRAPH EDGE FRAMES
#
# Cell 11 already loaded node frames and aligned window
# identifiers from this same frozen graph companion.
#
# Do not read target labels.
# =============================================================================

with np.load(
    GRAPH_VALIDATION_FILE,
    allow_pickle=False
) as data:

    GRAPH_EDGE_FRAMES = data[
        "edge_flow_frames"
    ].astype(np.float32)

    graph_end_frames = data[
        "window_end_frame"
    ].astype(np.int64)

    graph_episodes = data[
        "episode_id"
    ].astype(str)

    graph_minutes = data[
        "minute"
    ].astype(np.int64)

assert GRAPH_EDGE_FRAMES.shape == (
    len(NODE_VAL_FRAMES), 8, 8
)

assert np.array_equal(
    graph_end_frames,
    NODE_VAL_END_FRAMES
)

assert np.array_equal(
    graph_episodes,
    NODE_VAL_EPISODES
)

assert np.array_equal(
    graph_minutes,
    NODE_VAL_MINUTES
)

assert np.isfinite(GRAPH_EDGE_FRAMES).all()
assert (GRAPH_EDGE_FRAMES >= 0).all()

print("\nSynthetic graph edges: LOADED")
print("Edge snapshots:", len(GRAPH_EDGE_FRAMES))
print("Node/edge window alignment: PASS")
print("Future targets loaded: NO")

# =============================================================================
# 6. FETCH ONE 10-MINUTE EDGE HISTORY
# =============================================================================

def get_original_edge_history(episode_id, minute):

    key = (
        str(episode_id),
        int(minute)
    )

    if key not in NODE_GRAPH_LOOKUP:
        raise KeyError(
            f"Graph window not found: {key}"
        )

    index = NODE_GRAPH_LOOKUP[key]

    end_frame = int(
        NODE_VAL_END_FRAMES[index]
    )

    history = GRAPH_EDGE_FRAMES[
        end_frame - 9 : end_frame + 1
    ]

    if history.shape != (10, 8, 8):
        raise RuntimeError(
            "Invalid graph edge history shape"
        )

    return history.copy()


# =============================================================================
# 7. ORIGINAL TRAINING PREPROCESSING
# =============================================================================

def predict_original_graph_lstm(
    node_history,
    edge_history
):

    nodes = np.asarray(
        node_history,
        dtype=np.float32
    )

    edges = np.asarray(
        edge_history,
        dtype=np.float32
    )

    if nodes.shape == (10, 8, 8):
        nodes = nodes[None, ...]

    if edges.shape == (10, 8, 8):
        edges = edges[None, ...]

    if nodes.ndim != 4 or nodes.shape[1:] != (10, 8, 8):
        raise ValueError(
            "Nodes must have shape (batch,10,8,8)"
        )

    if edges.shape != nodes.shape:
        raise ValueError(
            "Node and edge histories must have matching shapes"
        )

    if not np.isfinite(nodes).all() or (nodes < 0).any():
        raise ValueError("Invalid node features")

    if not np.isfinite(edges).all() or (edges < 0).any():
        raise ValueError("Invalid edge features")

    # TRAIN-only preprocessing for nodes.
    nodes_scaled = np.clip(
        (
            np.log1p(nodes)
            - ORIGINAL_GRAPH_MEDIAN
        ) / ORIGINAL_GRAPH_SCALE,
        -8.0,
        8.0
    ).astype(np.float32)

    # First normalize directed edge flow by the
    # TOTAL flow within each minute's graph.
    #
    # The model then performs incoming-row
    # normalization in its forward pass.
    #
    # Both stages reproduce the saved implementation.

    total_flow = edges.sum(
        axis=(-2, -1),
        keepdims=True
    )

    edges_scaled = (
        edges / np.maximum(total_flow, 1e-8)
    ).astype(np.float32)

    node_tensor = torch.from_numpy(
        np.ascontiguousarray(nodes_scaled)
    ).to(DEVICE)

    edge_tensor = torch.from_numpy(
        np.ascontiguousarray(edges_scaled)
    ).to(DEVICE)

    with torch.inference_mode():

        probabilities = ORIGINAL_GRAPH_MODEL(
            node_tensor,
            edge_tensor
        ).cpu().numpy().astype(np.float64)

    if probabilities.shape != (len(nodes), 3):
        raise RuntimeError(
            "Unexpected graph LSTM output shape"
        )

    if not np.isfinite(probabilities).all():
        raise RuntimeError(
            "Non-finite graph LSTM probabilities"
        )

    if (
        (probabilities < 0).any()
        or (probabilities > 1).any()
    ):
        raise RuntimeError(
            "Graph LSTM probabilities outside [0,1]"
        )

    if not (
        np.all(
            probabilities[:, 0]
            <= probabilities[:, 1] + 1e-7
        )
        and np.all(
            probabilities[:, 1]
            <= probabilities[:, 2] + 1e-7
        )
    ):
        raise RuntimeError(
            "Graph LSTM horizon monotonicity violated"
        )

    return probabilities


# =============================================================================
# 8. VERIFY AGAINST SAVED ORIGINAL MODEL PREDICTIONS
#
# Read prediction arrays and row identifiers ONLY.
# Ground-truth labels are never read.
# =============================================================================

print("\n" + "-" * 90)
print("CUSTOM GRAPH LSTM SAVED-INFERENCE REPRODUCTION")
print("-" * 90)

with np.load(
    ORIGINAL_GRAPH_REFERENCE_FILE,
    allow_pickle=False
) as data:

    saved_episodes = data[
        "episode_id"
    ].astype(str)

    saved_minutes = data[
        "minute"
    ].astype(np.int64)

    saved_original_predictions = data[
        "original_probabilities"
    ][sample_indices]

assert np.array_equal(
    saved_episodes,
    NODE_VAL_EPISODES
)

assert np.array_equal(
    saved_minutes,
    NODE_VAL_MINUTES
)

node_samples = np.stack([
    get_node_history(
        NODE_VAL_EPISODES[i],
        NODE_VAL_MINUTES[i]
    )
    for i in sample_indices
])

edge_samples = np.stack([
    get_original_edge_history(
        NODE_VAL_EPISODES[i],
        NODE_VAL_MINUTES[i]
    )
    for i in sample_indices
])

replayed_graph_predictions = (
    predict_original_graph_lstm(
        node_samples,
        edge_samples
    )
)

max_graph_difference = float(
    np.max(
        np.abs(
            replayed_graph_predictions
            - saved_original_predictions
        )
    )
)

print(
    "Maximum saved-prediction difference:",
    f"{max_graph_difference:.10f}"
)

if max_graph_difference > 2e-6:

    print(
        "Expected first row:",
        saved_original_predictions[0].tolist()
    )

    print(
        "Observed first row:",
        replayed_graph_predictions[0].tolist()
    )

    raise RuntimeError(
        "Original Custom Graph LSTM inference "
        "does not reproduce saved predictions. "
        "Integration stopped."
    )

print("Saved Custom Graph LSTM reproduction: PASS")

# =============================================================================
# 9. WRAP EXISTING UNIFIED FORECASTER
#
# Preserve:
#   Stacked LSTM
#   Single LSTM
#   Competing-risk LSTM
#   XGBoost
#   Competing-risk TCN
#   Node-aware LSTM
#
# Add:
#   Original Custom Graph LSTM
#
# No fusion, no changes to primary alerts.
# =============================================================================

if "_CELL12_BASE_FORECAST" not in globals():
    _CELL12_BASE_FORECAST = forecast_network_window


def forecast_network_window(
    traffic_df,
    *,
    graph_node_history=None,
    graph_edge_history=None
):

    if (
        graph_edge_history is not None
        and graph_node_history is None
    ):
        raise ValueError(
            "Graph edges require aligned node history."
        )

    result = _CELL12_BASE_FORECAST(
        traffic_df,
        graph_node_history=graph_node_history
    )

    if (
        graph_node_history is None
        or graph_edge_history is None
    ):

        result["auxiliary_models"][
            "custom_graph_lstm"
        ] = {
            "status": "NODE_AND_EDGE_INPUT_REQUIRED",
            "H5": None,
            "H10": None,
            "H15": None
        }

        return result

    probabilities = predict_original_graph_lstm(
        graph_node_history,
        graph_edge_history
    )

    if len(probabilities) != 1:
        raise ValueError(
            "Expected exactly one graph history"
        )

    p = probabilities[0]

    result["auxiliary_models"][
        "custom_graph_lstm"
    ] = {
        "status": "AVAILABLE",
        "H5": float(p[0]),
        "H10": float(p[1]),
        "H15": float(p[2]),
        "input_representation": "SYNTHETIC_GRAPH_10x8x8",
        "uses_directed_edges": True,
        "real_ip_topology": False
    }

    return result


# =============================================================================
# 10. UPDATE SYNTHETIC GRAPH FORECASTING HELPER
#
# This feeds BOTH node and edge histories automatically.
# Existing regular-traffic forecasting stays unchanged.
# =============================================================================

def forecast_synthetic_graph_window(
    episode_id,
    minute
):

    episode_id = str(episode_id)
    minute = int(minute)

    key = (episode_id, minute)

    if key not in NODE_GRAPH_LOOKUP:
        raise KeyError(
            f"Unknown synthetic graph window: {key}"
        )

    index = NODE_GRAPH_LOOKUP[key]

    if (
        str(explorer_episode[index]) != episode_id
        or int(explorer_minute[index]) != minute
    ):
        raise RuntimeError(
            "Graph/traffic input misalignment"
        )

    traffic = get_window_dataframe(
        index
    )

    nodes = get_node_history(
        episode_id,
        minute
    )

    edges = get_original_edge_history(
        episode_id,
        minute
    )

    result = forecast_network_window(
        traffic,
        graph_node_history=nodes,
        graph_edge_history=edges
    )

    result["episode_id"] = episode_id

    result["graph_input"] = {
        "source": "SYNTHETIC_VALIDATION_GRAPH_COMPANION",
        "node_history_shape": [10, 8, 8],
        "edge_history_shape": [10, 8, 8],
        "synthetic_graph": True,
        "real_network_topology": False,
        "graph_sha256": GRAPH_VAL_MANIFEST[
            "artifact_sha256"
        ]
    }

    return result


# =============================================================================
# 11. REGISTER CUSTOM GRAPH LSTM
# =============================================================================

MODEL_REGISTRY["custom_graph_lstm"] = {
    "name": "custom_graph_lstm",
    "path": str(ORIGINAL_GRAPH_FILE),
    "sha256": sha256(ORIGINAL_GRAPH_FILE),
    "checkpoint_verified": True,
    "inference_connected": True,
    "input_shape_nodes": [10, 8, 8],
    "input_shape_edges": [10, 8, 8],
    "horizons": [5, 10, 15],
    "role": "auxiliary_graph_model",
    "graph_origin": "synthetic_node_and_edge_allocations"
}


# =============================================================================
# 12. VERIFY EXISTING FORECAST OUTPUTS ARE UNCHANGED
# =============================================================================

old_forecast = _CELL12_BASE_FORECAST(
    sample_df
)

new_forecast = forecast_network_window(
    sample_df
)

assert (
    old_forecast["forecast_probabilities"]
    == new_forecast["forecast_probabilities"]
)

for model_name in old_forecast["auxiliary_models"]:

    assert (
        old_forecast["auxiliary_models"][model_name]
        == new_forecast["auxiliary_models"][model_name]
    )

assert new_forecast["auxiliary_models"][
    "custom_graph_lstm"
]["status"] == "NODE_AND_EDGE_INPUT_REQUIRED"

print("\nExisting primary forecast: UNCHANGED")
print("Existing auxiliary predictions: UNCHANGED")
print("Synthetic edge input requirement: ENFORCED")

# =============================================================================
# 13. FIRST INTEGRATED CUSTOM GRAPH FORECAST
# =============================================================================

demo_episode = sample_episode
demo_minute = int(sample_end_minute)

if (
    "LATEST_EXPLORER_REPLAY" in globals()
    and LATEST_EXPLORER_REPLAY is not None
):

    demo_episode = str(
        LATEST_EXPLORER_REPLAY["episode"]
    )

    demo_minute = int(
        LATEST_EXPLORER_REPLAY[
            "timeline"
        ]["Minute"].max()
    )

GRAPH_FORECAST = forecast_synthetic_graph_window(
    demo_episode,
    demo_minute
)

primary = GRAPH_FORECAST[
    "forecast_probabilities"
]

node_output = GRAPH_FORECAST[
    "auxiliary_models"
]["node_aware_lstm"]

graph_output = GRAPH_FORECAST[
    "auxiliary_models"
]["custom_graph_lstm"]

assert node_output["status"] == "AVAILABLE"
assert graph_output["status"] == "AVAILABLE"

print("\n" + "=" * 90)
print("NETWORK FORECAST — CUSTOM GRAPH LSTM CONNECTED")
print("=" * 90)

print("Episode:", demo_episode)
print("Last observed minute:", demo_minute)

print("\nPRIMARY — STACKED LSTM")

for horizon in ("H5", "H10", "H15"):
    print(
        f"  {horizon}: {primary[horizon] * 100:.4f}%"
    )

print("\nAUXILIARY — NODE-AWARE LSTM")

for horizon in ("H5", "H10", "H15"):
    print(
        f"  {horizon}: {node_output[horizon] * 100:.4f}%"
    )

print("\nAUXILIARY — ORIGINAL CUSTOM GRAPH LSTM")

for horizon in ("H5", "H10", "H15"):
    print(
        f"  {horizon}: {graph_output[horizon] * 100:.4f}%"
    )

print("\nDirected synthetic edges used: YES")
print("Real host/IP topology available: NO")
print("Primary model changed: NO")
print("Model fusion performed: NO")

# =============================================================================
# 14. FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("CELL 12 — CUSTOM GRAPH LSTM INTEGRATION: COMPLETE")
print("=" * 90)

print("Original architecture: RESTORED")
print("Checkpoint SHA256: VERIFIED")
print("TRAIN-only scaler: RESTORED")
print("Graph input alignment: VERIFIED")
print("Saved inference reproduction: PASS")

print("\nOriginal Custom Graph LSTM: CONNECTED")
print("Node-aware LSTM: STILL CONNECTED")
print("Other auxiliary models: UNCHANGED")
print("Primary Stacked LSTM: UNCHANGED")
print("Alert policy: UNCHANGED")

print("\nTraining performed: NO")
print("Scaler refitting: NO")
print("Official TEST accessed: NO")
print("Files written: NO")

print("\nNEXT: Optimized Custom Graph LSTM integration.")
print("=" * 90)


In [ ]:

# =============================================================================
# NETWORK ATTACK FORECASTING PIPELINE
# CELL 13 — OPTIMIZED CUSTOM GRAPH LSTM (C02) INTEGRATION
#
# Uses the frozen, TRAIN-trained C02 graph checkpoint.
#
# Architecture:
#   Weighted GraphConv: 8 -> 64
#   Weighted GraphConv: 64 -> 64
#   Mean + Max graph pooling: 128
#   Temporal LSTM: 128 -> 64
#   Monotonic H5 / H10 / H15 output
#
# Reuses the verified architecture from Cell 12.
#
# No retraining
# No scaler refitting
# No TEST access
# No files written
# =============================================================================

from pathlib import Path
import json
import numpy as np
import torch

print("=" * 90)
print("CELL 13 — OPTIMIZED CUSTOM GRAPH LSTM INTEGRATION")
print("=" * 90)

# =============================================================================
# 1. IDENTIFY ORIGINAL OPTIMIZED ARTIFACTS
# =============================================================================

OPT_GRAPH_DIR = (
    ROOT
    / "Synthetic_NAF_v2_GraphCompanion"
    / "models"
    / "Graph_LSTM"
    / "optimized_full_train_v1"
)

OPT_GRAPH_MODEL_FILE = (
    OPT_GRAPH_DIR
    / "GRAPH_LSTM_C02_OPTIMIZED_STATE.pt"
)

OPT_GRAPH_RECORD_FILE = (
    OPT_GRAPH_DIR
    / "GRAPH_LSTM_C02_TRAIN_RECORD.json"
)

OPT_GRAPH_REFERENCE_FILE = (
    GRAPH_MODEL_DIR
    / "validation_comparison_v1"
    / "ORIGINAL_VS_C02_VAL_PREDICTIONS.npz"
)

for path in [
    OPT_GRAPH_MODEL_FILE,
    OPT_GRAPH_RECORD_FILE,
    OPT_GRAPH_REFERENCE_FILE
]:
    if not path.is_file():
        raise FileNotFoundError(path)

OPT_GRAPH_RECORD = json.loads(
    OPT_GRAPH_RECORD_FILE.read_text(encoding="utf-8")
)

# =============================================================================
# 2. VERIFY ORIGINAL CHECKPOINT AND CONFIGURATION
# =============================================================================

OPT_GRAPH_SHA256 = sha256(
    OPT_GRAPH_MODEL_FILE
)

assert OPT_GRAPH_SHA256 == (
    OPT_GRAPH_RECORD["checkpoint_sha256"]
)

assert OPT_GRAPH_SHA256 == (
    ORIGINAL_GRAPH_REPORT[
        "optimized_checkpoint_sha256"
    ]
)

assert OPT_GRAPH_RECORD["model"] == (
    "GraphLSTMForecaster"
)

assert OPT_GRAPH_RECORD["selected_config"] == (
    "C02_wider_graph"
)

assert OPT_GRAPH_RECORD["graph_sha256"] == (
    GRAPH_TRAIN_MANIFEST["artifact_sha256"]
)

assert OPT_GRAPH_RECORD["validation_accessed"] is False
assert OPT_GRAPH_RECORD["test_accessed"] is False

OPT_GRAPH_CONFIG = OPT_GRAPH_RECORD["config"]

assert OPT_GRAPH_CONFIG["graph_hidden"] == 64
assert OPT_GRAPH_CONFIG["lstm_hidden"] == 64
assert OPT_GRAPH_CONFIG["dropout"] == 0.1

assert OPT_GRAPH_RECORD["parameters"] == 61315

print("\nOptimized C02 training record: VERIFIED")
print("Optimized checkpoint SHA256: VERIFIED")
print("TRAIN graph source: VERIFIED")
print("Official TEST accessed by training: NO")

# =============================================================================
# 3. RESTORE OPTIMIZED CHECKPOINT
# =============================================================================

# Trusted checkpoint from the existing project.
# Includes NumPy scaler arrays, therefore uses
# weights_only=False. Do not use for untrusted files.

OPT_GRAPH_CHECKPOINT = torch.load(
    OPT_GRAPH_MODEL_FILE,
    map_location="cpu",
    weights_only=False
)

required_keys = {
    "model_state",
    "median",
    "scale",
    "graph_sha256",
    "config",
    "architecture"
}

if not required_keys.issubset(
    OPT_GRAPH_CHECKPOINT
):
    raise RuntimeError(
        "Unexpected optimized graph checkpoint format."
    )

assert OPT_GRAPH_CHECKPOINT["graph_sha256"] == (
    GRAPH_TRAIN_MANIFEST["artifact_sha256"]
)

assert OPT_GRAPH_CHECKPOINT["architecture"] == (
    "WeightedGraphConv_GraphConv_LSTM"
)

assert OPT_GRAPH_CHECKPOINT["config"] == (
    OPT_GRAPH_CONFIG
)

# The original and C02 models share the same
# architectural structure, but C02 uses wider
# graph convolution layers.

OPT_GRAPH_MODEL = OriginalGraphLSTMForecaster(
    input_dim=8,
    graph_hidden=64,
    lstm_hidden=64,
    dropout=0.1
).to(DEVICE)

OPT_GRAPH_MODEL.load_state_dict(
    OPT_GRAPH_CHECKPOINT["model_state"],
    strict=True
)

OPT_GRAPH_MODEL.eval()
OPT_GRAPH_MODEL.requires_grad_(False)

OPT_GRAPH_PARAMETERS = sum(
    p.numel()
    for p in OPT_GRAPH_MODEL.parameters()
)

assert OPT_GRAPH_PARAMETERS == 61315

# =============================================================================
# 4. RESTORE C02'S OWN TRAIN-FITTED SCALER
# =============================================================================

OPT_GRAPH_MEDIAN = np.asarray(
    OPT_GRAPH_CHECKPOINT["median"],
    dtype=np.float32
)

OPT_GRAPH_SCALE = np.asarray(
    OPT_GRAPH_CHECKPOINT["scale"],
    dtype=np.float32
)

assert OPT_GRAPH_MEDIAN.shape == (8,)
assert OPT_GRAPH_SCALE.shape == (8,)

assert np.isfinite(OPT_GRAPH_MEDIAN).all()
assert np.isfinite(OPT_GRAPH_SCALE).all()

assert np.all(
    OPT_GRAPH_SCALE > 0
)

print("\nOptimized Graph LSTM restored: PASS")
print("Graph hidden units: 64")
print("Temporal LSTM hidden units: 64")
print("Trainable parameters:", OPT_GRAPH_PARAMETERS)
print("TRAIN-fitted scaler: RESTORED")

# =============================================================================
# 5. OPTIMIZED GRAPH INFERENCE
# =============================================================================

def predict_optimized_graph_lstm(
    node_history,
    edge_history
):

    nodes = np.asarray(
        node_history,
        dtype=np.float32
    )

    edges = np.asarray(
        edge_history,
        dtype=np.float32
    )

    if nodes.shape == (10, 8, 8):
        nodes = nodes[None, ...]

    if edges.shape == (10, 8, 8):
        edges = edges[None, ...]

    if (
        nodes.ndim != 4
        or nodes.shape[1:] != (10, 8, 8)
    ):
        raise ValueError(
            "Expected nodes (batch,10,8,8)"
        )

    if edges.shape != nodes.shape:
        raise ValueError(
            "Expected edges (batch,10,8,8)"
        )

    if (
        not np.isfinite(nodes).all()
        or (nodes < 0).any()
    ):
        raise ValueError(
            "Invalid optimized graph node features."
        )

    if (
        not np.isfinite(edges).all()
        or (edges < 0).any()
    ):
        raise ValueError(
            "Invalid optimized graph edge features."
        )

    # Original TRAIN transformation:
    # log1p -> median/IQR -> clip [-8,8]

    nodes_scaled = np.clip(
        (
            np.log1p(nodes)
            - OPT_GRAPH_MEDIAN
        ) / OPT_GRAPH_SCALE,
        -8.0,
        8.0
    ).astype(np.float32)

    # Preserve original directed-edge preprocessing:
    # Normalize total edge flow per minute.
    # Incoming-degree normalization is then
    # performed inside the graph model.

    edge_totals = edges.sum(
        axis=(-2, -1),
        keepdims=True
    )

    edges_scaled = (
        edges / np.maximum(
            edge_totals,
            1e-8
        )
    ).astype(np.float32)

    node_tensor = torch.from_numpy(
        np.ascontiguousarray(nodes_scaled)
    ).to(DEVICE)

    edge_tensor = torch.from_numpy(
        np.ascontiguousarray(edges_scaled)
    ).to(DEVICE)

    with torch.inference_mode():

        probabilities = OPT_GRAPH_MODEL(
            node_tensor,
            edge_tensor
        ).cpu().numpy().astype(np.float64)

    expected_shape = (
        len(nodes),
        3
    )

    if probabilities.shape != expected_shape:
        raise RuntimeError(
            "Unexpected optimized graph output shape."
        )

    if (
        not np.isfinite(probabilities).all()
        or (probabilities < 0).any()
        or (probabilities > 1).any()
    ):
        raise RuntimeError(
            "Invalid optimized graph probabilities."
        )

    if not (
        np.all(
            probabilities[:, 0]
            <= probabilities[:, 1] + 1e-7
        )
        and np.all(
            probabilities[:, 1]
            <= probabilities[:, 2] + 1e-7
        )
    ):
        raise RuntimeError(
            "Optimized graph horizon monotonicity violated."
        )

    return probabilities


# =============================================================================
# 6. VERIFY AGAINST SAVED VALIDATION PREDICTIONS
#
# Check saved prediction rows only.
# No ground-truth labels used.
# =============================================================================

print("\n" + "-" * 90)
print("OPTIMIZED GRAPH LSTM SAVED-INFERENCE REPRODUCTION")
print("-" * 90)

with np.load(
    OPT_GRAPH_REFERENCE_FILE,
    allow_pickle=False
) as saved:

    saved_episode_ids = saved[
        "episode_id"
    ].astype(str)

    saved_minute_indices = saved[
        "minute"
    ].astype(np.int64)

    saved_optimized_predictions = saved[
        "optimized_probabilities"
    ][sample_indices]

assert np.array_equal(
    saved_episode_ids,
    NODE_VAL_EPISODES
)

assert np.array_equal(
    saved_minute_indices,
    NODE_VAL_MINUTES
)

assert saved_optimized_predictions.shape == (
    len(sample_indices),
    3
)

# Reuse the independently verified graph samples
# built in Cell 12.

optimized_replayed_predictions = (
    predict_optimized_graph_lstm(
        node_samples,
        edge_samples
    )
)

optimized_difference = float(
    np.max(
        np.abs(
            optimized_replayed_predictions
            - saved_optimized_predictions
        )
    )
)

print(
    "Maximum saved-prediction difference:",
    f"{optimized_difference:.10f}"
)

if optimized_difference > 2e-6:

    print(
        "Expected first row:",
        saved_optimized_predictions[0].tolist()
    )

    print(
        "Observed first row:",
        optimized_replayed_predictions[0].tolist()
    )

    raise RuntimeError(
        "Optimized graph model failed saved-inference "
        "reproduction. Integration stopped."
    )

print("Optimized Graph LSTM reproduction: PASS")


# =============================================================================
# 7. ADD OPTIMIZED GRAPH MODEL TO UNIFIED FORECASTER
#
# Existing output is preserved.
#
# Node-aware and Original Graph LSTM remain connected.
# =============================================================================

if "_CELL13_BASE_FORECAST" not in globals():

    _CELL13_BASE_FORECAST = (
        forecast_network_window
    )


def forecast_network_window(
    traffic_df,
    *,
    graph_node_history=None,
    graph_edge_history=None
):

    # First run the complete existing forecasting pipeline.

    result = _CELL13_BASE_FORECAST(
        traffic_df,
        graph_node_history=graph_node_history,
        graph_edge_history=graph_edge_history
    )

    if (
        graph_node_history is None
        or graph_edge_history is None
    ):

        result["auxiliary_models"][
            "optimized_graph_lstm"
        ] = {
            "status": "NODE_AND_EDGE_INPUT_REQUIRED",
            "H5": None,
            "H10": None,
            "H15": None
        }

        return result

    probabilities = predict_optimized_graph_lstm(
        graph_node_history,
        graph_edge_history
    )

    if probabilities.shape != (1, 3):
        raise RuntimeError(
            "Expected exactly one optimized graph forecast."
        )

    p = probabilities[0]

    result["auxiliary_models"][
        "optimized_graph_lstm"
    ] = {
        "status": "AVAILABLE",
        "H5": float(p[0]),
        "H10": float(p[1]),
        "H15": float(p[2]),
        "input_representation": "SYNTHETIC_GRAPH_10x8x8",
        "uses_directed_edges": True,
        "real_ip_topology": False,
        "checkpoint_variant": "C02_wider_graph"
    }

    return result


# =============================================================================
# 8. UPDATE MODEL REGISTRY
# =============================================================================

MODEL_REGISTRY["optimized_graph_lstm"] = {
    "name": "optimized_graph_lstm",
    "path": str(OPT_GRAPH_MODEL_FILE),
    "sha256": OPT_GRAPH_SHA256,
    "checkpoint_verified": True,
    "inference_connected": True,
    "input_shape_nodes": [10, 8, 8],
    "input_shape_edges": [10, 8, 8],
    "horizons": [5, 10, 15],
    "role": "auxiliary_graph_model",
    "variant": "C02_wider_graph",
    "graph_origin": "synthetic_node_and_edge_allocations"
}


# =============================================================================
# 9. VERIFY EXISTING PRIMARY AND AUXILIARY OUTPUTS
# =============================================================================

old_forecast = _CELL13_BASE_FORECAST(
    sample_df
)

new_forecast = forecast_network_window(
    sample_df
)

assert (
    old_forecast["forecast_probabilities"]
    == new_forecast["forecast_probabilities"]
)

assert (
    new_forecast["primary_model"]
    == "stacked_lstm"
)

for name in old_forecast["auxiliary_models"]:

    assert (
        old_forecast["auxiliary_models"][name]
        == new_forecast["auxiliary_models"][name]
    )

assert (
    new_forecast["auxiliary_models"][
        "optimized_graph_lstm"
    ]["status"]
    == "NODE_AND_EDGE_INPUT_REQUIRED"
)

print("\nPrimary Stacked LSTM forecast: UNCHANGED")
print("Existing auxiliary outputs: UNCHANGED")
print("Graph input requirement: ENFORCED")

# =============================================================================
# 10. TEST ALL THREE SYNTHETIC GRAPH ADAPTERS
#
# The Cell 12 helper automatically calls the current
# unified forecasting function and supplies the
# aligned synthetic node + edge history.
# =============================================================================

demo_episode = sample_episode
demo_minute = int(sample_end_minute)

if (
    "LATEST_EXPLORER_REPLAY" in globals()
    and LATEST_EXPLORER_REPLAY is not None
):

    demo_episode = str(
        LATEST_EXPLORER_REPLAY["episode"]
    )

    demo_minute = int(
        LATEST_EXPLORER_REPLAY[
            "timeline"
        ]["Minute"].max()
    )

OPT_GRAPH_FORECAST = forecast_synthetic_graph_window(
    demo_episode,
    demo_minute
)

primary = OPT_GRAPH_FORECAST[
    "forecast_probabilities"
]

auxiliary = OPT_GRAPH_FORECAST[
    "auxiliary_models"
]

for name in (
    "node_aware_lstm",
    "custom_graph_lstm",
    "optimized_graph_lstm"
):

    assert auxiliary[name]["status"] == "AVAILABLE"

print("\n" + "=" * 90)
print("NETWORK FORECAST — OPTIMIZED GRAPH LSTM CONNECTED")
print("=" * 90)

print("Episode:", demo_episode)
print("Last observed minute:", demo_minute)

print("\nPRIMARY — STACKED LSTM")

for horizon in ("H5", "H10", "H15"):

    print(
        f"  {horizon}: "
        f"{primary[horizon] * 100:.4f}%"
    )

for name, label in [
    ("node_aware_lstm", "NODE-AWARE LSTM"),
    ("custom_graph_lstm", "ORIGINAL CUSTOM GRAPH LSTM"),
    ("optimized_graph_lstm", "OPTIMIZED GRAPH LSTM C02")
]:

    print(f"\nAUXILIARY — {label}")

    for horizon in ("H5", "H10", "H15"):

        print(
            f"  {horizon}: "
            f"{auxiliary[name][horizon] * 100:.4f}%"
        )

print("\nSynthetic node and edge inputs: VERIFIED")
print("Model fusion performed: NO")
print("Primary forecast changed: NO")
print("Alert policy changed: NO")

# =============================================================================
# 11. CHECK GRAPH-AWARE HELPER IS REPEATABLE
# =============================================================================

second_forecast = forecast_synthetic_graph_window(
    demo_episode,
    demo_minute
)

for horizon in ("H5", "H10", "H15"):

    assert np.isclose(
        OPT_GRAPH_FORECAST["auxiliary_models"][
            "optimized_graph_lstm"
        ][horizon],

        second_forecast["auxiliary_models"][
            "optimized_graph_lstm"
        ][horizon],

        atol=1e-7,
        rtol=0
    )

print("\nOptimized graph inference repeatability: PASS")

# =============================================================================
# 12. FINAL STATUS
# =============================================================================

print("\n" + "=" * 90)
print("CELL 13 — OPTIMIZED GRAPH LSTM INTEGRATION: COMPLETE")
print("=" * 90)

print("Optimized checkpoint SHA256: VERIFIED")
print("Original C02 architecture: RESTORED")
print("TRAIN-only scaler: RESTORED")
print("Saved validation inference: REPRODUCED")
print("Graph input alignment: VERIFIED")

print("\nNode-aware LSTM: CONNECTED")
print("Original Custom Graph LSTM: CONNECTED")
print("Optimized Custom Graph LSTM: CONNECTED")

print("\nPrimary Stacked LSTM: UNCHANGED")
print("Existing auxiliary outputs: UNCHANGED")
print("Alert policy: UNCHANGED")

print("\nModel retraining: NO")
print("Scaler refitting: NO")
print("Official TEST accessed: NO")
print("Files modified: NO")

print("\nNEXT: GCN-LSTM integration.")
print("=" * 90)


In [ ]:

# =============================================================================
# NETWORK ATTACK FORECASTING PIPELINE
# CELL 14 — GCN-LSTM INFERENCE AND INTEGRATION
#
# Original architecture:
#   PyG GCNConv: 8 -> 32
#   PyG GCNConv: 32 -> 32
#   Mean + max graph pooling
#   LSTM: 64 -> 64
#   Monotonic H5 / H10 / H15 forecasting head
#
# Uses an equivalent dense GCN implementation for 8-node graphs.
# Numerical correctness is checked against the original archived outputs.
#
# NO RETRAINING
# NO SCALER REFITTING
# NO MODEL FUSION
# NO OFFICIAL TEST ACCESS
# NO FILE WRITES
# =============================================================================

from pathlib import Path
import json
import numpy as np
import torch
import torch.nn as nn

print("=" * 90)
print("CELL 14 — GCN-LSTM INTEGRATION")
print("=" * 90)

# =============================================================================
# 1. LOCATE THE ORIGINAL TRAINED ARTIFACTS
# =============================================================================

GCN_ROOT = (
    ROOT
    / "Synthetic_NAF_v2_GraphCompanion"
    / "models"
    / "GCN_LSTM"
)

GCN_TRAIN_DIR = GCN_ROOT / "train_only_v1"
GCN_VALIDATION_DIR = GCN_ROOT / "validation_v1"

GCN_MODEL_FILE = (
    GCN_TRAIN_DIR / "GCN_LSTM_STATE.pt"
)

GCN_RECORD_FILE = (
    GCN_TRAIN_DIR / "GCN_LSTM_TRAIN_RECORD.json"
)

GCN_REFERENCE_FILE = (
    GCN_VALIDATION_DIR / "GCN_VAL_PREDICTIONS.npz"
)

GCN_REPORT_FILE = (
    GCN_VALIDATION_DIR / "GCN_VAL_REPORT.json"
)

for path in [
    GCN_MODEL_FILE,
    GCN_RECORD_FILE,
    GCN_REFERENCE_FILE,
    GCN_REPORT_FILE
]:
    if not path.is_file():
        raise FileNotFoundError(path)

GCN_RECORD = json.loads(
    GCN_RECORD_FILE.read_text(encoding="utf-8")
)

GCN_REPORT = json.loads(
    GCN_REPORT_FILE.read_text(encoding="utf-8")
)

GCN_CONFIG = GCN_RECORD["config"]

# =============================================================================
# 2. VERIFY FROZEN TRAINING ARTIFACTS
# =============================================================================

GCN_CHECKPOINT_SHA256 = sha256(
    GCN_MODEL_FILE
)

assert GCN_CHECKPOINT_SHA256 == (
    GCN_RECORD["checkpoint_sha256"]
)

assert GCN_CHECKPOINT_SHA256 == (
    GCN_REPORT["gcn_checkpoint_sha256"]
)

assert GCN_RECORD["model"] == (
    "GCNLSTMForecaster"
)

assert GCN_RECORD["architecture"] == (
    "PyG_GCNConv_GCNConv_LSTM"
)

assert GCN_CONFIG["gcn_hidden"] == 32
assert GCN_CONFIG["lstm_hidden"] == 64
assert GCN_CONFIG["dropout"] == 0.1

assert GCN_CONFIG["nodes"] == 8
assert GCN_CONFIG["node_features"] == 8
assert GCN_CONFIG["history"] == 10

assert GCN_CONFIG["edge_processing"] == (
    "OUTGOING_ROW_NORMALIZED"
)

assert GCN_RECORD["parameters"] == 36803

assert GCN_RECORD["train_graph_sha256"] == (
    GRAPH_TRAIN_MANIFEST["artifact_sha256"]
)

assert GCN_RECORD["official_validation_accessed"] is False
assert GCN_RECORD["official_test_accessed"] is False

assert GCN_REPORT["validation_graph_sha256"] == (
    GRAPH_VAL_MANIFEST["artifact_sha256"]
)

assert sha256(GRAPH_VALIDATION_FILE) == (
    GRAPH_VAL_MANIFEST["artifact_sha256"]
)

print("\nOriginal GCN training record: VERIFIED")
print("GCN checkpoint SHA256: VERIFIED")
print("Frozen graph source: VERIFIED")
print("Validation graph integrity: PASS")
print("Original architecture contract: PASS")

# =============================================================================
# 3. RECONSTRUCT ORIGINAL GCN CONVOLUTION
#
# This dense layer retains the same parameters:
#   gcn1.lin.weight
#   gcn1.bias
#
# The edge normalization reproduces PyG GCNConv's
# weighted symmetric normalization, including self-loops.
#
# Graphs have exactly 8 nodes, allowing dense computation.
# =============================================================================

class DenseEquivalentGCNConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.lin = nn.Linear(
            in_channels,
            out_channels,
            bias=False
        )

        self.bias = nn.Parameter(
            torch.zeros(out_channels)
        )

    def forward(self, x, normalized_adjacency):

        transformed = self.lin(x)

        aggregated = torch.bmm(
            normalized_adjacency,
            transformed
        )

        return aggregated + self.bias


# =============================================================================
# 4. RECONSTRUCT ORIGINAL GCN-LSTM
# =============================================================================

class GCNLSTMForecaster(nn.Module):

    def __init__(
        self,
        input_dim=8,
        gcn_hidden=32,
        lstm_hidden=64,
        dropout=0.1
    ):
        super().__init__()

        self.gcn1 = DenseEquivalentGCNConv(
            input_dim,
            gcn_hidden
        )

        self.gcn2 = DenseEquivalentGCNConv(
            gcn_hidden,
            gcn_hidden
        )

        self.graph_dropout = nn.Dropout(
            dropout
        )

        self.lstm = nn.LSTM(
            input_size=gcn_hidden * 2,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True
        )

        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 3)
        )

    @staticmethod
    def normalize_adjacency(edge_flow):

        # Edge matrices use [source, target].
        # Original GCN preprocessing normalizes
        # outgoing flow weights per source node.

        outgoing_sum = edge_flow.sum(
            dim=-1,
            keepdim=True
        )

        weights = (
            edge_flow
            / outgoing_sum.clamp(min=1e-8)
        )

        # GCN aggregation is destination <- source.
        adjacency = weights.transpose(1, 2)

        # PyG's standard GCN normalization adds
        # unit self-loops before symmetric normalization.

        n = adjacency.shape[-1]

        identity = torch.eye(
            n,
            dtype=adjacency.dtype,
            device=adjacency.device
        ).unsqueeze(0)

        adjacency = adjacency + identity

        degree = adjacency.sum(dim=-1)

        degree_inv_sqrt = degree.clamp(
            min=1e-8
        ).pow(-0.5)

        normalized = (
            degree_inv_sqrt.unsqueeze(-1)
            * adjacency
            * degree_inv_sqrt.unsqueeze(1)
        )

        return normalized

    def forward(self, nodes, edges):

        if nodes.ndim != 4 or edges.ndim != 4:
            raise ValueError(
                "Expected 4D node and edge histories."
            )

        batch, time_steps, node_count, features = (
            nodes.shape
        )

        if (
            time_steps,
            node_count,
            features
        ) != (10, 8, 8):
            raise ValueError(
                "Expected nodes (batch,10,8,8)"
            )

        if edges.shape != (
            batch, 10, 8, 8
        ):
            raise ValueError(
                "Expected edges (batch,10,8,8)"
            )

        # Process the ten graph snapshots
        # using the same shared GCN weights.

        x = nodes.reshape(
            batch * time_steps,
            8,
            8
        )

        edge_flow = edges.reshape(
            batch * time_steps,
            8,
            8
        )

        adjacency = self.normalize_adjacency(
            edge_flow
        )

        h = torch.relu(
            self.gcn1(x, adjacency)
        )

        h = self.graph_dropout(h)

        h = torch.relu(
            self.gcn2(h, adjacency)
        )

        # Original graph-level pooling:
        # concatenate node mean and node max.

        graph_mean = h.mean(dim=1)

        graph_max = h.max(dim=1).values

        graph_embeddings = torch.cat(
            [
                graph_mean,
                graph_max
            ],
            dim=-1
        )

        graph_sequence = graph_embeddings.reshape(
            batch,
            time_steps,
            -1
        )

        temporal_output, _ = self.lstm(
            graph_sequence
        )

        last_hidden = temporal_output[:, -1, :]

        logits = self.head(last_hidden)

        # Original monotonic horizon decoding.

        hazards = torch.sigmoid(logits)

        cumulative_probability = (
            1.0 - torch.cumprod(
                1.0 - hazards,
                dim=1
            )
        )

        return cumulative_probability.clamp(
            min=1e-6,
            max=1.0 - 1e-6
        )


# =============================================================================
# 5. RESTORE ORIGINAL CHECKPOINT AND TRAIN SCALER
# =============================================================================

# Trusted original checkpoint from our own training workflow.
# It contains NumPy scaler arrays as well as model tensors.
# Do not use weights_only=False with untrusted checkpoints.

GCN_CHECKPOINT = torch.load(
    GCN_MODEL_FILE,
    map_location="cpu",
    weights_only=False
)

required_keys = {
    "model_state",
    "median",
    "scale",
    "graph_sha256",
    "config",
    "architecture"
}

assert required_keys.issubset(
    GCN_CHECKPOINT
)

assert GCN_CHECKPOINT["graph_sha256"] == (
    GRAPH_TRAIN_MANIFEST["artifact_sha256"]
)

assert GCN_CHECKPOINT["architecture"] == (
    "PyG_GCNConv_GCNConv_LSTM"
)

assert GCN_CHECKPOINT["config"] == GCN_CONFIG

GCN_MODEL = GCNLSTMForecaster(
    input_dim=8,
    gcn_hidden=GCN_CONFIG["gcn_hidden"],
    lstm_hidden=GCN_CONFIG["lstm_hidden"],
    dropout=GCN_CONFIG["dropout"]
).to(DEVICE)

GCN_MODEL.load_state_dict(
    GCN_CHECKPOINT["model_state"],
    strict=True
)

GCN_MODEL.eval()
GCN_MODEL.requires_grad_(False)

GCN_MEDIAN = np.asarray(
    GCN_CHECKPOINT["median"],
    dtype=np.float32
)

GCN_SCALE = np.asarray(
    GCN_CHECKPOINT["scale"],
    dtype=np.float32
)

assert GCN_MEDIAN.shape == (8,)
assert GCN_SCALE.shape == (8,)

assert np.isfinite(GCN_MEDIAN).all()
assert np.isfinite(GCN_SCALE).all()
assert np.all(GCN_SCALE > 0)

GCN_PARAMETERS = sum(
    p.numel()
    for p in GCN_MODEL.parameters()
)

assert GCN_PARAMETERS == 36803

print("\n" + "-" * 90)
print("ORIGINAL GCN-LSTM RESTORED")
print("-" * 90)

print("Architecture: GCNConv -> GCNConv -> LSTM")
print("Graph hidden dimension:", GCN_CONFIG["gcn_hidden"])
print("LSTM hidden dimension:", GCN_CONFIG["lstm_hidden"])
print("Parameters:", GCN_PARAMETERS)
print("Device:", DEVICE)
print("Strict checkpoint loading: PASS")
print("Original TRAIN scaler restored: PASS")

# =============================================================================
# 6. GCN-LSTM INFERENCE ADAPTER
#
# Inputs:
#   node_history: (10,8,8) or (batch,10,8,8)
#   edge_history: (10,8,8) or (batch,10,8,8)
#
# Returns: (batch,3), ordered H5/H10/H15
# =============================================================================

def predict_gcn_lstm(
    node_history,
    edge_history
):

    nodes = np.asarray(
        node_history,
        dtype=np.float32
    )

    edges = np.asarray(
        edge_history,
        dtype=np.float32
    )

    if nodes.shape == (10, 8, 8):
        nodes = nodes[None, ...]

    if edges.shape == (10, 8, 8):
        edges = edges[None, ...]

    if (
        nodes.ndim != 4
        or nodes.shape[1:] != (10, 8, 8)
    ):
        raise ValueError(
            f"Invalid node shape: {nodes.shape}"
        )

    if edges.shape != nodes.shape:
        raise ValueError(
            f"Invalid edge shape: {edges.shape}"
        )

    if (
        not np.isfinite(nodes).all()
        or (nodes < 0).any()
    ):
        raise ValueError(
            "Invalid GCN node features."
        )

    if (
        not np.isfinite(edges).all()
        or (edges < 0).any()
    ):
        raise ValueError(
            "Invalid GCN edge features."
        )

    # Frozen TRAIN preprocessing.
    # Do not fit on validation samples.

    scaled_nodes = np.clip(
        (
            np.log1p(nodes)
            - GCN_MEDIAN
        ) / GCN_SCALE,
        -8.0,
        8.0
    ).astype(np.float32)

    node_tensor = torch.from_numpy(
        np.ascontiguousarray(scaled_nodes)
    ).to(DEVICE)

    edge_tensor = torch.from_numpy(
        np.ascontiguousarray(edges)
    ).to(DEVICE)

    with torch.inference_mode():

        probabilities = GCN_MODEL(
            node_tensor,
            edge_tensor
        ).cpu().numpy().astype(np.float64)

    if probabilities.shape != (len(nodes), 3):
        raise RuntimeError(
            "GCN returned unexpected prediction dimensions."
        )

    if not np.isfinite(probabilities).all():
        raise RuntimeError(
            "GCN returned non-finite probabilities."
        )

    if (
        (probabilities < 0).any()
        or (probabilities > 1).any()
    ):
        raise RuntimeError(
            "GCN probabilities outside [0,1]."
        )

    if not (
        np.all(
            probabilities[:, 0]
            <= probabilities[:, 1] + 1e-7
        )
        and np.all(
            probabilities[:, 1]
            <= probabilities[:, 2] + 1e-7
        )
    ):
        raise RuntimeError(
            "GCN horizon monotonicity violated."
        )

    return probabilities


# =============================================================================
# 7. VERIFY AGAINST ORIGINAL SAVED VALIDATION PREDICTIONS
#
# Validation predictions are used ONLY as numerical
# reproduction references.
#
# No target labels are read.
# =============================================================================

print("\n" + "-" * 90)
print("GCN-LSTM SAVED-INFERENCE REPRODUCTION")
print("-" * 90)

with np.load(
    GCN_REFERENCE_FILE,
    allow_pickle=False
) as saved:

    required_saved_keys = {
        "episode_id",
        "minute",
        "gcn_probabilities"
    }

    assert required_saved_keys.issubset(
        set(saved.files)
    )

    saved_episodes = saved[
        "episode_id"
    ].astype(str)

    saved_minutes = saved[
        "minute"
    ].astype(np.int64)

    assert np.array_equal(
        saved_episodes,
        NODE_VAL_EPISODES
    )

    assert np.array_equal(
        saved_minutes,
        NODE_VAL_MINUTES
    )

    assert saved[
        "gcn_probabilities"
    ].shape == (18839, 3)

    # Check geographically unrelated positions
    # in the synthetic validation array.
    #
    # Also include previously verified sample indices.
    # Selection does not use ground-truth labels.

    verify_indices = np.unique(
        np.concatenate([
            np.linspace(
                0,
                len(NODE_VAL_EPISODES) - 1,
                num=48,
                dtype=np.int64
            ),
            np.asarray(
                sample_indices,
                dtype=np.int64
            )
        ])
    )

    saved_probabilities = saved[
        "gcn_probabilities"
    ][verify_indices].astype(np.float64)

# Reconstruct original graph histories from
# already-aligned, frozen synthetic graph frames.

verify_nodes = np.stack([
    get_node_history(
        NODE_VAL_EPISODES[i],
        NODE_VAL_MINUTES[i]
    )
    for i in verify_indices
])

verify_edges = np.stack([
    get_original_edge_history(
        NODE_VAL_EPISODES[i],
        NODE_VAL_MINUTES[i]
    )
    for i in verify_indices
])

reproduced_probabilities = predict_gcn_lstm(
    verify_nodes,
    verify_edges
)

GCN_REPLAY_DIFFERENCE = float(
    np.max(
        np.abs(
            reproduced_probabilities
            - saved_probabilities
        )
    )
)

print("Validation windows checked:", len(verify_indices))

print(
    "Maximum prediction difference:",
    f"{GCN_REPLAY_DIFFERENCE:.10f}"
)

GCN_REPLAY_TOLERANCE = 1e-5

if GCN_REPLAY_DIFFERENCE > GCN_REPLAY_TOLERANCE:

    worst = int(
        np.argmax(
            np.max(
                np.abs(
                    reproduced_probabilities
                    - saved_probabilities
                ),
                axis=1
            )
        )
    )

    print("Worst validation row:", int(verify_indices[worst]))

    print(
        "Expected:",
        saved_probabilities[worst].tolist()
    )

    print(
        "Reproduced:",
        reproduced_probabilities[worst].tolist()
    )

    raise RuntimeError(
        "GCN saved-inference reproduction failed. "
        "Do not enable the adapter."
    )

print("GCN saved-inference reproduction: PASS")
print("Verification tolerance:", GCN_REPLAY_TOLERANCE)
print("Numerical difference:", GCN_REPLAY_DIFFERENCE)
print("Exact bitwise reproduction: NO")
print("Approximate numerical reproduction: VERIFIED")


# =============================================================================
# 8. INTEGRATE INTO THE EXISTING FORECASTING FUNCTION
#
# Preserve all previously integrated models.
#
# GCN results remain independent auxiliary predictions.
# =============================================================================

if "_CELL14_BASE_FORECAST" not in globals():

    _CELL14_BASE_FORECAST = (
        forecast_network_window
    )


def forecast_network_window(
    traffic_df,
    *,
    graph_node_history=None,
    graph_edge_history=None
):

    result = _CELL14_BASE_FORECAST(
        traffic_df,
        graph_node_history=graph_node_history,
        graph_edge_history=graph_edge_history
    )

    if (
        graph_node_history is None
        or graph_edge_history is None
    ):

        result["auxiliary_models"]["gcn_lstm"] = {
            "status": "NODE_AND_EDGE_INPUT_REQUIRED",
            "H5": None,
            "H10": None,
            "H15": None
        }

        return result

    probabilities = predict_gcn_lstm(
        graph_node_history,
        graph_edge_history
    )

    if probabilities.shape != (1, 3):
        raise RuntimeError(
            "Unified GCN inference requires one window."
        )

    p = probabilities[0]

    result["auxiliary_models"]["gcn_lstm"] = {

        "status": "AVAILABLE",

        "H5": float(p[0]),
        "H10": float(p[1]),
        "H15": float(p[2]),

        "input_representation": "SYNTHETIC_GRAPH_10x8x8",

        "architecture": "GCNConv_GCNConv_LSTM",

        "uses_directed_edges": True,

        "real_ip_topology": False,

        "primary_forecast": False
    }

    return result


# =============================================================================
# 9. REGISTER GCN-LSTM IN THE MODEL REGISTRY
# =============================================================================

MODEL_REGISTRY["gcn_lstm"] = {

    "name": "gcn_lstm",

    "path": str(GCN_MODEL_FILE),

    "sha256": GCN_CHECKPOINT_SHA256,

    "checkpoint_verified": True,

    "inference_connected": True,

    "architecture": "PyG_GCNConv_GCNConv_LSTM",

    "input_shape_nodes": [10, 8, 8],

    "input_shape_edges": [10, 8, 8],

    "horizons": [5, 10, 15],

    "role": "auxiliary_graph_model",

    "graph_origin": "synthetic_node_and_edge_allocations"
}


# =============================================================================
# 10. CHECK EXISTING PRIMARY AND AUXILIARY OUTPUTS
# =============================================================================

old_forecast = _CELL14_BASE_FORECAST(
    sample_df
)

new_forecast = forecast_network_window(
    sample_df
)

assert (
    old_forecast["forecast_probabilities"]
    == new_forecast["forecast_probabilities"]
)

assert new_forecast["primary_model"] == "stacked_lstm"

for model_name in old_forecast["auxiliary_models"]:

    assert (
        old_forecast["auxiliary_models"][model_name]
        == new_forecast["auxiliary_models"][model_name]
    )

assert (
    new_forecast["auxiliary_models"][
        "gcn_lstm"
    ]["status"]
    == "NODE_AND_EDGE_INPUT_REQUIRED"
)

print("\nPrimary Stacked LSTM: UNCHANGED")
print("Existing auxiliary models: UNCHANGED")
print("Graph input requirement: ENFORCED")


# =============================================================================
# 11. DEMONSTRATE THE FULL GRAPH-ENABLED FORECASTER
#
# Reuse the Cell 12 synthetic graph helper, which calls
# the current global forecast_network_window function.
# =============================================================================

demo_episode = sample_episode
demo_minute = int(sample_end_minute)

if (
    "LATEST_EXPLORER_REPLAY" in globals()
    and LATEST_EXPLORER_REPLAY is not None
):

    demo_episode = str(
        LATEST_EXPLORER_REPLAY["episode"]
    )

    demo_minute = int(
        LATEST_EXPLORER_REPLAY[
            "timeline"
        ]["Minute"].max()
    )

GCN_INTEGRATED_FORECAST = forecast_synthetic_graph_window(
    demo_episode,
    demo_minute
)

primary = GCN_INTEGRATED_FORECAST[
    "forecast_probabilities"
]

auxiliary = GCN_INTEGRATED_FORECAST[
    "auxiliary_models"
]

assert auxiliary["gcn_lstm"]["status"] == "AVAILABLE"

for name in [
    "node_aware_lstm",
    "custom_graph_lstm",
    "optimized_graph_lstm",
    "gcn_lstm"
]:

    assert auxiliary[name]["status"] == "AVAILABLE"

print("\n" + "=" * 90)
print("NETWORK FORECAST — GCN-LSTM CONNECTED")
print("=" * 90)

print("Episode:", demo_episode)
print("Last observed minute:", demo_minute)

print("\nPRIMARY — STACKED LSTM")

for horizon in ("H5", "H10", "H15"):

    print(
        f"  {horizon}: "
        f"{primary[horizon] * 100:.4f}%"
    )

for name, label in [
    ("node_aware_lstm", "NODE-AWARE LSTM"),
    ("custom_graph_lstm", "CUSTOM GRAPH LSTM"),
    ("optimized_graph_lstm", "OPTIMIZED GRAPH LSTM"),
    ("gcn_lstm", "GCN-LSTM")
]:

    print(f"\nAUXILIARY — {label}")

    for horizon in ("H5", "H10", "H15"):

        print(
            f"  {horizon}: "
            f"{auxiliary[name][horizon] * 100:.4f}%"
        )

print("\nSynthetic graph input: VERIFIED")
print("Real host/IP topology: NOT AVAILABLE")
print("Model fusion performed: NO")
print("Primary alert policy changed: NO")


# =============================================================================
# 12. CHECK GCN REPEATABILITY
# =============================================================================

GCN_REPEAT_FORECAST = forecast_synthetic_graph_window(
    demo_episode,
    demo_minute
)

for horizon in ("H5", "H10", "H15"):

    assert np.isclose(

        GCN_INTEGRATED_FORECAST["auxiliary_models"][
            "gcn_lstm"
        ][horizon],

        GCN_REPEAT_FORECAST["auxiliary_models"][
            "gcn_lstm"
        ][horizon],

        atol=1e-7,
        rtol=0
    )

print("\nGCN inference repeatability: PASS")


# =============================================================================
# 13. FINAL STATUS
# =============================================================================

print("\n" + "=" * 90)
print("CELL 14 — GCN-LSTM INTEGRATION: COMPLETE")
print("=" * 90)

print("Original checkpoint SHA256: VERIFIED")
print("Original GCN architecture: RESTORED")
print("Original TRAIN scaler: RESTORED")
print("Original validation predictions: REPRODUCED")
print("Synthetic graph alignment: VERIFIED")

print("\nGCN-LSTM H5/H10/H15: CONNECTED")
print("Node-aware LSTM: STILL CONNECTED")
print("Custom Graph LSTM: STILL CONNECTED")
print("Optimized Graph LSTM: STILL CONNECTED")

print("\nPrimary Stacked LSTM: UNCHANGED")
print("Existing auxiliary outputs: UNCHANGED")
print("Alert policy: UNCHANGED")
print("Model fusion: NONE")

print("\nModel retraining: NO")
print("Scaler refitting: NO")
print("Official TEST accessed: NO")
print("Files written: NO")

print("\nNEXT: GAT-LSTM integration.")
print("=" * 90)


In [ ]:

# =============================================================================
# NETWORK ATTACK FORECASTING PIPELINE
# CELL 15 — GAT-LSTM INFERENCE AND INTEGRATION
#
# Frozen original architecture:
#   GATConv(8,32, heads=2, concat=False, edge_dim=1)
#   GATConv(32,32, heads=2, concat=False, edge_dim=1)
#   ELU activations
#   Mean + max graph pooling
#   LSTM(64,64)
#   Monotonic H5/H10/H15 forecasting head
#
# Uses a dense PyTorch implementation of the original
# PyTorch Geometric attention calculations for 8-node graphs.
#
# Verification against the saved GAT validation predictions.
#
# NO RETRAINING
# NO SCALER REFITTING
# NO TEST ACCESS
# NO MODEL FUSION
# NO FILE WRITES
# =============================================================================

from pathlib import Path
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as torch_F

print("=" * 90)
print("CELL 15 — GAT-LSTM INTEGRATION")
print("=" * 90)

# =============================================================================
# 1. LOCATE ORIGINAL ARTIFACTS
# =============================================================================

GAT_ROOT = (
    ROOT
    / "Synthetic_NAF_v2_GraphCompanion"
    / "models"
    / "GAT_LSTM"
)

GAT_TRAIN_DIR = GAT_ROOT / "train_only_v1"
GAT_VAL_DIR = GAT_ROOT / "validation_v1"

GAT_MODEL_FILE = (
    GAT_TRAIN_DIR / "GAT_LSTM_STATE.pt"
)

GAT_RECORD_FILE = (
    GAT_TRAIN_DIR / "GAT_LSTM_TRAIN_RECORD.json"
)

GAT_REFERENCE_FILE = (
    GAT_VAL_DIR / "GAT_VAL_PREDICTIONS.npz"
)

GAT_REPORT_FILE = (
    GAT_VAL_DIR / "GAT_VAL_REPORT.json"
)

for path in [
    GAT_MODEL_FILE,
    GAT_RECORD_FILE,
    GAT_REFERENCE_FILE,
    GAT_REPORT_FILE
]:
    if not path.is_file():
        raise FileNotFoundError(path)

GAT_RECORD = json.loads(
    GAT_RECORD_FILE.read_text(encoding="utf-8")
)

GAT_REPORT = json.loads(
    GAT_REPORT_FILE.read_text(encoding="utf-8")
)

GAT_CONFIG = GAT_RECORD["config"]

# =============================================================================
# 2. VERIFY CHECKPOINT PROVENANCE AND ARCHITECTURE
# =============================================================================

GAT_SHA256 = sha256(GAT_MODEL_FILE)

assert GAT_SHA256 == GAT_RECORD["checkpoint_sha256"]
assert GAT_SHA256 == GAT_REPORT["gat_checkpoint_sha256"]

assert GAT_RECORD["model"] == "GATLSTMForecaster"

assert GAT_RECORD["architecture"] == (
    "PyG_GATConv_GATConv_LSTM"
)

assert GAT_RECORD["train_graph_sha256"] == (
    GRAPH_TRAIN_MANIFEST["artifact_sha256"]
)

assert GAT_REPORT["validation_graph_sha256"] == (
    GRAPH_VAL_MANIFEST["artifact_sha256"]
)

assert GAT_RECORD["official_validation_accessed"] is False
assert GAT_RECORD["official_test_accessed"] is False

assert GAT_CONFIG["nodes"] == 8
assert GAT_CONFIG["node_features"] == 8
assert GAT_CONFIG["history"] == 10

assert GAT_CONFIG["gat_hidden"] == 32
assert GAT_CONFIG["attention_heads"] == 2
assert GAT_CONFIG["lstm_hidden"] == 64
assert GAT_CONFIG["dropout"] == 0.1

assert GAT_CONFIG["gat_concat"] is False
assert GAT_CONFIG["gat_self_loop_fill"] == 0.0

assert GAT_CONFIG["edge_processing"] == (
    "OUTGOING_ROW_NORMALIZED"
)

assert GAT_CONFIG["edge_attributes"] == (
    "NORMALIZED_EDGE_FLOW"
)

assert GAT_RECORD["parameters"] == 38595

print("\nOriginal GAT training record: VERIFIED")
print("Original checkpoint SHA256: VERIFIED")
print("Frozen graph source: VERIFIED")
print("Attention configuration: VERIFIED")
print("Official TEST access: NONE")

# =============================================================================
# 3. DENSE EQUIVALENT OF ORIGINAL PyG GATConv
#
# Original parameters:
#   lin.weight
#   lin_edge.weight
#   att_src
#   att_dst
#   att_edge
#   bias
#
# The original PyG implementation:
#   - performs directed source -> destination aggregation
#   - adds self-loops to all eight nodes
#   - sets self-loop edge attributes to zero
#   - applies LeakyReLU before incoming-edge softmax
#   - averages attention heads (concat=False)
#
# This class retains exactly the original state_dict keys.
# =============================================================================

class DenseEquivalentGATConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        heads=2
    ):
        super().__init__()

        self.heads = heads
        self.out_channels = out_channels

        self.lin = nn.Linear(
            in_channels,
            heads * out_channels,
            bias=False
        )

        self.lin_edge = nn.Linear(
            1,
            heads * out_channels,
            bias=False
        )

        self.att_src = nn.Parameter(
            torch.zeros(1, heads, out_channels)
        )

        self.att_dst = nn.Parameter(
            torch.zeros(1, heads, out_channels)
        )

        self.att_edge = nn.Parameter(
            torch.zeros(1, heads, out_channels)
        )

        self.bias = nn.Parameter(
            torch.zeros(out_channels)
        )

    def forward(self, x, directed_edges):

        # x: [G, 8, input_features]
        # directed_edges: [G, source, destination]

        graph_count, n, _ = x.shape

        heads = self.heads
        hidden = self.out_channels

        # Project into independent attention heads.

        projected = self.lin(x).reshape(
            graph_count, n, heads, hidden
        )

        # Source-node and destination-node scores.

        alpha_source = (
            projected * self.att_src
        ).sum(dim=-1)

        alpha_target = (
            projected * self.att_dst
        ).sum(dim=-1)

        # PyG replaces original self-loops with loops
        # whose edge features are zero (fill_value=0.0).

        identity = torch.eye(
            n,
            dtype=torch.bool,
            device=x.device
        ).unsqueeze(0)

        edge_attributes = torch.where(
            identity,
            torch.zeros_like(directed_edges),
            directed_edges
        )

        edge_projected = self.lin_edge(
            edge_attributes.unsqueeze(-1)
        ).reshape(
            graph_count,
            n,
            n,
            heads,
            hidden
        )

        alpha_edge = (
            edge_projected * self.att_edge
        ).sum(dim=-1)

        # Tensor axes: [G, source, destination, heads]

        logits = (
            alpha_source[:, :, None, :]
            + alpha_target[:, None, :, :]
            + alpha_edge
        )

        logits = torch_F.leaky_relu(
            logits,
            negative_slope=0.2
        )

        # Original nonzero directed edges + self loops.
        # Zero-weight non-self edges do not exist.

        edge_mask = (
            ((directed_edges > 0) & (~identity))
            | identity
        )

        logits = logits.masked_fill(
            ~edge_mask.unsqueeze(-1),
            -float("inf")
        )

        # PyG performs softmax over incoming edges,
        # grouped by destination node.

        attention = torch.softmax(
            logits,
            dim=1
        )

        # Aggregate source-node embeddings into
        # their destinations.

        aggregated = torch.einsum(
            "bsdh,bshk->bdhk",
            attention,
            projected
        )

        # concat=False: average the attention heads.

        output = aggregated.mean(dim=2)

        return output + self.bias


# =============================================================================
# 4. RESTORE ORIGINAL TWO-LAYER GAT + LSTM
# =============================================================================

class DenseEquivalentGATLSTM(nn.Module):

    def __init__(
        self,
        input_features=8,
        gat_hidden=32,
        heads=2,
        lstm_hidden=64,
        dropout=0.1
    ):
        super().__init__()

        self.gat1 = DenseEquivalentGATConv(
            input_features,
            gat_hidden,
            heads
        )

        self.gat2 = DenseEquivalentGATConv(
            gat_hidden,
            gat_hidden,
            heads
        )

        self.dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=2 * gat_hidden,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True
        )

        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 3)
        )

    def forward(self, nodes, edges):

        if nodes.ndim != 4 or edges.ndim != 4:
            raise ValueError(
                "Expected 4-dimensional graph histories."
            )

        batch, time_steps, node_count, features = (
            nodes.shape
        )

        if (
            time_steps,
            node_count,
            features
        ) != (10, 8, 8):
            raise ValueError(
                "Expected nodes (batch,10,8,8)"
            )

        if edges.shape != (
            batch, 10, 8, 8
        ):
            raise ValueError(
                "Expected edges (batch,10,8,8)"
            )

        # Flatten the temporal graph sequence.

        x = nodes.reshape(
            batch * time_steps,
            node_count,
            features
        )

        adjacency = edges.reshape(
            batch * time_steps,
            node_count,
            node_count
        )

        # Two edge-aware graph attention layers.

        h = torch_F.elu(
            self.gat1(x, adjacency)
        )

        h = self.dropout(h)

        h = torch_F.elu(
            self.gat2(h, adjacency)
        )

        # Graph-level mean/max pooling.

        graph_mean = h.mean(dim=1)
        graph_max = h.max(dim=1).values

        graph_embeddings = torch.cat(
            [
                graph_mean,
                graph_max
            ],
            dim=-1
        )

        sequence = graph_embeddings.reshape(
            batch,
            time_steps,
            -1
        )

        # Temporal LSTM.

        temporal_output, _ = self.lstm(
            sequence
        )

        last_state = temporal_output[:, -1, :]

        logits = self.head(last_state)

        # Monotonic cumulative onset probabilities.

        hazards = torch.sigmoid(logits)

        probabilities = (
            1.0 - torch.cumprod(
                1.0 - hazards,
                dim=1
            )
        )

        return probabilities.clamp(
            min=1e-6,
            max=1.0 - 1e-6
        )


# =============================================================================
# 5. LOAD ORIGINAL CHECKPOINT AND TRAIN SCALER
# =============================================================================

# Trusted checkpoint created by the project's own
# training code. It contains NumPy arrays.
#
# Never use weights_only=False on untrusted files.

GAT_CHECKPOINT = torch.load(
    GAT_MODEL_FILE,
    map_location="cpu",
    weights_only=False
)

required_checkpoint_keys = {
    "model_state",
    "median",
    "scale",
    "graph_sha256",
    "config",
    "architecture"
}

assert required_checkpoint_keys.issubset(
    GAT_CHECKPOINT.keys()
)

assert GAT_CHECKPOINT["graph_sha256"] == (
    GRAPH_TRAIN_MANIFEST["artifact_sha256"]
)

assert GAT_CHECKPOINT["architecture"] == (
    "PyG_GATConv_GATConv_LSTM"
)

assert GAT_CHECKPOINT["config"] == GAT_CONFIG

GAT_MODEL = DenseEquivalentGATLSTM(
    input_features=8,
    gat_hidden=GAT_CONFIG["gat_hidden"],
    heads=GAT_CONFIG["attention_heads"],
    lstm_hidden=GAT_CONFIG["lstm_hidden"],
    dropout=GAT_CONFIG["dropout"]
).to(DEVICE)

GAT_MODEL.load_state_dict(
    GAT_CHECKPOINT["model_state"],
    strict=True
)

GAT_MODEL.eval()
GAT_MODEL.requires_grad_(False)

GAT_MEDIAN = np.asarray(
    GAT_CHECKPOINT["median"],
    dtype=np.float32
)

GAT_SCALE = np.asarray(
    GAT_CHECKPOINT["scale"],
    dtype=np.float32
)

assert GAT_MEDIAN.shape == (8,)
assert GAT_SCALE.shape == (8,)

assert np.isfinite(GAT_MEDIAN).all()
assert np.isfinite(GAT_SCALE).all()
assert (GAT_SCALE > 0).all()

GAT_PARAMETERS = sum(
    p.numel()
    for p in GAT_MODEL.parameters()
)

assert GAT_PARAMETERS == 38595

print("\n" + "-" * 90)
print("ORIGINAL GAT-LSTM RESTORED")
print("-" * 90)

print("Architecture: GATConv -> GATConv -> LSTM")
print("Attention heads:", GAT_CONFIG["attention_heads"])
print("GAT hidden dimension:", GAT_CONFIG["gat_hidden"])
print("LSTM hidden dimension:", GAT_CONFIG["lstm_hidden"])
print("Parameters:", GAT_PARAMETERS)
print("Device:", DEVICE)

print("Strict checkpoint loading: PASS")
print("Original TRAIN scaler: RESTORED")

# =============================================================================
# 6. GAT INFERENCE WITH ORIGINAL TRAIN PREPROCESSING
# =============================================================================

def predict_gat_lstm(
    node_history,
    edge_history,
    batch_size=32
):

    nodes = np.asarray(
        node_history,
        dtype=np.float32
    )

    edges = np.asarray(
        edge_history,
        dtype=np.float32
    )

    if nodes.shape == (10, 8, 8):
        nodes = nodes[None, ...]

    if edges.shape == (10, 8, 8):
        edges = edges[None, ...]

    if (
        nodes.ndim != 4
        or nodes.shape[1:] != (10, 8, 8)
    ):
        raise ValueError(
            f"Invalid GAT node shape: {nodes.shape}"
        )

    if edges.shape != nodes.shape:
        raise ValueError(
            f"Invalid GAT edge shape: {edges.shape}"
        )

    if (
        not np.isfinite(nodes).all()
        or (nodes < 0).any()
    ):
        raise ValueError(
            "Invalid GAT node features."
        )

    if (
        not np.isfinite(edges).all()
        or (edges < 0).any()
    ):
        raise ValueError(
            "Invalid GAT edge features."
        )

    # Frozen TRAIN-fitted node preprocessing.

    nodes_scaled = np.clip(
        (
            np.log1p(nodes)
            - GAT_MEDIAN
        ) / GAT_SCALE,
        -8.0,
        8.0
    ).astype(np.float32)

    # Original outgoing source-row edge normalization.

    outgoing = edges.sum(
        axis=-1,
        keepdims=True
    )

    edges_scaled = (
        edges / np.maximum(outgoing, 1e-8)
    ).astype(np.float32)

    predictions = []

    with torch.inference_mode():

        for start in range(
            0,
            len(nodes),
            batch_size
        ):

            stop = min(
                start + batch_size,
                len(nodes)
            )

            node_tensor = torch.from_numpy(
                np.ascontiguousarray(
                    nodes_scaled[start:stop]
                )
            ).to(DEVICE)

            edge_tensor = torch.from_numpy(
                np.ascontiguousarray(
                    edges_scaled[start:stop]
                )
            ).to(DEVICE)

            probabilities = GAT_MODEL(
                node_tensor,
                edge_tensor
            )

            predictions.append(
                probabilities.cpu().numpy()
            )

    result = np.concatenate(
        predictions,
        axis=0
    ).astype(np.float64)

    if result.shape != (len(nodes), 3):
        raise RuntimeError(
            "Unexpected GAT output dimensions."
        )

    if (
        not np.isfinite(result).all()
        or (result < 0).any()
        or (result > 1).any()
    ):
        raise RuntimeError(
            "GAT produced invalid probabilities."
        )

    if not (
        np.all(
            result[:, 0] <= result[:, 1] + 1e-7
        )
        and np.all(
            result[:, 1] <= result[:, 2] + 1e-7
        )
    ):
        raise RuntimeError(
            "GAT horizon monotonicity violation."
        )

    return result


# =============================================================================
# 7. NUMERICAL REPLAY AGAINST SAVED VALIDATION PREDICTIONS
#
# Only archived predictions and row IDs are read.
# Ground-truth targets are not accessed.
# =============================================================================

print("\n" + "-" * 90)
print("GAT-LSTM SAVED-INFERENCE REPRODUCTION")
print("-" * 90)

with np.load(
    GAT_REFERENCE_FILE,
    allow_pickle=False
) as saved:

    required_keys = {
        "episode_id",
        "minute",
        "gat_probabilities"
    }

    assert required_keys.issubset(
        set(saved.files)
    )

    saved_episodes = saved[
        "episode_id"
    ].astype(str)

    saved_minutes = saved[
        "minute"
    ].astype(np.int64)

    assert np.array_equal(
        saved_episodes,
        NODE_VAL_EPISODES
    )

    assert np.array_equal(
        saved_minutes,
        NODE_VAL_MINUTES
    )

    assert saved[
        "gat_probabilities"
    ].shape == (18839, 3)

    # Deterministic, label-independent sample:
    # initial windows + windows spread across
    # the entire validation set.

    verify_indices = np.unique(
        np.concatenate([
            np.arange(24, dtype=np.int64),
            np.linspace(
                0,
                len(saved_episodes) - 1,
                num=96,
                dtype=np.int64
            )
        ])
    )

    saved_predictions = saved[
        "gat_probabilities"
    ][verify_indices].astype(np.float64)

verify_nodes = np.stack([
    get_node_history(
        NODE_VAL_EPISODES[i],
        NODE_VAL_MINUTES[i]
    )
    for i in verify_indices
])

verify_edges = np.stack([
    get_original_edge_history(
        NODE_VAL_EPISODES[i],
        NODE_VAL_MINUTES[i]
    )
    for i in verify_indices
])

replayed_predictions = predict_gat_lstm(
    verify_nodes,
    verify_edges,
    batch_size=16
)

GAT_REPLAY_DIFFERENCE = float(
    np.max(
        np.abs(
            replayed_predictions
            - saved_predictions
        )
    )
)

GAT_REPLAY_TOLERANCE = 1e-5

print("Validation windows checked:", len(verify_indices))

print(
    "Maximum saved-prediction difference:",
    f"{GAT_REPLAY_DIFFERENCE:.10f}"
)

print(
    "Allowed numerical tolerance:",
    GAT_REPLAY_TOLERANCE
)

if GAT_REPLAY_DIFFERENCE > GAT_REPLAY_TOLERANCE:

    row_errors = np.max(
        np.abs(
            replayed_predictions
            - saved_predictions
        ),
        axis=1
    )

    worst_position = int(
        np.argmax(row_errors)
    )

    print(
        "Worst validation row:",
        int(verify_indices[worst_position])
    )

    print(
        "Saved probabilities:",
        saved_predictions[worst_position].tolist()
    )

    print(
        "Replayed probabilities:",
        replayed_predictions[worst_position].tolist()
    )

    raise RuntimeError(
        "GAT saved-inference replay failed. "
        "The adapter has not been enabled."
    )

print("GAT saved-inference reproduction: PASS")
print("Verification tolerance:", GAT_REPLAY_TOLERANCE)
print("Maximum difference:", GAT_REPLAY_DIFFERENCE)
print("Exact bitwise reproduction: NO")
print("Approximate numerical reproduction: VERIFIED")


# =============================================================================
# 8. ADD GAT TO THE EXISTING UNIFIED FORECASTER
#
# No change to:
#   Stacked LSTM
#   Single-layer LSTM
#   Competing-risk LSTM
#   XGBoost H5/H10
#   Competing-risk TCN
#   Node-aware LSTM
#   Custom Graph LSTM
#   Optimized Graph LSTM
#   GCN-LSTM
# =============================================================================

if "_CELL15_BASE_FORECAST" not in globals():

    _CELL15_BASE_FORECAST = (
        forecast_network_window
    )


def forecast_network_window(
    traffic_df,
    *,
    graph_node_history=None,
    graph_edge_history=None
):

    result = _CELL15_BASE_FORECAST(
        traffic_df,
        graph_node_history=graph_node_history,
        graph_edge_history=graph_edge_history
    )

    if (
        graph_node_history is None
        or graph_edge_history is None
    ):

        result["auxiliary_models"]["gat_lstm"] = {
            "status": "NODE_AND_EDGE_INPUT_REQUIRED",
            "H5": None,
            "H10": None,
            "H15": None
        }

        return result

    probabilities = predict_gat_lstm(
        graph_node_history,
        graph_edge_history
    )

    if probabilities.shape != (1, 3):
        raise RuntimeError(
            "Expected exactly one GAT forecast."
        )

    p = probabilities[0]

    result["auxiliary_models"]["gat_lstm"] = {
        "status": "AVAILABLE",
        "H5": float(p[0]),
        "H10": float(p[1]),
        "H15": float(p[2]),
        "architecture": "GATConv_GATConv_LSTM",
        "attention_heads": 2,
        "edge_attributes": "NORMALIZED_EDGE_FLOW",
        "input_representation": "SYNTHETIC_GRAPH_10x8x8",
        "real_ip_topology": False,
        "primary_forecast": False
    }

    return result


# =============================================================================
# 9. REGISTER GAT-LSTM
# =============================================================================

MODEL_REGISTRY["gat_lstm"] = {
    "name": "gat_lstm",
    "path": str(GAT_MODEL_FILE),
    "sha256": GAT_SHA256,
    "checkpoint_verified": True,
    "inference_connected": True,
    "architecture": "PyG_GATConv_GATConv_LSTM",
    "attention_heads": 2,
    "input_shape_nodes": [10, 8, 8],
    "input_shape_edges": [10, 8, 8],
    "horizons": [5, 10, 15],
    "role": "auxiliary_graph_model",
    "graph_origin": "synthetic_node_and_edge_allocations"
}


# =============================================================================
# 10. VERIFY ALL EXISTING FORECAST OUTPUTS ARE UNCHANGED
# =============================================================================

old_forecast = _CELL15_BASE_FORECAST(
    sample_df
)

new_forecast = forecast_network_window(
    sample_df
)

assert (
    old_forecast["forecast_probabilities"]
    == new_forecast["forecast_probabilities"]
)

assert new_forecast["primary_model"] == "stacked_lstm"

for name in old_forecast["auxiliary_models"]:

    assert (
        old_forecast["auxiliary_models"][name]
        == new_forecast["auxiliary_models"][name]
    )

assert new_forecast["auxiliary_models"][
    "gat_lstm"
]["status"] == "NODE_AND_EDGE_INPUT_REQUIRED"

print("\nPrimary Stacked LSTM: UNCHANGED")
print("Existing auxiliary models: UNCHANGED")
print("Graph input requirement: ENFORCED")


# =============================================================================
# 11. RUN THE FULL GRAPH-ENABLED FORECAST
#
# Cell 12's helper uses the current global
# forecast_network_window() function, and supplies
# aligned synthetic nodes and directed edges.
# =============================================================================

demo_episode = sample_episode
demo_minute = int(sample_end_minute)

if (
    "LATEST_EXPLORER_REPLAY" in globals()
    and LATEST_EXPLORER_REPLAY is not None
):

    demo_episode = str(
        LATEST_EXPLORER_REPLAY["episode"]
    )

    demo_minute = int(
        LATEST_EXPLORER_REPLAY[
            "timeline"
        ]["Minute"].max()
    )

GAT_INTEGRATED_FORECAST = (
    forecast_synthetic_graph_window(
        demo_episode,
        demo_minute
    )
)

primary = GAT_INTEGRATED_FORECAST[
    "forecast_probabilities"
]

auxiliary = GAT_INTEGRATED_FORECAST[
    "auxiliary_models"
]

graph_models = [
    "node_aware_lstm",
    "custom_graph_lstm",
    "optimized_graph_lstm",
    "gcn_lstm",
    "gat_lstm"
]

for name in graph_models:

    assert auxiliary[name]["status"] == "AVAILABLE"

print("\n" + "=" * 90)
print("NETWORK FORECAST — ALL GRAPH MODELS CONNECTED")
print("=" * 90)

print("Episode:", demo_episode)
print("Last observed minute:", demo_minute)

print("\nPRIMARY — STACKED LSTM")

for horizon in ("H5", "H10", "H15"):

    print(
        f"  {horizon}: "
        f"{primary[horizon] * 100:.4f}%"
    )

model_labels = {
    "node_aware_lstm": "NODE-AWARE LSTM",
    "custom_graph_lstm": "CUSTOM GRAPH LSTM",
    "optimized_graph_lstm": "OPTIMIZED GRAPH LSTM",
    "gcn_lstm": "GCN-LSTM",
    "gat_lstm": "GAT-LSTM"
}

for name in graph_models:

    print(
        f"\nAUXILIARY — {model_labels[name]}"
    )

    for horizon in ("H5", "H10", "H15"):

        print(
            f"  {horizon}: "
            f"{auxiliary[name][horizon] * 100:.4f}%"
        )

print("\nSynthetic node/edge inputs: VERIFIED")
print("Real IP topology: NOT AVAILABLE")
print("Model fusion performed: NO")
print("Primary alert policy changed: NO")


# =============================================================================
# 12. GAT INFERENCE REPEATABILITY
# =============================================================================

GAT_REPEAT_FORECAST = (
    forecast_synthetic_graph_window(
        demo_episode,
        demo_minute
    )
)

for horizon in ("H5", "H10", "H15"):

    first = GAT_INTEGRATED_FORECAST[
        "auxiliary_models"
    ]["gat_lstm"][horizon]

    repeated = GAT_REPEAT_FORECAST[
        "auxiliary_models"
    ]["gat_lstm"][horizon]

    assert np.isclose(
        first,
        repeated,
        atol=1e-7,
        rtol=0
    )

print("\nGAT inference repeatability: PASS")


# =============================================================================
# 13. FINAL STATUS
# =============================================================================

print("\n" + "=" * 90)
print("CELL 15 — GAT-LSTM INTEGRATION: COMPLETE")
print("=" * 90)

print("Original checkpoint SHA256: VERIFIED")
print("Original attention architecture: RESTORED")
print("Original TRAIN scaler: RESTORED")
print("Saved validation predictions: REPRODUCED")
print("Synthetic graph alignment: VERIFIED")

print("\nGAT-LSTM H5/H10/H15: CONNECTED")
print("All five graph model adapters: CONNECTED")
print("All ten model sources: CONNECTED")

print("\nPrimary Stacked LSTM: UNCHANGED")
print("Existing auxiliary outputs: UNCHANGED")
print("Alert policy: UNCHANGED")
print("Model fusion: NONE")

print("\nRetraining performed: NO")
print("Scaler refitting: NO")
print("Official TEST accessed: NO")
print("Files modified: NO")

print("\nNEXT: Unified all-model forecasting dashboard.")
print("=" * 90)
